# Notebook 11 — Off-Time and Temporal Semantics

## Scope

This notebook asks:

> What does the source `off` field represent, how consistently is it formatted, and what temporal assumptions can safely be made during race reconstruction?

The source SQLite database remains read-only throughout this work.

All source-data queries must continue to apply `DATA_ROW_PREDICATE = "rowid <> 1"`.

The `off` field is part of the current candidate provisional race key:

`date + course + off`

This notebook will therefore investigate whether `off` can safely continue to support race reconstruction.

It will examine:

* observed raw formats;
* blank, malformed or exceptional values;
* consistency within provisional races;
* duplication within the same date and course;
* possible midnight or date-rollover behaviour;
* whether the value represents scheduled time, actual off time or another source convention;
* whether timezone or jurisdictional assumptions can be derived from the source alone.

Raw `off` values will remain preserved exactly as supplied.

This notebook will not yet:

* convert all race times into UTC;
* assign jurisdictional timezones;
* infer daylight-saving rules;
* redesign the race identity;
* design the final staging schema.

Any timezone or scheduled-versus-actual interpretation not supported by the source will remain unresolved rather than guessed.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

# Locate the project and source database.
PROJECT_ROOT = Path.cwd().resolve().parent
DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

DATA_ROW_PREDICATE = "rowid <> 1"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Source database not found: {DB_PATH}")

# Open SQLite explicitly in read-only mode.
connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)

off_inventory = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS data_like_runner_rows,
        COUNT(DISTINCT off) AS distinct_raw_off_values,
        SUM(CASE WHEN TRIM(off) = '' THEN 1 ELSE 0 END) AS blank_off_rows,
        MIN(LENGTH(off)) AS minimum_string_length,
        MAX(LENGTH(off)) AS maximum_string_length
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

off_inventory

,data_like_runner_rows,distinct_raw_off_values,blank_off_rows,minimum_string_length,maximum_string_length
0,1851285,1380,0,4,5


In [2]:
# Make the local src-layout package importable from a fresh notebook kernel.

import sys
from pathlib import Path

current_path = Path.cwd().resolve()

project_root = next(
    (
        path
        for path in (current_path, *current_path.parents)
        if (path / "pyproject.toml").exists()
        and (path / "src" / "inside_rails").exists()
    ),
    None,
)

if project_root is None:
    raise RuntimeError(
        f"Could not locate the project root from notebook directory: {current_path}"
    )

src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Project root:", project_root)
print("Added package path:", src_path)

Project root: /home/rob/Documents/inside-rails-horse-racing
Added package path: /home/rob/Documents/inside-rails-horse-racing/src


## Raw value formats

The initial inventory found no blank `off` values.

All observed values are four or five characters long. The next step examines the exact character patterns and representative raw values before treating them as times.

In [3]:
# Group raw values by length and broad character pattern.
#
# The pattern replaces digits with "9" while preserving punctuation and
# whitespace. This lets us distinguish structures such as "9:99" and "99:99"
# without yet assuming that every value is a valid clock time.

off_format_profile = pd.read_sql_query(
    f"""
    WITH distinct_off AS (
        SELECT DISTINCT off
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    ),
    classified AS (
        SELECT
            off,
            LENGTH(off) AS string_length,
            REPLACE(
                REPLACE(
                    REPLACE(
                        REPLACE(
                            REPLACE(
                                REPLACE(
                                    REPLACE(
                                        REPLACE(
                                            REPLACE(
                                                REPLACE(off, '0', '9'),
                                            '1', '9'),
                                        '2', '9'),
                                    '3', '9'),
                                '4', '9'),
                            '5', '9'),
                        '6', '9'),
                    '7', '9'),
                '8', '9'),
            '9', '9') AS character_pattern
        FROM distinct_off
    )
    SELECT
        string_length,
        character_pattern,
        COUNT(*) AS distinct_raw_values,
        MIN(off) AS example_minimum,
        MAX(off) AS example_maximum
    FROM classified
    GROUP BY
        string_length,
        character_pattern
    ORDER BY
        string_length,
        character_pattern
    """,
    connection,
)

off_format_profile

,string_length,character_pattern,distinct_raw_values,example_minimum,example_maximum
0,4,9:99,540,1:00,9:59
1,5,99:99,840,00:01,23:59


### Clock-component validity

All distinct raw values follow either the `H:MM` or `HH:MM` character pattern.

The next check separates the hour and minute components and tests whether they fall within ordinary 24-hour clock ranges. This validates the syntax only; it does not establish whether `off` is scheduled time, actual starting time or local time.

In [4]:
# Parse the two apparent clock components without altering the raw value.
#
# Because the previous profile established a single colon-separated structure,
# SQLite substrings can now be used for this bounded validity check.

off_clock_validity = pd.read_sql_query(
    f"""
    WITH distinct_off AS (
        SELECT DISTINCT off
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    ),
    parsed AS (
        SELECT
            off,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value,
            CAST(SUBSTR(off, INSTR(off, ':') + 1) AS INTEGER) AS minute_value
        FROM distinct_off
    )
    SELECT
        COUNT(*) AS distinct_raw_values,
        SUM(
            CASE
                WHEN hour_value BETWEEN 0 AND 23
                 AND minute_value BETWEEN 0 AND 59
                THEN 1 ELSE 0
            END
        ) AS valid_24_hour_values,
        SUM(
            CASE
                WHEN hour_value NOT BETWEEN 0 AND 23
                  OR minute_value NOT BETWEEN 0 AND 59
                THEN 1 ELSE 0
            END
        ) AS invalid_24_hour_values,
        MIN(hour_value) AS minimum_hour,
        MAX(hour_value) AS maximum_hour,
        MIN(minute_value) AS minimum_minute,
        MAX(minute_value) AS maximum_minute
    FROM parsed
    """,
    connection,
)

off_clock_validity

,distinct_raw_values,valid_24_hour_values,invalid_24_hour_values,minimum_hour,maximum_hour,minimum_minute,maximum_minute
0,1380,1380,0,0,23,0,59


## Consistency of the candidate provisional race key

The raw `off` field is uniformly time-shaped and every distinct value falls within ordinary 24-hour clock ranges.

This supports deriving clock components for profiling, but it does not establish whether the value is scheduled, advertised or actual off-time.

Because `off` forms part of the current provisional race key, the next step tests whether any `date + course + off` group contains multiple distinct source race references or race names. Such groups could indicate candidate-key collisions, source inconsistencies or descriptive variation requiring inspection.

In [5]:
# Profile the apparent internal consistency of each date + course + off group.
#
# race_id is not trusted as a unique race identifier, and race_name is
# descriptive rather than authoritative. However, multiple distinct values
# within one candidate group are useful warning signals.

candidate_race_consistency = pd.read_sql_query(
    f"""
    WITH candidate_groups AS (
        SELECT
            date,
            course,
            off,
            COUNT(*) AS runner_rows,
            COUNT(DISTINCT race_id) AS distinct_source_race_ids,
            COUNT(DISTINCT race_name) AS distinct_race_names
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    )
    SELECT
        COUNT(*) AS provisional_races,
        SUM(
            CASE
                WHEN distinct_source_race_ids = 1
                 AND distinct_race_names = 1
                THEN 1 ELSE 0
            END
        ) AS internally_consistent_groups,
        SUM(
            CASE
                WHEN distinct_source_race_ids > 1
                THEN 1 ELSE 0
            END
        ) AS groups_with_multiple_source_race_ids,
        SUM(
            CASE
                WHEN distinct_race_names > 1
                THEN 1 ELSE 0
            END
        ) AS groups_with_multiple_race_names,
        MAX(distinct_source_race_ids) AS maximum_source_race_ids_in_group,
        MAX(distinct_race_names) AS maximum_race_names_in_group
    FROM candidate_groups
    """,
    connection,
)

candidate_race_consistency

,provisional_races,internally_consistent_groups,groups_with_multiple_source_race_ids,groups_with_multiple_race_names,maximum_source_race_ids_in_group,maximum_race_names_in_group
0,189043,189043,0,0,1,1


### Duplicate off-times within a meeting

Every observed `date + course + off` group contains exactly one distinct source race reference and one distinct race name.

This provides strong empirical support for the current provisional race key within the available source population. It does not prove that the key is universally valid outside this database.

The next step examines whether a course can stage multiple races with the same raw `off` value on the same source date. Although the previous grouping result implies that such values do not split across different source races, a direct meeting-level profile will document the result and identify the number and scale of affected meetings, if any.

In [6]:
# Check directly whether distinct source races at the same course and date
# share an identical raw off value.
#
# race_id is not globally unique, but within a date-and-course meeting it is
# useful here as a source-level discriminator between apparent races.

meeting_off_duplication = pd.read_sql_query(
    f"""
    WITH source_races AS (
        SELECT DISTINCT
            date,
            course,
            race_id,
            race_name,
            off
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    ),
    meeting_profile AS (
        SELECT
            date,
            course,
            COUNT(*) AS source_races,
            COUNT(DISTINCT off) AS distinct_off_values
        FROM source_races
        GROUP BY
            date,
            course
    )
    SELECT
        COUNT(*) AS meetings,
        SUM(
            CASE
                WHEN source_races = distinct_off_values
                THEN 1 ELSE 0
            END
        ) AS meetings_with_unique_off_per_race,
        SUM(
            CASE
                WHEN source_races > distinct_off_values
                THEN 1 ELSE 0
            END
        ) AS meetings_with_repeated_off_values,
        MAX(source_races - distinct_off_values) AS maximum_duplicate_off_count,
        MAX(source_races) AS maximum_races_in_meeting
    FROM meeting_profile
    """,
    connection,
)

meeting_off_duplication

,meetings,meetings_with_unique_off_per_race,meetings_with_repeated_off_values,maximum_duplicate_off_count,maximum_races_in_meeting
0,33146,33146,0,0,15


## Race ordering within meetings

No meeting contains repeated raw `off` values across distinct source races.

The next analysis converts each syntactically valid `off` value into minutes after midnight and orders races by their physical source appearance. It then checks whether clock time ever moves backwards within a `date + course` meeting.

A backward movement may indicate:

* a meeting crossing midnight;
* a source-date convention that groups post-midnight races with the preceding racing day;
* source rows not being stored in race-time order;
* or another source-specific ordering convention.

The result will be treated as a diagnostic signal rather than proof of date rollover.

In [7]:
# Compare clock-time order with the physical source order of provisional races.
#
# MIN(rowid) gives each candidate race its earliest physical source position.
# A negative change in off_minutes means the displayed clock time moved
# backwards relative to the previous race in that date-and-course meeting.

meeting_time_order_profile = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            MIN(rowid) AS first_source_rowid,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) * 60
                + CAST(SUBSTR(off, INSTR(off, ':') + 1) AS INTEGER)
                AS off_minutes
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    ordered_races AS (
        SELECT
            date,
            course,
            off,
            first_source_rowid,
            off_minutes,
            LAG(off_minutes) OVER (
                PARTITION BY date, course
                ORDER BY first_source_rowid
            ) AS previous_off_minutes
        FROM provisional_races
    ),
    meeting_profile AS (
        SELECT
            date,
            course,
            COUNT(*) AS races,
            SUM(
                CASE
                    WHEN previous_off_minutes IS NOT NULL
                     AND off_minutes < previous_off_minutes
                    THEN 1 ELSE 0
                END
            ) AS backward_time_steps
        FROM ordered_races
        GROUP BY
            date,
            course
    )
    SELECT
        COUNT(*) AS meetings,
        SUM(
            CASE WHEN backward_time_steps = 0 THEN 1 ELSE 0 END
        ) AS meetings_without_backward_steps,
        SUM(
            CASE WHEN backward_time_steps > 0 THEN 1 ELSE 0 END
        ) AS meetings_with_backward_steps,
        SUM(backward_time_steps) AS total_backward_steps,
        MAX(backward_time_steps) AS maximum_backward_steps_in_meeting
    FROM meeting_profile
    """,
    connection,
)

meeting_time_order_profile

,meetings,meetings_without_backward_steps,meetings_with_backward_steps,total_backward_steps,maximum_backward_steps_in_meeting
0,33146,7692,25454,77829,9


### Physical row order is not reliable chronology

Most meetings show at least one backward movement in clock time when races are ordered by their earliest physical source `rowid`.

The scale of the result is inconsistent with treating every backward step as a midnight rollover. Instead, it demonstrates that physical source order cannot safely be assumed to represent race-time order.

The next step displays representative high-backward-step meetings so that the relationship between source order and clock-time order can be inspected directly.

In [8]:
# Inspect races from meetings with the most backward clock-time steps.
#
# This keeps both the earliest source row position and the parsed clock value
# visible. The output is descriptive only and does not impose chronology.

backward_step_examples = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            race_id,
            race_name,
            MIN(rowid) AS first_source_rowid,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) * 60
                + CAST(SUBSTR(off, INSTR(off, ':') + 1) AS INTEGER)
                AS off_minutes
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off,
            race_id,
            race_name
    ),
    ordered_races AS (
        SELECT
            *,
            LAG(off_minutes) OVER (
                PARTITION BY date, course
                ORDER BY first_source_rowid
            ) AS previous_off_minutes
        FROM provisional_races
    ),
    meeting_profile AS (
        SELECT
            date,
            course,
            SUM(
                CASE
                    WHEN previous_off_minutes IS NOT NULL
                     AND off_minutes < previous_off_minutes
                    THEN 1 ELSE 0
                END
            ) AS backward_time_steps
        FROM ordered_races
        GROUP BY
            date,
            course
    ),
    selected_meetings AS (
        SELECT
            date,
            course,
            backward_time_steps
        FROM meeting_profile
        WHERE backward_time_steps > 0
        ORDER BY
            backward_time_steps DESC,
            date,
            course
        LIMIT 5
    )
    SELECT
        r.date,
        r.course,
        m.backward_time_steps,
        r.first_source_rowid,
        r.off,
        r.off_minutes,
        r.race_id,
        r.race_name
    FROM provisional_races AS r
    INNER JOIN selected_meetings AS m
        ON r.date = m.date
       AND r.course = m.course
    ORDER BY
        m.backward_time_steps DESC,
        r.date,
        r.course,
        r.first_source_rowid
    """,
    connection,
)

backward_step_examples

,date,course,backward_time_steps,first_source_rowid,off,off_minutes,race_id,race_name
0,2017-03-05,Auteuil (FR),9,338580,11:10,670,670348,Prix Rohan (Hurdle) () (4yo+) (Turf)
1,2017-03-05,Auteuil (FR),9,338584,10:40,640,670347,Prix Rivoli (Chase) (AQPS Conditions) (4yo) (T...
2,2017-03-05,Auteuil (FR),9,338611,1:10,70,670352,Prix Tofano (Chase) (Conditions) (5yo+) (Turf)
3,2017-03-05,Auteuil (FR),9,338616,12:30,750,670351,Prix Jean Doumen (Hurdle) (Conditions) (5yo+) ...
4,2017-03-05,Auteuil (FR),9,338626,11:50,710,670350,Prix Agitato (Chase) (Conditions) (4yo) (Turf)
5,2017-03-05,Auteuil (FR),9,338711,4:50,290,670362,Prix Bougie (Hurdle) (Handicap) (5yo+) (Turf)
6,2017-03-05,Auteuil (FR),9,338725,4:20,260,670361,Prix Pont dIena (Chase) (Handicap) (5yo+) (Turf)
7,2017-03-05,Auteuil (FR),9,338768,3:50,230,670360,Prix Beugnot (Hurdle) (Listed Handicap) (5yo+)...
8,2017-03-05,Auteuil (FR),9,338769,3:20,200,670356,Prix Robert de Clermont-Tonnerre (Chase) (5yo...
9,2017-03-05,Auteuil (FR),9,338774,2:45,165,670354,Prix Juigne (Hurdle) (5yo+) (Turf)


### Possible mixed clock conventions

The inspected meetings confirm that physical source order is irregular, but they also reveal a more important issue.

Some meetings contain sequences such as:

`11:50 → 12:30 → 1:10 → 1:40 → 2:15`

These values are consistent with an afternoon meeting expressed using a 12-hour clock without an `AM` or `PM` suffix. Interpreting `1:10` mechanically as 01:10 would therefore create a false midnight rollover.

Other raw values use hours from 13 through 23, demonstrating that the source also contains apparent 24-hour-clock values.

The next step profiles clock-hour usage by course to determine whether these conventions are course-specific, mixed within courses or otherwise jurisdiction-dependent.

In [9]:
# Profile apparent clock conventions at course level.
#
# Hours above 12 provide direct evidence of 24-hour-style values.
# Hours from 1 to 12 alone may represent either morning times or an
# unlabelled 12-hour clock, so they are classified only as ambiguous here.

course_clock_profile = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    )
    SELECT
        course,
        COUNT(*) AS provisional_races,
        COUNT(DISTINCT date) AS meeting_dates,
        MIN(hour_value) AS minimum_hour,
        MAX(hour_value) AS maximum_hour,
        SUM(CASE WHEN hour_value = 0 THEN 1 ELSE 0 END) AS hour_zero_races,
        SUM(CASE WHEN hour_value BETWEEN 1 AND 12 THEN 1 ELSE 0 END)
            AS hour_1_to_12_races,
        SUM(CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END)
            AS hour_13_to_23_races,
        CASE
            WHEN SUM(CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END) > 0
             AND SUM(CASE WHEN hour_value BETWEEN 1 AND 12 THEN 1 ELSE 0 END) > 0
                THEN 'contains_low_and_high_hours'
            WHEN SUM(CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END) > 0
                THEN 'high_hours_only'
            WHEN SUM(CASE WHEN hour_value = 0 THEN 1 ELSE 0 END) > 0
                THEN 'zero_and_low_hours_only'
            ELSE 'hours_1_to_12_only'
        END AS observed_hour_profile
    FROM provisional_races
    GROUP BY course
    ORDER BY
        hour_13_to_23_races DESC,
        provisional_races DESC,
        course
    """,
    connection,
)

course_clock_profile

,course,provisional_races,meeting_dates,minimum_hour,maximum_hour,hour_zero_races,hour_1_to_12_races,hour_13_to_23_races,observed_hour_profile
0,Wolverhampton (AW),7042,930,1,21,0,6529,513,contains_low_and_high_hours
1,Southwell (AW),3816,515,1,21,0,3406,410,contains_low_and_high_hours
2,Newcastle (AW),4409,583,1,20,0,4014,395,contains_low_and_high_hours
3,Dundalk (AW),279,37,13,20,0,0,279,high_hours_only
4,Lingfield (AW),4803,742,1,20,0,4545,258,contains_low_and_high_hours
...,...,...,...,...,...,...,...,...,...
523,Wagga Wagga (AUS),1,1,5,5,0,1,0,hours_1_to_12_only
524,Wangaratta (AUS),1,1,3,3,0,1,0,hours_1_to_12_only
525,Waregem (BEL),1,1,3,3,0,1,0,hours_1_to_12_only
526,Werribee (AUS),1,1,2,2,0,1,0,hours_1_to_12_only


### Distribution of course-level hour profiles

Course-level results show several distinct patterns:

* some courses contain both hours `1–12` and explicit hours `13–23`;
* some contain only explicit high hours;
* many contain only hours `1–12`;
* a smaller group also contains hour `0`.

These patterns describe the observed data only. Courses restricted to hours `1–12` remain ambiguous because those values could reflect morning racing, a 12-hour display convention, or both.

The next step summarises the scale of each course-level profile.

In [10]:
# Summarise the course-level hour profiles by number of courses,
# provisional races and meeting dates represented.

course_clock_profile_summary = (
    course_clock_profile
    .groupby("observed_hour_profile", as_index=False)
    .agg(
        courses=("course", "nunique"),
        provisional_races=("provisional_races", "sum"),
        meeting_dates=("meeting_dates", "sum"),
    )
    .sort_values(
        ["provisional_races", "courses"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

course_clock_profile_summary

,observed_hour_profile,courses,provisional_races,meeting_dates
0,contains_low_and_high_hours,102,113238,16094
1,hours_1_to_12_only,369,75009,16810
2,high_hours_only,55,785,234
3,zero_and_low_hours_only,2,11,8


### Meeting-level hour profiles

Course-level classification is too broad to determine the meaning of an individual `off` value.

More than half of all provisional races occur at courses that contain both low hours and explicit hours above 12 somewhere in the source history. This may reflect different conventions across dates, cards or source records rather than a single stable course rule.

The next step therefore classifies each `date + course` meeting by its observed hour range.

In [11]:
# Classify each meeting by the hour values present on that specific date.
#
# This narrows the analysis from long-term course behaviour to the level at
# which race times should form one internally coherent card.

meeting_clock_profile = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    meeting_profiles AS (
        SELECT
            date,
            course,
            COUNT(*) AS provisional_races,
            MIN(hour_value) AS minimum_hour,
            MAX(hour_value) AS maximum_hour,
            SUM(CASE WHEN hour_value = 0 THEN 1 ELSE 0 END) AS hour_zero_races,
            SUM(CASE WHEN hour_value BETWEEN 1 AND 12 THEN 1 ELSE 0 END)
                AS hour_1_to_12_races,
            SUM(CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END)
                AS hour_13_to_23_races,
            CASE
                WHEN SUM(
                    CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END
                ) > 0
                 AND SUM(
                    CASE WHEN hour_value BETWEEN 1 AND 12 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'contains_low_and_high_hours'
                WHEN SUM(
                    CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'high_hours_only'
                WHEN SUM(
                    CASE WHEN hour_value = 0 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'zero_and_low_hours_only'
                ELSE 'hours_1_to_12_only'
            END AS observed_hour_profile
        FROM provisional_races
        GROUP BY
            date,
            course
    )
    SELECT
        observed_hour_profile,
        COUNT(*) AS meetings,
        SUM(provisional_races) AS provisional_races,
        MIN(provisional_races) AS minimum_races_per_meeting,
        MAX(provisional_races) AS maximum_races_per_meeting
    FROM meeting_profiles
    GROUP BY observed_hour_profile
    ORDER BY
        provisional_races DESC,
        meetings DESC
    """,
    connection,
)

meeting_clock_profile

,observed_hour_profile,meetings,provisional_races,minimum_races_per_meeting,maximum_races_per_meeting
0,hours_1_to_12_only,31686,179474,1,15
1,high_hours_only,1013,6308,1,10
2,contains_low_and_high_hours,432,3245,2,10
3,zero_and_low_hours_only,15,16,1,2


### Meetings containing both low and high hours

Only 432 meetings contain both hours `1–12` and explicit hours `13–23`.

These meetings are especially useful because they can show whether low-hour values and high-hour values form one plausible chronological card, or whether the source mixes temporal conventions within the same meeting.

The next step inspects representative mixed-hour meetings ordered by parsed clock time.

In [12]:
# Inspect representative meetings containing both low and high hour values.
#
# Ordering by parsed minutes shows the apparent clock sequence directly.
# The output remains descriptive: no AM/PM adjustment or rollover is applied.

mixed_hour_meeting_examples = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            race_id,
            race_name,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) * 60
                + CAST(SUBSTR(off, INSTR(off, ':') + 1) AS INTEGER)
                AS off_minutes
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off,
            race_id,
            race_name
    ),
    mixed_meetings AS (
        SELECT
            date,
            course,
            COUNT(*) AS provisional_races,
            MIN(hour_value) AS minimum_hour,
            MAX(hour_value) AS maximum_hour
        FROM provisional_races
        GROUP BY
            date,
            course
        HAVING
            SUM(CASE WHEN hour_value BETWEEN 1 AND 12 THEN 1 ELSE 0 END) > 0
            AND
            SUM(CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END) > 0
    ),
    selected_meetings AS (
        SELECT
            date,
            course,
            provisional_races,
            minimum_hour,
            maximum_hour
        FROM mixed_meetings
        ORDER BY
            provisional_races DESC,
            date,
            course
        LIMIT 10
    )
    SELECT
        r.date,
        r.course,
        m.provisional_races,
        m.minimum_hour,
        m.maximum_hour,
        r.off,
        r.off_minutes,
        r.race_id,
        r.race_name
    FROM provisional_races AS r
    INNER JOIN selected_meetings AS m
        ON r.date = m.date
       AND r.course = m.course
    ORDER BY
        r.date,
        r.course,
        r.off_minutes
    """,
    connection,
)

mixed_hour_meeting_examples

,date,course,provisional_races,minimum_hour,maximum_hour,off,off_minutes,race_id,race_name
0,2025-10-15,Happy Valley,9,11,15,11:40,700,905870,Ngau Chi Wan Handicap (3yo+) (Course A) (Turf)
1,2025-10-15,Happy Valley,9,11,15,12:10,730,905871,Hung Luen Handicap (Div I) (3yo+) (Course A) (...
2,2025-10-15,Happy Valley,9,11,15,12:40,760,905872,Fung Mo Handicap (Div I) (3yo+) (Course A) (Turf)
3,2025-10-15,Happy Valley,9,11,15,13:10,790,905873,Fung Mo Handicap (Div II) (3yo+) (Course A) (T...
4,2025-10-15,Happy Valley,9,11,15,13:40,820,905874,Hung Leun Handicap (Div II) (3yo+) (Course A) ...
...,...,...,...,...,...,...,...,...,...
88,2026-05-19,Chantilly,10,12,18,15:50,950,920846,Prix Texanita (Turf)
89,2026-05-19,Chantilly,10,12,18,16:25,985,920852,Prix de la Foret dErmenonville (Handicap) (All...
90,2026-05-19,Chantilly,10,12,18,17:00,1020,920853,Prix de la Theve (Conditions) (Turf)
91,2026-05-19,Chantilly,10,12,18,17:37,1057,920854,Prix de la Foret du Lys (Handicap) (All Weathe...


### Possible change in source convention over time

Representative mixed-hour meetings form coherent chronological sequences when the raw values are interpreted directly as 24-hour clock times.

This contrasts with older examples where afternoon races after noon were represented as `1:10`, `2:15` and similar low-hour values.

The difference may reflect a change in source formatting over time rather than a stable course or jurisdiction rule.

The next step profiles meeting-level hour patterns by calendar year.

In [13]:
# Summarise meeting-level clock profiles by source calendar year.
#
# This tests whether the observed hour conventions are associated with
# different periods in the source history. It remains a format profile only:
# no claim is yet made about the exact date or cause of any transition.

meeting_clock_profile_by_year = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    meeting_profiles AS (
        SELECT
            date,
            course,
            COUNT(*) AS provisional_races,
            CASE
                WHEN SUM(
                    CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END
                ) > 0
                 AND SUM(
                    CASE WHEN hour_value BETWEEN 1 AND 12 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'contains_low_and_high_hours'
                WHEN SUM(
                    CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'high_hours_only'
                WHEN SUM(
                    CASE WHEN hour_value = 0 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'zero_and_low_hours_only'
                ELSE 'hours_1_to_12_only'
            END AS observed_hour_profile
        FROM provisional_races
        GROUP BY
            date,
            course
    )
    SELECT
        SUBSTR(date, 1, 4) AS calendar_year,
        observed_hour_profile,
        COUNT(*) AS meetings,
        SUM(provisional_races) AS provisional_races
    FROM meeting_profiles
    GROUP BY
        SUBSTR(date, 1, 4),
        observed_hour_profile
    ORDER BY
        calendar_year,
        observed_hour_profile
    """,
    connection,
)

meeting_clock_profile_by_year

,calendar_year,observed_hour_profile,meetings,provisional_races
0,2015,hours_1_to_12_only,3220,16609
1,2016,hours_1_to_12_only,2968,16235
2,2017,hours_1_to_12_only,3040,17110
3,2018,hours_1_to_12_only,3167,17546
4,2019,hours_1_to_12_only,3172,17345
5,2020,hours_1_to_12_only,1953,11875
6,2021,hours_1_to_12_only,3023,17706
7,2022,hours_1_to_12_only,3002,17350
8,2023,hours_1_to_12_only,2674,16119
9,2024,hours_1_to_12_only,2924,17170


### Timing of the format transition

The year-level profile shows a clear discontinuity:

* all meetings from 2015 through 2024 use only hours `1–12`;
* explicit hours `13–23` first appear in 2025;
* by 2026, high-hour and mixed-hour meetings dominate.

This strongly suggests a change in source formatting or ingestion convention rather than a timeless semantic property of `off`.

The next step profiles meeting-level clock patterns by calendar month from 2025 onward to locate the transition more precisely.

In [14]:
# Profile the apparent clock-format transition by calendar month.
#
# Restricting to 2025 onward keeps the output focused on the period where
# explicit high-hour values first appear.

meeting_clock_profile_by_month = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date >= '2025-01-01'
        GROUP BY
            date,
            course,
            off
    ),
    meeting_profiles AS (
        SELECT
            date,
            course,
            COUNT(*) AS provisional_races,
            CASE
                WHEN SUM(
                    CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END
                ) > 0
                 AND SUM(
                    CASE WHEN hour_value BETWEEN 1 AND 12 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'contains_low_and_high_hours'
                WHEN SUM(
                    CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'high_hours_only'
                WHEN SUM(
                    CASE WHEN hour_value = 0 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'zero_and_low_hours_only'
                ELSE 'hours_1_to_12_only'
            END AS observed_hour_profile
        FROM provisional_races
        GROUP BY
            date,
            course
    )
    SELECT
        SUBSTR(date, 1, 7) AS calendar_month,
        observed_hour_profile,
        COUNT(*) AS meetings,
        SUM(provisional_races) AS provisional_races
    FROM meeting_profiles
    GROUP BY
        SUBSTR(date, 1, 7),
        observed_hour_profile
    ORDER BY
        calendar_month,
        observed_hour_profile
    """,
    connection,
)

meeting_clock_profile_by_month

,calendar_month,observed_hour_profile,meetings,provisional_races
0,2025-01,hours_1_to_12_only,187,1117
1,2025-02,hours_1_to_12_only,202,1138
2,2025-03,hours_1_to_12_only,232,1392
3,2025-04,hours_1_to_12_only,250,1533
4,2025-05,hours_1_to_12_only,301,1790
5,2025-06,hours_1_to_12_only,246,1473
6,2025-07,hours_1_to_12_only,244,1442
7,2025-08,hours_1_to_12_only,258,1445
8,2025-09,hours_1_to_12_only,251,1507
9,2025-10,contains_low_and_high_hours,32,247


### First appearance of explicit high-hour values

The monthly profile places the source-format transition in October 2025.

Before October 2025, every observed meeting contains only hours `1–12`. From October onward, explicit hours `13–23` appear and rapidly become common.

The next step identifies the earliest meetings containing explicit high-hour values and compares them with low-hour-only meetings from the same transition period.

In [15]:
# Identify the earliest meetings containing explicit hours above 12.
#
# A small comparison set of low-hour-only meetings from the same dates is
# included where available. This may show whether the transition occurred
# globally, by jurisdiction, by course or incrementally within the source.

first_high_hour_meetings = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            race_id,
            race_name,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off,
            race_id,
            race_name
    ),
    meeting_profiles AS (
        SELECT
            date,
            course,
            COUNT(*) AS provisional_races,
            MIN(hour_value) AS minimum_hour,
            MAX(hour_value) AS maximum_hour,
            CASE
                WHEN SUM(
                    CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END
                ) > 0
                 AND SUM(
                    CASE WHEN hour_value BETWEEN 1 AND 12 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'contains_low_and_high_hours'
                WHEN SUM(
                    CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'high_hours_only'
                WHEN SUM(
                    CASE WHEN hour_value = 0 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'zero_and_low_hours_only'
                ELSE 'hours_1_to_12_only'
            END AS observed_hour_profile
        FROM provisional_races
        GROUP BY
            date,
            course
    ),
    earliest_high_dates AS (
        SELECT DISTINCT date
        FROM meeting_profiles
        WHERE observed_hour_profile IN (
            'contains_low_and_high_hours',
            'high_hours_only'
        )
        ORDER BY date
        LIMIT 10
    )
    SELECT
        m.date,
        m.course,
        m.observed_hour_profile,
        m.provisional_races,
        m.minimum_hour,
        m.maximum_hour
    FROM meeting_profiles AS m
    INNER JOIN earliest_high_dates AS d
        ON m.date = d.date
    ORDER BY
        m.date,
        CASE
            WHEN m.observed_hour_profile IN (
                'contains_low_and_high_hours',
                'high_hours_only'
            )
            THEN 0 ELSE 1
        END,
        m.course
    """,
    connection,
)

first_high_hour_meetings

,date,course,observed_hour_profile,provisional_races,minimum_hour,maximum_hour
0,2025-10-15,Happy Valley,contains_low_and_high_hours,9,11,15
1,2025-10-15,Kempton (AW),high_hours_only,8,16,20
2,2025-10-15,Nottingham,high_hours_only,8,13,17
3,2025-10-15,Punchestown,high_hours_only,7,13,17
4,2025-10-15,Wetherby,high_hours_only,7,13,16
...,...,...,...,...,...,...
82,2025-10-24,Keeneland,high_hours_only,1,22,22
83,2025-10-24,Newbury,high_hours_only,8,13,17
84,2025-10-24,Sligo,high_hours_only,7,14,17
85,2025-10-24,Southwell (AW),high_hours_only,9,16,20


### Candidate source-format boundary

The earliest explicit high-hour meetings all occur on 15 October 2025.

The simultaneous appearance across unrelated courses and jurisdictions strongly suggests a source-system or ingestion-format change rather than independent local clock conventions.

However, low-hour-only meetings continue to appear after that date. The next step quantifies the daily mix around the apparent boundary and identifies the final pre-boundary and post-boundary low-hour-only records.

In [16]:
# Profile daily meeting formats around the apparent 15 October 2025 boundary.
#
# This checks whether the transition is abrupt at source level and measures
# how much low-hour-only data persists after explicit high hours first appear.

daily_clock_profile_transition = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date BETWEEN '2025-10-01' AND '2025-10-31'
        GROUP BY
            date,
            course,
            off
    ),
    meeting_profiles AS (
        SELECT
            date,
            course,
            COUNT(*) AS provisional_races,
            CASE
                WHEN SUM(
                    CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END
                ) > 0
                 AND SUM(
                    CASE WHEN hour_value BETWEEN 1 AND 12 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'contains_low_and_high_hours'
                WHEN SUM(
                    CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'high_hours_only'
                WHEN SUM(
                    CASE WHEN hour_value = 0 THEN 1 ELSE 0 END
                ) > 0
                    THEN 'zero_and_low_hours_only'
                ELSE 'hours_1_to_12_only'
            END AS observed_hour_profile
        FROM provisional_races
        GROUP BY
            date,
            course
    )
    SELECT
        date,
        COUNT(*) AS meetings,
        SUM(
            CASE
                WHEN observed_hour_profile = 'hours_1_to_12_only'
                THEN 1 ELSE 0
            END
        ) AS low_hour_only_meetings,
        SUM(
            CASE
                WHEN observed_hour_profile = 'high_hours_only'
                THEN 1 ELSE 0
            END
        ) AS high_hour_only_meetings,
        SUM(
            CASE
                WHEN observed_hour_profile = 'contains_low_and_high_hours'
                THEN 1 ELSE 0
            END
        ) AS mixed_hour_meetings,
        SUM(
            CASE
                WHEN observed_hour_profile = 'zero_and_low_hours_only'
                THEN 1 ELSE 0
            END
        ) AS zero_and_low_hour_meetings,
        SUM(provisional_races) AS provisional_races
    FROM meeting_profiles
    GROUP BY date
    ORDER BY date
    """,
    connection,
)

daily_clock_profile_transition

,date,meetings,low_hour_only_meetings,high_hour_only_meetings,mixed_hour_meetings,zero_and_low_hour_meetings,provisional_races
0,2025-10-01,9,9,0,0,0,62
1,2025-10-02,9,9,0,0,0,69
2,2025-10-03,8,8,0,0,0,46
3,2025-10-04,17,17,0,0,0,99
4,2025-10-05,11,11,0,0,0,48
5,2025-10-06,7,7,0,0,0,45
6,2025-10-07,6,6,0,0,0,44
7,2025-10-08,6,6,0,0,0,47
8,2025-10-09,8,8,0,0,0,58
9,2025-10-10,8,8,0,0,0,48


### Post-boundary low-hour-only meetings

The daily profile supports 15 October 2025 as the first date of a source-format change.

The change is abrupt but incomplete. After the boundary, some meetings still contain only hours `1–12`.

These residual meetings may represent:

* jurisdictions still supplied in an older format;
* isolated imported races rather than complete cards;
* meetings occurring entirely before 13:00;
* or inconsistent source processing.

The next step profiles these post-boundary low-hour-only meetings directly.

In [17]:
# Inspect meetings that remain low-hour-only after the apparent format boundary.
#
# The race count and observed hour range help distinguish full afternoon cards
# from sparse or genuinely morning-only records.

post_boundary_low_hour_meetings = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            race_id,
            race_name,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date >= '2025-10-15'
        GROUP BY
            date,
            course,
            off,
            race_id,
            race_name
    ),
    meeting_profiles AS (
        SELECT
            date,
            course,
            COUNT(*) AS provisional_races,
            MIN(hour_value) AS minimum_hour,
            MAX(hour_value) AS maximum_hour,
            SUM(
                CASE WHEN hour_value BETWEEN 13 AND 23 THEN 1 ELSE 0 END
            ) AS high_hour_races,
            GROUP_CONCAT(off, ', ') AS observed_off_values
        FROM provisional_races
        GROUP BY
            date,
            course
    )
    SELECT
        date,
        course,
        provisional_races,
        minimum_hour,
        maximum_hour,
        observed_off_values
    FROM meeting_profiles
    WHERE high_hour_races = 0
      AND minimum_hour >= 1
    ORDER BY
        date,
        course
    """,
    connection,
)

post_boundary_low_hour_meetings

,date,course,provisional_races,minimum_hour,maximum_hour,observed_off_values
0,2025-10-15,Caulfield,1,7,7,07:10
1,2025-10-15,Nantes,1,12,12,12:27
2,2025-10-17,Baden-Baden,1,12,12,12:15
3,2025-10-18,Caulfield,7,2,7,"02:50, 03:25, 05:10, 05:45, 06:30, 07:15, 07:50"
4,2025-10-18,Ellerslie,1,4,4,04:59
...,...,...,...,...,...,...
240,2026-05-23,Kyoto,1,7,7,07:45
241,2026-05-24,Sha Tin,11,5,10,"05:30, 06:00, 06:30, 07:00, 07:30, 08:00, 08:3..."
242,2026-05-24,Tokyo,1,7,7,07:40
243,2026-05-26,Auteuil,8,8,12,"08:44, 09:16, 09:48, 10:20, 10:52, 11:28, 11:5..."


### Zero-padding as a format-transition signal

Post-boundary low-hour-only meetings frequently contain values such as `07:10`, `04:59` and `01:07`.

These are structurally different from earlier low-hour values such as `7:10`, `4:59` and `1:07`.

The apparent October 2025 transition may therefore involve a change from variable-width clock text to fixed-width `HH:MM` formatting, alongside the introduction of explicit afternoon hours above 12.

The next step profiles four-character and five-character `off` values over time.

In [18]:
# Profile raw string width by calendar month.
#
# Four-character values have the form H:MM.
# Five-character values have the form HH:MM.
#
# This directly tests whether the October 2025 transition is associated with
# fixed-width zero-padded clock formatting.

off_length_by_month = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            LENGTH(off) AS string_length
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    )
    SELECT
        SUBSTR(date, 1, 7) AS calendar_month,
        SUM(CASE WHEN string_length = 4 THEN 1 ELSE 0 END)
            AS four_character_races,
        SUM(CASE WHEN string_length = 5 THEN 1 ELSE 0 END)
            AS five_character_races,
        COUNT(*) AS provisional_races,
        ROUND(
            100.0 * SUM(CASE WHEN string_length = 5 THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS five_character_percentage
    FROM provisional_races
    GROUP BY SUBSTR(date, 1, 7)
    ORDER BY calendar_month
    """,
    connection,
)

off_length_by_month.tail(24)

,calendar_month,four_character_races,five_character_races,provisional_races,five_character_percentage
113,2024-06,1463,91,1554,5.86
114,2024-07,1380,48,1428,3.36
115,2024-08,1401,49,1450,3.38
116,2024-09,1384,95,1479,6.42
117,2024-10,1612,109,1721,6.33
118,2024-11,1179,304,1483,20.50
119,2024-12,990,279,1269,21.99
120,2025-01,961,156,1117,13.97
121,2025-02,1046,92,1138,8.08
122,2025-03,1294,98,1392,7.04


### Zero-padded low-hour values

String length alone cannot distinguish two different five-character forms:

* naturally five-character values such as `10:30`, `11:45` and `12:05`;
* zero-padded low-hour values such as `01:30`, `07:10` and `09:55`.

The second form is the clearest evidence of fixed-width `HH:MM` formatting.

The next step profiles zero-padded low-hour values over time and identifies their first appearance.

In [19]:
# Profile zero-padded low-hour values over time.
#
# Values beginning with "0" are unambiguous evidence of fixed-width HH:MM
# formatting because the earlier H:MM representation would omit that zero.

zero_padded_low_hours_by_month = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    )
    SELECT
        SUBSTR(date, 1, 7) AS calendar_month,
        SUM(
            CASE
                WHEN LENGTH(off) = 5
                 AND SUBSTR(off, 1, 1) = '0'
                THEN 1 ELSE 0
            END
        ) AS zero_padded_low_hour_races,
        COUNT(*) AS provisional_races,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN LENGTH(off) = 5
                     AND SUBSTR(off, 1, 1) = '0'
                    THEN 1 ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS zero_padded_percentage
    FROM provisional_races
    GROUP BY SUBSTR(date, 1, 7)
    HAVING zero_padded_low_hour_races > 0
    ORDER BY calendar_month
    """,
    connection,
)

zero_padded_low_hours_by_month

,calendar_month,zero_padded_low_hour_races,provisional_races,zero_padded_percentage
0,2025-10,53,1726,3.07
1,2025-11,115,1449,7.94
2,2025-12,66,1282,5.15
3,2026-01,83,1180,7.03
4,2026-02,110,1143,9.62
5,2026-03,123,1346,9.14
6,2026-04,92,1503,6.12
7,2026-05,81,1512,5.36


### Exact onset of fixed-width formatting

Zero-padded low-hour values first appear in October 2025 and do not occur anywhere earlier in the source.

This aligns with the first appearance of explicit hours above 12 and supports a common source-format transition rather than two unrelated changes.

The next step identifies the earliest zero-padded low-hour records and checks whether they begin on the same date as the explicit high-hour values.

In [20]:
# Identify the earliest zero-padded low-hour values.
#
# These records provide the most precise observed start date for fixed-width
# HH:MM formatting in the current source population.

first_zero_padded_low_hours = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            race_id,
            race_name
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off,
            race_id,
            race_name
    )
    SELECT
        date,
        course,
        off,
        race_id,
        race_name
    FROM provisional_races
    WHERE LENGTH(off) = 5
      AND SUBSTR(off, 1, 1) = '0'
    ORDER BY
        date,
        course,
        off
    LIMIT 25
    """,
    connection,
)

first_zero_padded_low_hours

,date,course,off,race_id,race_name
0,2025-10-15,Caulfield,07:10,906003,Sportsbet Coongy Cup Handicap) (3yo+) (Turf)
1,2025-10-18,Caulfield,02:50,906009,Sportsbet Classic (3yo) (Turf)
2,2025-10-18,Caulfield,03:25,906169,Schweppes Ethereal Stakes (3yo Fillies) (Turf)
3,2025-10-18,Caulfield,05:10,906012,Herald Sun Sprint Handicap) (3yo+) (Turf)
4,2025-10-18,Caulfield,05:45,906013,Schweppes Thousand Guineas (3yo Fillies) (Turf)
5,2025-10-18,Caulfield,06:30,906172,Sportsbet Moonga Stakes (4yo+) (Turf)
6,2025-10-18,Caulfield,07:15,906014,Sportsbet Caulfield Cup Handicap) (3yo+) (Turf)
7,2025-10-18,Caulfield,07:50,906175,Sharp EIT Solutions Tristarc Stakes (4yo+ Mare...
8,2025-10-18,Ellerslie,04:59,906171,Livamol Classic (3yo+) (Turf)
9,2025-10-18,Randwick,02:30,906168,Bisley Workwear Reginald Allen Quality Handica...


### Validation of the candidate format boundary

Both defining features of fixed-width 24-hour-style formatting first appear on 15 October 2025:

* zero-padded low hours such as `07:10`;
* explicit high hours such as `13:10` and `20:00`.

The next step tests whether this date cleanly separates the two observed raw-format regimes:

* before 15 October 2025: `H:MM` or naturally five-character `10:MM`–`12:MM`;
* from 15 October 2025 onward: fixed-width `HH:MM`.

In [21]:
# Test the proposed 15 October 2025 format boundary directly.
#
# Pre-boundary exceptions would be leading-zero values or hours above 12.
# Post-boundary exceptions would be any value not exactly five characters.

off_format_boundary_validation = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            LENGTH(off) AS string_length,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    )
    SELECT
        SUM(CASE WHEN date < '2025-10-15' THEN 1 ELSE 0 END)
            AS pre_boundary_races,
        SUM(
            CASE
                WHEN date < '2025-10-15'
                 AND (
                     SUBSTR(off, 1, 1) = '0'
                     OR hour_value > 12
                 )
                THEN 1 ELSE 0
            END
        ) AS pre_boundary_format_exceptions,
        SUM(CASE WHEN date >= '2025-10-15' THEN 1 ELSE 0 END)
            AS post_boundary_races,
        SUM(
            CASE
                WHEN date >= '2025-10-15'
                 AND string_length <> 5
                THEN 1 ELSE 0
            END
        ) AS post_boundary_format_exceptions,
        MIN(
            CASE
                WHEN date >= '2025-10-15'
                 AND string_length <> 5
                THEN date
            END
        ) AS first_post_boundary_exception_date,
        MAX(
            CASE
                WHEN date >= '2025-10-15'
                 AND string_length <> 5
                THEN date
            END
        ) AS last_post_boundary_exception_date
    FROM provisional_races
    """,
    connection,
)

off_format_boundary_validation

,pre_boundary_races,pre_boundary_format_exceptions,post_boundary_races,post_boundary_format_exceptions,first_post_boundary_exception_date,last_post_boundary_exception_date
0,178691,0,10352,0,None,None


### Confirmed raw-format regimes

The proposed boundary is exact in the observed source population.

Before 15 October 2025:

* 178,691 provisional races are present;
* no `off` value begins with a leading zero;
* no parsed hour exceeds 12.

From 15 October 2025 onward:

* 10,352 provisional races are present;
* every `off` value is exactly five characters long;
* low hours are zero-padded;
* explicit hours above 12 are permitted.

There are no observed exceptions on either side of the boundary.

This establishes two deterministic raw-format regimes. It does not yet establish whether the pre-boundary values are scheduled times, actual off-times, local times or another temporal convention.

In [22]:
# Summarise the two confirmed raw-format regimes for later conclusions.

off_format_regimes = pd.DataFrame(
    [
        {
            "regime": "pre_2025_10_15",
            "date_range": "before 2025-10-15",
            "provisional_races": int(
                off_format_boundary_validation.loc[0, "pre_boundary_races"]
            ),
            "observed_format": "H:MM or 10:MM–12:MM",
            "leading_zero_low_hours": False,
            "explicit_hours_above_12": False,
            "interpretation_status": (
                "time-shaped text; AM/PM and temporal meaning unresolved"
            ),
        },
        {
            "regime": "from_2025_10_15",
            "date_range": "2025-10-15 onward",
            "provisional_races": int(
                off_format_boundary_validation.loc[0, "post_boundary_races"]
            ),
            "observed_format": "HH:MM",
            "leading_zero_low_hours": True,
            "explicit_hours_above_12": True,
            "interpretation_status": (
                "24-hour-style text; timezone and temporal meaning unresolved"
            ),
        },
    ]
)

off_format_regimes

,regime,date_range,provisional_races,observed_format,leading_zero_low_hours,explicit_hours_above_12,interpretation_status
0,pre_2025_10_15,before 2025-10-15,178691,H:MM or 10:MM–12:MM,False,False,time-shaped text; AM/PM and temporal meaning u...
1,from_2025_10_15,2025-10-15 onward,10352,HH:MM,True,True,24-hour-style text; timezone and temporal mean...


## Candidate normalisation of pre-boundary clock values

Before 15 October 2025, every observed `off` value uses hours `1–12` without an AM/PM suffix.

Many ordinary afternoon cards strongly suggest that values after `12:xx`, such as `1:10` and `2:15`, should be interpreted as afternoon times. A possible derived representation is therefore:

* retain `12:xx` as hour 12;
* add 12 hours to values from `1:xx` through `11:xx`.

However, this rule would incorrectly convert genuine morning races unless the source convention consistently expresses all pre-boundary times on an unlabeled 12-hour clock.

The next step tests the proposed conversion at meeting level by comparing the raw and adjusted time spans. It is an exploratory diagnostic only and does not yet establish a parsing rule.

In [23]:
# Compare raw clock spans with a candidate PM-adjusted representation for
# pre-boundary meetings.
#
# Candidate adjustment:
#   12:xx remains 12:xx
#   1:xx through 11:xx receive an additional 12 hours
#
# This deliberately tests one hypothesis. It does not prove that all
# pre-boundary races occurred in the afternoon.

pre_boundary_adjustment_profile = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value,
            CAST(SUBSTR(off, INSTR(off, ':') + 1) AS INTEGER) AS minute_value
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date < '2025-10-15'
        GROUP BY
            date,
            course,
            off
    ),
    derived AS (
        SELECT
            date,
            course,
            off,
            hour_value * 60 + minute_value AS raw_minutes,
            CASE
                WHEN hour_value = 12
                    THEN 12 * 60 + minute_value
                ELSE (hour_value + 12) * 60 + minute_value
            END AS candidate_adjusted_minutes
        FROM provisional_races
    ),
    meeting_profiles AS (
        SELECT
            date,
            course,
            COUNT(*) AS provisional_races,
            MIN(raw_minutes) AS minimum_raw_minutes,
            MAX(raw_minutes) AS maximum_raw_minutes,
            MAX(raw_minutes) - MIN(raw_minutes) AS raw_span_minutes,
            MIN(candidate_adjusted_minutes) AS minimum_adjusted_minutes,
            MAX(candidate_adjusted_minutes) AS maximum_adjusted_minutes,
            MAX(candidate_adjusted_minutes)
                - MIN(candidate_adjusted_minutes) AS adjusted_span_minutes
        FROM derived
        GROUP BY
            date,
            course
    )
    SELECT
        COUNT(*) AS meetings,
        SUM(
            CASE
                WHEN adjusted_span_minutes < raw_span_minutes
                THEN 1 ELSE 0
            END
        ) AS meetings_with_shorter_adjusted_span,
        SUM(
            CASE
                WHEN adjusted_span_minutes = raw_span_minutes
                THEN 1 ELSE 0
            END
        ) AS meetings_with_equal_span,
        SUM(
            CASE
                WHEN adjusted_span_minutes > raw_span_minutes
                THEN 1 ELSE 0
            END
        ) AS meetings_with_longer_adjusted_span,
        MIN(raw_span_minutes) AS minimum_raw_span_minutes,
        MAX(raw_span_minutes) AS maximum_raw_span_minutes,
        MIN(adjusted_span_minutes) AS minimum_adjusted_span_minutes,
        MAX(adjusted_span_minutes) AS maximum_adjusted_span_minutes
    FROM meeting_profiles
    """,
    connection,
)

pre_boundary_adjustment_profile

,meetings,meetings_with_shorter_adjusted_span,meetings_with_equal_span,meetings_with_longer_adjusted_span,minimum_raw_span_minutes,maximum_raw_span_minutes,minimum_adjusted_span_minutes,maximum_adjusted_span_minutes
0,31441,3629,26894,918,0,700,0,716


## Possible UK-facing reference time

The overseas examples are inconsistent with racecourse-local time.

A plausible alternative is that `off` is expressed for a UK-facing audience. Two distinct possibilities must be separated:

* fixed UTC or GMT throughout the year;
* UK civil time, using GMT in winter and BST in summer.

Racecourse-local daylight-saving changes could complicate comparisons for Australia, New Zealand, Europe and North America.

Hong Kong and Japan provide cleaner tests because they do not seasonally change their civil clocks. If comparable meetings from those jurisdictions appear approximately one hour later in the source while the UK is on BST, that would support a UK-civil-time interpretation.

The next step labels post-boundary Hong Kong and Japanese meetings using the actual `Europe/London` UTC offset on each source date. It does not yet assume that the source date itself is a UK calendar date.

In [24]:
from datetime import date, datetime, time
from zoneinfo import ZoneInfo

# Load post-boundary Hong Kong and Japanese provisional races.
#
# These jurisdictions are useful because their local civil clocks do not
# observe daylight saving. Any systematic one-hour seasonal movement in the
# source values may therefore reflect the UK reference clock rather than a
# local clock change.

east_asia_off_times = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value,
        CAST(SUBSTR(off, INSTR(off, ':') + 1) AS INTEGER) AS minute_value
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND date >= '2025-10-15'
      AND (
          course IN ('Sha Tin', 'Happy Valley', 'Tokyo', 'Kyoto')
          OR course LIKE '%(HK)%'
          OR course LIKE '%(JPN)%'
      )
    GROUP BY
        date,
        course,
        off
    ORDER BY
        date,
        course,
        hour_value,
        minute_value
    """,
    connection,
)

london_zone = ZoneInfo("Europe/London")


def london_offset_label(source_date: str) -> str:
    """Return the London civil-time regime at noon on the source date."""
    parsed_date = date.fromisoformat(source_date)
    london_noon = datetime.combine(
        parsed_date,
        time(12, 0),
        tzinfo=london_zone,
    )

    offset_hours = int(
        london_noon.utcoffset().total_seconds() // 3600
    )

    return "BST_UTC_plus_1" if offset_hours == 1 else "GMT_UTC_plus_0"


east_asia_off_times["london_clock_regime"] = (
    east_asia_off_times["date"].map(london_offset_label)
)

east_asia_off_times["off_minutes"] = (
    east_asia_off_times["hour_value"] * 60
    + east_asia_off_times["minute_value"]
)

east_asia_clock_summary = (
    east_asia_off_times
    .groupby(
        ["course", "london_clock_regime"],
        as_index=False,
    )
    .agg(
        provisional_races=("off", "size"),
        meeting_dates=("date", "nunique"),
        minimum_off_minutes=("off_minutes", "min"),
        median_off_minutes=("off_minutes", "median"),
        maximum_off_minutes=("off_minutes", "max"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(["course", "london_clock_regime"])
    .reset_index(drop=True)
)

east_asia_clock_summary

,course,london_clock_regime,provisional_races,meeting_dates,minimum_off_minutes,median_off_minutes,maximum_off_minutes,first_date,last_date
0,Happy Valley,BST_UTC_plus_1,80,9,695,820.0,950,2025-10-15,2026-05-27
1,Happy Valley,GMT_UTC_plus_0,171,19,285,760.0,895,2025-11-02,2026-03-25
2,Kyoto,BST_UTC_plus_1,6,6,450,455.0,465,2025-10-19,2026-05-23
3,Kyoto,GMT_UTC_plus_0,15,15,390,400.0,435,2025-10-26,2026-02-15
4,Sha Tin,BST_UTC_plus_1,126,12,330,520.0,950,2025-10-19,2026-05-24
5,Sha Tin,GMT_UTC_plus_0,257,25,265,450.0,890,2025-10-26,2026-03-22
6,Tokyo,BST_UTC_plus_1,9,9,460,465.0,465,2025-10-18,2026-05-24
7,Tokyo,GMT_UTC_plus_0,12,12,390,400.0,405,2025-11-02,2026-02-22


### Evidence for UK civil time

For Hong Kong and Japanese courses, median source `off` values are approximately one hour later while London is observing BST than while London is observing GMT.

Because Hong Kong and Japan do not seasonally change their civil clocks, this pattern is consistent with the source displaying race times in UK civil time:

* GMT during the UK winter;
* BST during the UK summer.

The comparison is not exact because the underlying meetings and race schedules differ. It therefore provides strong evidence rather than definitive proof.

The next step compares the nearest meetings immediately before and after the UK clock changes to test for an abrupt one-hour movement.

In [25]:
# Compare the nearest Hong Kong and Japanese meetings around the UK clock
# changes at the end of October 2025 and the end of March 2026.
#
# An abrupt movement of approximately one hour would be stronger evidence
# of UK civil-time alignment than broad seasonal averages alone.

east_asia_meeting_ranges = (
    east_asia_off_times
    .groupby(["date", "course"], as_index=False)
    .agg(
        provisional_races=("off", "size"),
        first_off_minutes=("off_minutes", "min"),
        median_off_minutes=("off_minutes", "median"),
        last_off_minutes=("off_minutes", "max"),
    )
)

clock_change_dates = pd.DataFrame(
    [
        {
            "clock_change": "BST_to_GMT_2025",
            "change_date": "2025-10-26",
        },
        {
            "clock_change": "GMT_to_BST_2026",
            "change_date": "2026-03-29",
        },
    ]
)

comparison_rows = []

for _, change in clock_change_dates.iterrows():
    change_date = pd.Timestamp(change["change_date"])

    for course in sorted(east_asia_meeting_ranges["course"].unique()):
        course_meetings = east_asia_meeting_ranges[
            east_asia_meeting_ranges["course"] == course
        ].copy()

        course_meetings["parsed_date"] = pd.to_datetime(
            course_meetings["date"]
        )

        before = (
            course_meetings[
                course_meetings["parsed_date"] < change_date
            ]
            .sort_values("parsed_date")
            .tail(1)
        )

        after = (
            course_meetings[
                course_meetings["parsed_date"] >= change_date
            ]
            .sort_values("parsed_date")
            .head(1)
        )

        if before.empty or after.empty:
            continue

        before_row = before.iloc[0]
        after_row = after.iloc[0]

        comparison_rows.append(
            {
                "clock_change": change["clock_change"],
                "course": course,
                "before_date": before_row["date"],
                "after_date": after_row["date"],
                "before_first_off": before_row["first_off_minutes"],
                "after_first_off": after_row["first_off_minutes"],
                "first_off_change_minutes": (
                    after_row["first_off_minutes"]
                    - before_row["first_off_minutes"]
                ),
                "before_median_off": before_row["median_off_minutes"],
                "after_median_off": after_row["median_off_minutes"],
                "median_change_minutes": (
                    after_row["median_off_minutes"]
                    - before_row["median_off_minutes"]
                ),
                "before_last_off": before_row["last_off_minutes"],
                "after_last_off": after_row["last_off_minutes"],
                "last_off_change_minutes": (
                    after_row["last_off_minutes"]
                    - before_row["last_off_minutes"]
                ),
            }
        )

clock_change_comparisons = pd.DataFrame(comparison_rows)

clock_change_comparisons

,clock_change,course,before_date,after_date,before_first_off,after_first_off,first_off_change_minutes,before_median_off,after_median_off,median_change_minutes,before_last_off,after_last_off,last_off_change_minutes
0,BST_to_GMT_2025,Happy Valley,2025-10-22,2025-11-02,700,285,-415,820.0,445.0,-375.0,950,590,-360
1,BST_to_GMT_2025,Kyoto,2025-10-19,2025-10-26,460,400,-60,460.0,400.0,-60.0,460,400,-60
2,BST_to_GMT_2025,Sha Tin,2025-10-19,2025-10-26,345,285,-60,505.0,445.0,-60.0,655,595,-60
3,BST_to_GMT_2025,Tokyo,2025-10-25,2025-11-02,465,400,-65,465.0,400.0,-65.0,465,400,-65
4,GMT_to_BST_2026,Happy Valley,2026-03-25,2026-04-08,640,700,60,760.0,820.0,60.0,890,950,60
5,GMT_to_BST_2026,Kyoto,2026-02-15,2026-04-26,390,450,60,390.0,450.0,60.0,390,450,60
6,GMT_to_BST_2026,Sha Tin,2026-03-22,2026-03-29,300,345,45,435.0,515.0,80.0,595,675,80
7,GMT_to_BST_2026,Tokyo,2026-02-22,2026-04-25,400,465,65,400.0,465.0,65.0,400,465,65


### UK civil-time alignment

The nearest-meeting comparisons around UK clock changes provide strong evidence that `off` is expressed in UK civil time.

Around the end of BST in October 2025:

* Kyoto moves from `7:40` to `6:40`, exactly 60 minutes earlier;
* Sha Tin moves exactly 60 minutes earlier across its card;
* Tokyo moves approximately 65 minutes earlier.

Around the start of BST in March 2026:

* Happy Valley moves exactly 60 minutes later across its card;
* Kyoto moves exactly 60 minutes later;
* Tokyo moves approximately 65 minutes later;
* Sha Tin also moves later, although differences in the compared race schedules prevent an exact like-for-like measurement.

The first Happy Valley comparison around October 2025 is not comparable because the two cards occupy substantially different source-time windows. It should not be interpreted as a daylight-saving shift.

Hong Kong and Japan do not change their own civil clocks seasonally. The consistent movement around UK clock changes therefore strongly supports the following interpretation:

> The source `off` field is displayed in UK civil time, using GMT during the UK winter and BST during the UK summer.

This interpretation is strongly supported by the observed source data, although the source does not explicitly label the timezone.

The raw value must still be preserved exactly as supplied.

For derived temporal work:

* post-15 October 2025 values can be interpreted as fixed-width UK civil clock times;
* pre-15 October 2025 values remain ambiguous because the source uses an unlabeled 12-hour display and omits AM/PM;
* overseas course-local dates and times must not be inferred directly from `date + off` without later timezone-aware reconstruction.

## External validation strategy

Source-only meeting context may recover the missing AM/PM interpretation for many pre-15 October 2025 records, but it cannot establish the answer with certainty in every case.

Candidate reconstructions should therefore be externally validated against authoritative or contemporaneous race records.

External validation should test:

* whether `off` represents scheduled, advertised or actual off-time;
* whether the displayed clock is consistently aligned to UK civil time;
* whether the source follows GMT and BST transitions;
* whether meeting-context AM/PM reconstruction is correct;
* whether any jurisdiction, course or period follows a different convention.

Validation should use a stratified sample rather than isolated convenient examples. The sample should include:

* UK and Irish afternoon meetings;
* French meetings spanning noon;
* Hong Kong and Japanese meetings around UK clock changes;
* Australian and New Zealand meetings shown during UK morning hours;
* complete cards and single-race records;
* meetings with uniquely plausible and competing AM/PM interpretations.

The source-only analysis will first classify candidate recoverability. External evidence will then be used to validate the method and resolve selected ambiguous cases.

## Source-only recovery of relative meeting chronology

For pre-15 October 2025 records, the missing AM/PM marker creates a fundamental 12-hour ambiguity.

Meeting context can still recover useful structure. A set of race times can be placed around a 12-hour clock and unwrapped across the point that produces the shortest coherent meeting span.

For example:

`11:10, 11:50, 12:30, 1:10, 1:40, 2:15`

has a natural relative sequence crossing `12`:

`11:10 → 11:50 → 12:30 → 1:10 → 1:40 → 2:15`

However, source context alone does not determine whether that sequence represents:

* `11:10–14:15`; or
* `23:10–02:15`.

The two interpretations differ by 12 hours but have the same internal order and duration.

The next step calculates the shortest circular clock span for every pre-boundary meeting. This measures how well relative chronology can be recovered without pretending that the absolute AM/PM interpretation is known.

In [26]:
# Measure the shortest coherent span of each pre-boundary meeting on a
# 12-hour clock.
#
# Each raw time is converted to a minute position from 0 to 719:
#   12:xx becomes 0:xx on the circular clock
#   1:xx through 11:xx retain their ordinary positions
#
# The largest gap between consecutive values identifies the most natural
# place to "cut" the circle. The remaining arc is the shortest meeting span.
#
# This recovers relative ordering and duration only. The complete sequence
# remains ambiguous by a 12-hour shift until externally validated.

pre_boundary_clock_values = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value,
        CAST(SUBSTR(off, INSTR(off, ':') + 1) AS INTEGER) AS minute_value
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND date < '2025-10-15'
    GROUP BY
        date,
        course,
        off
    ORDER BY
        date,
        course
    """,
    connection,
)

pre_boundary_clock_values["clock_12_minutes"] = (
    (pre_boundary_clock_values["hour_value"] % 12) * 60
    + pre_boundary_clock_values["minute_value"]
)


def shortest_circular_span(values: pd.Series) -> pd.Series:
    """Return the shortest span containing all values on a 720-minute clock."""
    ordered = sorted(values.astype(int).tolist())
    count = len(ordered)

    if count <= 1:
        return pd.Series(
            {
                "provisional_races": count,
                "shortest_span_minutes": 0,
                "largest_unused_gap_minutes": 720,
                "possible_cut_count": 1,
            }
        )

    circular_gaps = [
        ordered[index + 1] - ordered[index]
        for index in range(count - 1)
    ]
    circular_gaps.append(ordered[0] + 720 - ordered[-1])

    largest_gap = max(circular_gaps)

    return pd.Series(
        {
            "provisional_races": count,
            "shortest_span_minutes": 720 - largest_gap,
            "largest_unused_gap_minutes": largest_gap,
            "possible_cut_count": circular_gaps.count(largest_gap),
        }
    )


pre_boundary_circular_profile = (
    pre_boundary_clock_values
    .groupby(["date", "course"])["clock_12_minutes"]
    .apply(shortest_circular_span)
    .unstack()
    .reset_index()
)

pre_boundary_circular_summary = pd.DataFrame(
    [
        {
            "meetings": len(pre_boundary_circular_profile),
            "single_race_meetings": int(
                (
                    pre_boundary_circular_profile["provisional_races"] == 1
                ).sum()
            ),
            "meetings_with_unique_shortest_cut": int(
                (
                    pre_boundary_circular_profile["possible_cut_count"] == 1
                ).sum()
            ),
            "meetings_with_tied_shortest_cuts": int(
                (
                    pre_boundary_circular_profile["possible_cut_count"] > 1
                ).sum()
            ),
            "median_shortest_span_minutes": float(
                pre_boundary_circular_profile[
                    "shortest_span_minutes"
                ].median()
            ),
            "maximum_shortest_span_minutes": int(
                pre_boundary_circular_profile[
                    "shortest_span_minutes"
                ].max()
            ),
            "meetings_with_span_over_6_hours": int(
                (
                    pre_boundary_circular_profile[
                        "shortest_span_minutes"
                    ] > 360
                ).sum()
            ),
        }
    ]
)

pre_boundary_circular_summary

,meetings,single_race_meetings,meetings_with_unique_shortest_cut,meetings_with_tied_shortest_cuts,median_shortest_span_minutes,maximum_shortest_span_minutes,meetings_with_span_over_6_hours
0,31441,6061,31441,0,190.0,500,32


### Meetings with unusually long recovered spans

The circular-clock method produces a compact relative sequence for almost every pre-boundary meeting.

The median shortest span is 190 minutes, and only 32 meetings exceed six hours. This supports the method as a candidate for recovering within-meeting chronology.

However:

* the recovered sequence remains ambiguous by a 12-hour shift;
* single-race meetings provide no contextual AM/PM evidence;
* unusually long spans may indicate incomplete cards, unusual schedules, source anomalies or limitations of the shortest-span assumption.

The next step inspects all meetings whose shortest recovered span exceeds six hours.

In [27]:
# Inspect the small set of pre-boundary meetings whose shortest possible
# circular span exceeds six hours.
#
# The raw values are retained in circular-clock order so that unusual cards
# can be reviewed without yet assigning AM or PM.

long_span_meetings = (
    pre_boundary_circular_profile[
        pre_boundary_circular_profile["shortest_span_minutes"] > 360
    ]
    .sort_values(
        ["shortest_span_minutes", "date", "course"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

long_span_meeting_values = (
    pre_boundary_clock_values
    .merge(
        long_span_meetings[
            [
                "date",
                "course",
                "provisional_races",
                "shortest_span_minutes",
                "largest_unused_gap_minutes",
            ]
        ],
        on=["date", "course"],
        how="inner",
    )
    .sort_values(
        ["shortest_span_minutes", "date", "course", "clock_12_minutes"],
        ascending=[False, True, True, True],
    )
    .groupby(
        [
            "date",
            "course",
            "provisional_races",
            "shortest_span_minutes",
            "largest_unused_gap_minutes",
        ],
        as_index=False,
    )
    .agg(
        observed_off_values=(
            "off",
            lambda values: ", ".join(values.astype(str)),
        )
    )
)

long_span_meeting_values

,date,course,provisional_races,shortest_span_minutes,largest_unused_gap_minutes,observed_off_values
0,2015-05-01,Churchill Downs (USA),7,379,341,"4:30, 6:26, 7:08, 8:02, 9:02, 9:52, 10:49"
1,2015-06-06,Belmont Park (USA),10,435,285,"4:35, 5:39, 6:15, 6:52, 7:34, 8:15, 9:00, 9:49..."
2,2016-06-27,Auteuil (FR),13,385,335,"12:05, 12:47, 1:20, 1:50, 2:20, 2:55, 3:25, 3:..."
3,2017-03-05,Auteuil (FR),12,370,350,"12:30, 1:10, 1:40, 2:15, 2:45, 3:20, 3:50, 4:2..."
4,2017-11-04,Del Mar (USA),11,427,293,"12:17, 5:10, 6:20, 7:00, 7:37, 8:14, 9:00, 9:3..."
5,2017-11-07,Flemington (AUS),3,380,340,"12:00, 4:00, 6:20"
6,2018-01-04,Deauville (FR),13,385,335,"12:25, 1:05, 1:35, 2:05, 2:40, 3:15, 3:50, 9:2..."
7,2018-01-05,Deauville (FR),14,415,305,"12:10, 12:47, 1:20, 1:50, 2:20, 2:55, 3:25, 3:..."
8,2018-03-22,Fontainebleau (FR),12,370,350,"12:10, 12:47, 1:20, 1:55, 2:30, 3:00, 3:45, 4:..."
9,2018-05-05,Churchill Downs (USA),8,407,313,"5:03, 6:13, 6:55, 7:45, 8:37, 9:28, 10:25, 11:50"


## Stratified external-validation sample

The 32 meetings with source-only spans above six hours are largely plausible long cards rather than clear reconstruction failures.

They include:

* major United States cards extending across much of the UK evening;
* large French cards containing up to 15 races;
* sparse imported records where only selected races are present.

A six-hour threshold is therefore not a suitable validity rule.

The principal unresolved issue remains the absolute 12-hour interpretation of pre-boundary values. External validation should now compare selected source records with contemporaneous racecards or results.

The validation sample will include:

* ordinary UK and Irish afternoon cards;
* French cards crossing noon in the source clock;
* Australian and East Asian cards appearing during the UK morning;
* major North American evening cards;
* single-race and sparse imported records;
* unusually long meetings;
* meetings near UK daylight-saving transitions.

The next step constructs a reproducible stratified sample from the source.

In [28]:
# Construct a genuinely stratified external-validation sample.
#
# Each broad region contributes examples from several analytically useful
# meeting types where available:
#
#   * single-race or sparse records;
#   * large cards;
#   * cards containing very low source-clock hours;
#   * cards crossing 12 on the old unlabeled clock;
#   * ordinary multi-race cards.
#
# A maximum of two meetings is selected per region and stratum so that sparse
# records cannot dominate the sample.

validation_candidates = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            race_id,
            race_name,
            MIN(rowid) AS first_source_rowid,
            CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value,
            CAST(SUBSTR(off, INSTR(off, ':') + 1) AS INTEGER) AS minute_value
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date < '2025-10-15'
        GROUP BY
            date,
            course,
            off,
            race_id,
            race_name
    ),
    classified_races AS (
        SELECT
            *,
            CASE
                WHEN course LIKE '%(AUS)%'
                    THEN 'Australia'
                WHEN course LIKE '%(NZ)%'
                    THEN 'New Zealand'
                WHEN course LIKE '%(JPN)%'
                    THEN 'Japan'
                WHEN course LIKE '%(HK)%'
                    THEN 'Hong Kong'
                WHEN course LIKE '%(USA)%'
                    THEN 'United States'
                WHEN course LIKE '%(FR)%'
                    THEN 'France'
                WHEN course LIKE '%(IRE)%'
                    THEN 'Ireland'
                WHEN course NOT LIKE '%(%'
                     OR course LIKE '%(GB)%'
                    THEN 'Britain'
                ELSE 'Other'
            END AS broad_region
        FROM provisional_races
    ),
    meeting_profiles AS (
        SELECT
            date,
            course,
            broad_region,
            COUNT(*) AS provisional_races,
            MIN(hour_value) AS minimum_hour,
            MAX(hour_value) AS maximum_hour,
            MIN(first_source_rowid) AS first_source_rowid,
            CASE
                WHEN COUNT(*) = 1
                    THEN 'single_race'
                WHEN COUNT(*) >= 10
                    THEN 'large_card'
                WHEN MIN(hour_value) <= 3
                    THEN 'low_hour_card'
                WHEN MIN(hour_value) = 12
                  OR (
                      MIN(hour_value) <= 2
                      AND MAX(hour_value) >= 10
                  )
                    THEN 'crosses_12'
                ELSE 'ordinary_multi_race_card'
            END AS validation_stratum
        FROM classified_races
        GROUP BY
            date,
            course,
            broad_region
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY broad_region, validation_stratum
                ORDER BY
                    provisional_races DESC,
                    date DESC,
                    course
            ) AS stratum_rank
        FROM meeting_profiles
    )
    SELECT
        date,
        course,
        broad_region,
        validation_stratum,
        provisional_races,
        minimum_hour,
        maximum_hour
    FROM ranked
    WHERE stratum_rank <= 2
      AND broad_region IN (
          'Britain',
          'Ireland',
          'France',
          'Australia',
          'New Zealand',
          'Hong Kong',
          'Japan',
          'United States'
      )
    ORDER BY
        broad_region,
        CASE validation_stratum
            WHEN 'ordinary_multi_race_card' THEN 1
            WHEN 'crosses_12' THEN 2
            WHEN 'low_hour_card' THEN 3
            WHEN 'large_card' THEN 4
            WHEN 'single_race' THEN 5
        END,
        date,
        course
    """,
    connection,
)

validation_candidates

,date,course,broad_region,validation_stratum,provisional_races,minimum_hour,maximum_hour
0,2025-04-12,Randwick (AUS),Australia,ordinary_multi_race_card,7,4,8
1,2025-04-26,Morphettville (AUS),Australia,ordinary_multi_race_card,7,4,7
2,2025-03-22,Rosehill (AUS),Australia,low_hour_card,9,2,6
3,2025-04-05,Randwick (AUS),Australia,low_hour_card,9,2,7
4,2018-10-20,Caulfield (AUS),Australia,large_card,10,2,7
...,...,...,...,...,...,...,...
57,2020-05-27,Fonner Park (USA),United States,low_hour_card,9,1,12
58,2020-03-28,Gulfstream Park (USA),United States,large_card,14,3,10
59,2020-04-11,Gulfstream Park (USA),United States,large_card,13,5,11
60,2025-10-11,Keeneland (USA),United States,single_race,1,10,10


### Race-level external-validation sheet

The corrected sample contains 62 meetings distributed across the required regions and meeting types.

External validation will begin with representative races from each meeting rather than every race. For multi-race cards, the sample will include:

* the first race by reconstructed relative order;
* a middle race;
* the final race.

For single-race meetings, the sole available race will be included.

This should normally be sufficient to determine:

* which 12-hour-shifted interpretation is correct;
* whether the reconstructed meeting sequence is coherent;
* whether `off` corresponds to an advertised race time or an actual starting time;
* and whether the UK civil-time interpretation holds across the card.

Any meeting showing disagreement can then be expanded to all races.

In [29]:
# Expand the stratified meeting sample into representative race-level records.
#
# Pre-boundary times are ordered on the 12-hour circular clock using the
# largest unused gap as the cut point. This recovers a relative sequence but
# does not yet choose between the two absolute UK-time candidates separated
# by 12 hours.

sample_races = pre_boundary_clock_values.merge(
    validation_candidates[
        [
            "date",
            "course",
            "broad_region",
            "validation_stratum",
            "provisional_races",
        ]
    ],
    on=["date", "course"],
    how="inner",
)


def unwrap_meeting(group: pd.DataFrame) -> pd.DataFrame:
    """Recover the shortest relative ordering on the 12-hour clock."""
    group = group.copy()

    ordered_positions = sorted(
        group["clock_12_minutes"].astype(int).tolist()
    )

    if len(ordered_positions) == 1:
        cut_after = ordered_positions[0]
    else:
        gaps = []

        for index, value in enumerate(ordered_positions):
            next_value = (
                ordered_positions[index + 1]
                if index + 1 < len(ordered_positions)
                else ordered_positions[0] + 720
            )

            gaps.append(
                {
                    "value": value,
                    "gap": next_value - value,
                }
            )

        cut_after = max(
            gaps,
            key=lambda item: item["gap"],
        )["value"]

    group["relative_minutes"] = group["clock_12_minutes"].where(
        group["clock_12_minutes"] > cut_after,
        group["clock_12_minutes"] + 720,
    )

    group["relative_minutes"] = (
        group["relative_minutes"]
        - group["relative_minutes"].min()
    )

    return (
        group
        .sort_values(
            ["relative_minutes", "off"],
        )
        .reset_index(drop=True)
    )


# Process each meeting explicitly so that date and course remain ordinary
# columns regardless of pandas groupby.apply behaviour.
ordered_meeting_frames = []

for (meeting_date, meeting_course), meeting_group in sample_races.groupby(
    ["date", "course"],
    sort=False,
):
    ordered_meeting = unwrap_meeting(meeting_group)

    ordered_meeting["date"] = meeting_date
    ordered_meeting["course"] = meeting_course

    ordered_meeting_frames.append(ordered_meeting)

ordered_sample_races = pd.concat(
    ordered_meeting_frames,
    ignore_index=True,
)

ordered_sample_races["race_sequence"] = (
    ordered_sample_races
    .groupby(["date", "course"])
    .cumcount()
    + 1
)

ordered_sample_races["meeting_race_count"] = (
    ordered_sample_races
    .groupby(["date", "course"])["off"]
    .transform("size")
)


def representative_position(row: pd.Series) -> str | None:
    """Label the first, middle and final representative races."""
    sequence = int(row["race_sequence"])
    count = int(row["meeting_race_count"])
    middle = (count + 1) // 2

    labels = []

    if sequence == 1:
        labels.append("first")

    if sequence == middle:
        labels.append("middle")

    if sequence == count:
        labels.append("last")

    return "_".join(labels) if labels else None


ordered_sample_races["representative_position"] = (
    ordered_sample_races.apply(
        representative_position,
        axis=1,
    )
)

external_validation_sheet = (
    ordered_sample_races[
        ordered_sample_races["representative_position"].notna()
    ]
    [
        [
            "date",
            "course",
            "broad_region",
            "validation_stratum",
            "meeting_race_count",
            "representative_position",
            "race_sequence",
            "off",
            "relative_minutes",
        ]
    ]
    .sort_values(
        [
            "broad_region",
            "date",
            "course",
            "race_sequence",
        ]
    )
    .reset_index(drop=True)
)

external_validation_sheet

,date,course,broad_region,validation_stratum,meeting_race_count,representative_position,race_sequence,off,relative_minutes
0,2018-10-20,Caulfield (AUS),Australia,large_card,10,first,1,2:15,0
1,2018-10-20,Caulfield (AUS),Australia,large_card,10,middle,5,4:45,150
2,2018-10-20,Caulfield (AUS),Australia,large_card,10,last,10,7:50,335
3,2024-04-06,Randwick (AUS),Australia,large_card,10,first,1,2:25,0
4,2024-04-06,Randwick (AUS),Australia,large_card,10,middle,5,4:45,140
...,...,...,...,...,...,...,...,...,...
143,2025-03-01,Gulfstream Park (USA),United States,ordinary_multi_race_card,9,first,1,6:01,0
144,2025-03-01,Gulfstream Park (USA),United States,ordinary_multi_race_card,9,middle,5,9:04,183
145,2025-03-01,Gulfstream Park (USA),United States,ordinary_multi_race_card,9,last,9,11:14,313
146,2025-10-11,Keeneland (USA),United States,single_race,1,first_middle_last,1,10:16,0


### Pilot external-validation set

The race-level sheet contains 148 representative records across 62 meetings.

External validation will begin with a smaller pilot designed to test the principal reconstruction cases:

* Australian racing shown during the UK morning;
* Hong Kong and Japanese racing across GMT and BST;
* French and Irish afternoon cards crossing `12`;
* North American racing shown during the UK evening;
* one sparse or single-race record.

The pilot will establish:

* whether the external advertised time matches the source `off`;
* whether the correct pre-boundary interpretation is the morning or afternoon candidate;
* whether the date is a UK-facing racing date or the racecourse-local date;
* and whether the source time is scheduled or actual off-time.

If the pilot confirms a stable method, the same procedure can be applied to the remaining validation sheet.

In [30]:
# Select a small, explicit pilot sample for external validation.
#
# These meetings cover the major regions and ambiguity types without relying
# on random selection. Three representative races are retained for full cards
# and the sole race is retained for single-race records.

pilot_meetings = pd.DataFrame(
    [
        {
            "date": "2018-10-20",
            "course": "Caulfield (AUS)",
            "pilot_reason": "Australian full card in UK morning",
        },
        {
            "date": "2024-04-06",
            "course": "Randwick (AUS)",
            "pilot_reason": "Recent Australian full card",
        },
        {
            "date": "2024-06-30",
            "course": "Curragh (IRE)",
            "pilot_reason": "Irish afternoon card using low hours",
        },
        {
            "date": "2017-03-05",
            "course": "Auteuil (FR)",
            "pilot_reason": "French card crossing 12",
        },
        {
            "date": "2021-04-11",
            "course": "Sha Tin (HK)",
            "pilot_reason": "Hong Kong card during UK BST",
        },
        {
            "date": "2025-03-01",
            "course": "Gulfstream Park (USA)",
            "pilot_reason": "North American evening card",
        },
        {
            "date": "2025-10-11",
            "course": "Keeneland (USA)",
            "pilot_reason": "Single-race record near format boundary",
        },
    ]
)

pilot_external_validation = (
    external_validation_sheet
    .merge(
        pilot_meetings,
        on=["date", "course"],
        how="inner",
    )
    [
        [
            "date",
            "course",
            "pilot_reason",
            "representative_position",
            "race_sequence",
            "off",
            "relative_minutes",
        ]
    ]
    .sort_values(
        ["date", "course", "race_sequence"]
    )
    .reset_index(drop=True)
)

pilot_external_validation

,date,course,pilot_reason,representative_position,race_sequence,off,relative_minutes
0,2018-10-20,Caulfield (AUS),Australian full card in UK morning,first,1,2:15,0
1,2018-10-20,Caulfield (AUS),Australian full card in UK morning,middle,5,4:45,150
2,2018-10-20,Caulfield (AUS),Australian full card in UK morning,last,10,7:50,335
3,2024-04-06,Randwick (AUS),Recent Australian full card,first,1,2:25,0
4,2024-04-06,Randwick (AUS),Recent Australian full card,middle,5,4:45,140
5,2024-04-06,Randwick (AUS),Recent Australian full card,last,10,7:50,325
6,2025-03-01,Gulfstream Park (USA),North American evening card,first,1,6:01,0
7,2025-03-01,Gulfstream Park (USA),North American evening card,middle,5,9:04,183
8,2025-03-01,Gulfstream Park (USA),North American evening card,last,9,11:14,313
9,2025-10-11,Keeneland (USA),Single-race record near format boundary,first_middle_last,1,10:16,0


### Pilot records prepared for external lookup

Four requested pilot meetings are present in the stratified race-level validation sheet:

* Caulfield on 20 October 2018;
* Randwick on 6 April 2024;
* Gulfstream Park on 1 March 2025;
* Keeneland on 11 October 2025.

The requested Curragh, Auteuil and Sha Tin meetings were not selected into the current 62-meeting sheet, so they did not appear in the pilot output. They can be added separately later.

Before external lookup, each pilot record needs its source race name and source race reference. These fields will make contemporaneous racecards and results much easier to identify accurately.

In [31]:
# Enrich the pilot records with source race identity and two possible
# pre-boundary UK clock interpretations.
#
# For hours 1 through 11:
#   candidate A retains the apparent morning time;
#   candidate B adds 12 hours.
#
# Hour 12 has two possible interpretations:
#   00:xx or 12:xx.
#
# External evidence will determine which candidate is correct.

pilot_validation_lookup = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_id,
        race_name
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND (
          (date = '2018-10-20' AND course = 'Caulfield (AUS)')
          OR
          (date = '2024-04-06' AND course = 'Randwick (AUS)')
          OR
          (date = '2025-03-01' AND course = 'Gulfstream Park (USA)')
          OR
          (date = '2025-10-11' AND course = 'Keeneland (USA)')
      )
    GROUP BY
        date,
        course,
        off,
        race_id,
        race_name
    """,
    connection,
)

pilot_validation_lookup = (
    pilot_external_validation
    .merge(
        pilot_validation_lookup,
        on=["date", "course", "off"],
        how="left",
        validate="one_to_one",
    )
)


def format_minutes(total_minutes: int) -> str:
    """Format minutes after midnight as HH:MM."""
    total_minutes %= 24 * 60
    return f"{total_minutes // 60:02d}:{total_minutes % 60:02d}"


def absolute_time_candidates(raw_off: str) -> tuple[str, str]:
    """Return the two UK-clock candidates separated by 12 hours."""
    raw_hour, raw_minute = map(int, raw_off.split(":"))

    first_minutes = (raw_hour % 12) * 60 + raw_minute
    second_minutes = first_minutes + 12 * 60

    return (
        format_minutes(first_minutes),
        format_minutes(second_minutes),
    )


candidate_pairs = pilot_validation_lookup["off"].map(
    absolute_time_candidates
)

pilot_validation_lookup["candidate_uk_time_a"] = candidate_pairs.map(
    lambda pair: pair[0]
)

pilot_validation_lookup["candidate_uk_time_b"] = candidate_pairs.map(
    lambda pair: pair[1]
)

pilot_validation_lookup = pilot_validation_lookup[
    [
        "date",
        "course",
        "pilot_reason",
        "representative_position",
        "race_sequence",
        "off",
        "candidate_uk_time_a",
        "candidate_uk_time_b",
        "race_id",
        "race_name",
    ]
].sort_values(
    ["date", "course", "race_sequence"]
).reset_index(drop=True)

pilot_validation_lookup

,date,course,pilot_reason,representative_position,race_sequence,off,candidate_uk_time_a,candidate_uk_time_b,race_id,race_name
0,2018-10-20,Caulfield (AUS),Australian full card in UK morning,first,1,2:15,02:15,14:15,714541,QMS Media Plate (Conditions) (3yo Fillies) (Turf)
1,2018-10-20,Caulfield (AUS),Australian full card in UK morning,middle,5,4:45,04:45,16:45,714545,New Zealand Bloodstock Ethereal Stakes (3yo F...
2,2018-10-20,Caulfield (AUS),Australian full card in UK morning,last,10,7:50,07:50,19:50,714667,Ladbrokes Moonga Stakes (4yo+) (Turf)
3,2024-04-06,Randwick (AUS),Recent Australian full card,first,1,2:25,02:25,14:25,865155,Widden Kindergarten Stakes (2yo) (Turf)
4,2024-04-06,Randwick (AUS),Recent Australian full card,middle,5,4:45,04:45,16:45,865159,Newhaven Park Country Championships Final (Con...
5,2024-04-06,Randwick (AUS),Recent Australian full card,last,10,7:50,07:50,19:50,864987,China Horse Club P J Bell Stakes (3yo Fillies...
6,2025-03-01,Gulfstream Park (USA),North American evening card,first,1,6:01,06:01,18:01,889057,Herecomesthebride Stakes (3yo Fillies) (Turf)
7,2025-03-01,Gulfstream Park (USA),North American evening card,middle,5,9:04,09:04,21:04,889795,The Very One Stakes presented by MyRacehorse ...
8,2025-03-01,Gulfstream Park (USA),North American evening card,last,9,11:14,11:14,23:14,889061,Mac Diarmida Stakes presented by FanDuel TV (...
9,2025-10-11,Keeneland (USA),Single-race record near format boundary,first_middle_last,1,10:16,10:16,22:16,905191,Queen Elizabeth II Challenge Cup Stakes presen...


### Initial external-validation findings

The first external checks support the proposed UK-time reconstruction.

For the Australian meetings:

* Caulfield values `2:15`, `4:45` and `7:50` correspond to UK morning times;
* Randwick `2:25` corresponds to `02:25` UK time;
* the 12-hour-shifted afternoon candidates are not credible.

For the North American meetings:

* Gulfstream values `6:01`, `9:04` and `11:14` correspond to `18:01`, `21:04` and `23:14` UK time;
* Keeneland `10:16` corresponds to `22:16` UK time;
* the unshifted morning candidates are not credible.

The pilot therefore confirms that pre-boundary low-hour values cannot be assigned globally to either AM or PM. Their correct interpretation depends on the meeting.

The Keeneland validation also distinguishes two temporal concepts:

* the source value is `10:16`, reconstructed as an advertised UK time of `22:16`;
* an external result records the actual off-time as `22:17`.

This supports treating source `off` provisionally as an advertised or scheduled UK-facing time rather than assuming it records the exact moment the race started.

Further external validation is still required before applying the reconstruction method to the full pre-boundary population.

In [32]:
# Record the initial external-validation findings.
#
# These results distinguish the selected absolute UK-time candidate and,
# where evidence permits, advertised time from actual off-time.
#
# The evidence URLs and access date should be retained for auditability.

pilot_validation_results = pd.DataFrame(
    [
        {
            "date": "2018-10-20",
            "course": "Caulfield (AUS)",
            "race_id": 714541,
            "raw_off": "2:15",
            "reconstructed_uk_time": "02:15",
            "selected_candidate": "A",
            "external_advertised_time": "02:15",
            "external_actual_off_time": None,
            "external_source": "Racing Post",
            "validation_result": "confirmed",
            "temporal_semantics": "advertised_or_scheduled_time",
            "confidence": "high",
        },
        {
            "date": "2018-10-20",
            "course": "Caulfield (AUS)",
            "race_id": 714545,
            "raw_off": "4:45",
            "reconstructed_uk_time": "04:45",
            "selected_candidate": "A",
            "external_advertised_time": "04:45",
            "external_actual_off_time": None,
            "external_source": "Racing Post",
            "validation_result": "confirmed",
            "temporal_semantics": "advertised_or_scheduled_time",
            "confidence": "high",
        },
        {
            "date": "2018-10-20",
            "course": "Caulfield (AUS)",
            "race_id": 714667,
            "raw_off": "7:50",
            "reconstructed_uk_time": "07:50",
            "selected_candidate": "A",
            "external_advertised_time": "07:50",
            "external_actual_off_time": None,
            "external_source": "Racing Post",
            "validation_result": "confirmed",
            "temporal_semantics": "advertised_or_scheduled_time",
            "confidence": "high",
        },
        {
            "date": "2024-04-06",
            "course": "Randwick (AUS)",
            "race_id": 865155,
            "raw_off": "2:25",
            "reconstructed_uk_time": "02:25",
            "selected_candidate": "A",
            "external_advertised_time": "02:25",
            "external_actual_off_time": None,
            "external_source": "At The Races",
            "validation_result": "confirmed",
            "temporal_semantics": "advertised_or_scheduled_time",
            "confidence": "high",
        },
        {
            "date": "2025-03-01",
            "course": "Gulfstream Park (USA)",
            "race_id": 889057,
            "raw_off": "6:01",
            "reconstructed_uk_time": "18:01",
            "selected_candidate": "B",
            "external_advertised_time": "18:01",
            "external_actual_off_time": None,
            "external_source": "At The Races / Sky Sports",
            "validation_result": "confirmed",
            "temporal_semantics": "advertised_or_scheduled_time",
            "confidence": "high",
        },
        {
            "date": "2025-03-01",
            "course": "Gulfstream Park (USA)",
            "race_id": 889795,
            "raw_off": "9:04",
            "reconstructed_uk_time": "21:04",
            "selected_candidate": "B",
            "external_advertised_time": "21:04",
            "external_actual_off_time": None,
            "external_source": "At The Races / Sky Sports",
            "validation_result": "confirmed",
            "temporal_semantics": "advertised_or_scheduled_time",
            "confidence": "high",
        },
        {
            "date": "2025-03-01",
            "course": "Gulfstream Park (USA)",
            "race_id": 889061,
            "raw_off": "11:14",
            "reconstructed_uk_time": "23:14",
            "selected_candidate": "B",
            "external_advertised_time": "23:14",
            "external_actual_off_time": None,
            "external_source": "At The Races / Sky Sports",
            "validation_result": "confirmed",
            "temporal_semantics": "advertised_or_scheduled_time",
            "confidence": "high",
        },
        {
            "date": "2025-10-11",
            "course": "Keeneland (USA)",
            "race_id": 905191,
            "raw_off": "10:16",
            "reconstructed_uk_time": "22:16",
            "selected_candidate": "B",
            "external_advertised_time": "22:16",
            "external_actual_off_time": "22:17",
            "external_source": "Racing TV / Sporting Life",
            "validation_result": "confirmed",
            "temporal_semantics": "advertised_time_not_exact_actual_off",
            "confidence": "high",
        },
    ]
)

pilot_validation_results

,date,course,race_id,raw_off,reconstructed_uk_time,selected_candidate,external_advertised_time,external_actual_off_time,external_source,validation_result,temporal_semantics,confidence
0,2018-10-20,Caulfield (AUS),714541,2:15,02:15,A,02:15,NaN,Racing Post,confirmed,advertised_or_scheduled_time,high
1,2018-10-20,Caulfield (AUS),714545,4:45,04:45,A,04:45,NaN,Racing Post,confirmed,advertised_or_scheduled_time,high
2,2018-10-20,Caulfield (AUS),714667,7:50,07:50,A,07:50,NaN,Racing Post,confirmed,advertised_or_scheduled_time,high
3,2024-04-06,Randwick (AUS),865155,2:25,02:25,A,02:25,NaN,At The Races,confirmed,advertised_or_scheduled_time,high
4,2025-03-01,Gulfstream Park (USA),889057,6:01,18:01,B,18:01,NaN,At The Races / Sky Sports,confirmed,advertised_or_scheduled_time,high
5,2025-03-01,Gulfstream Park (USA),889795,9:04,21:04,B,21:04,NaN,At The Races / Sky Sports,confirmed,advertised_or_scheduled_time,high
6,2025-03-01,Gulfstream Park (USA),889061,11:14,23:14,B,23:14,NaN,At The Races / Sky Sports,confirmed,advertised_or_scheduled_time,high
7,2025-10-11,Keeneland (USA),905191,10:16,22:16,B,22:16,22:17,Racing TV / Sporting Life,confirmed,advertised_time_not_exact_actual_off,high


### Second external-validation pilot

The initial pilot confirms that different meetings can require different
12-hour candidates.

The correct interpretation must not be generalised by country or continent
alone. It depends on:

* the individual racecourse location;
* the source date;
* the course-local timezone and daylight-saving regime;
* the UK GMT/BST regime;
* the plausible local race schedule;
* and external validation.

The validated Caulfield and Randwick examples use the unshifted UK-morning
candidate. The validated Gulfstream Park and Keeneland examples use the
candidate shifted forward by 12 hours.

These are meeting-specific findings, not universal jurisdictional rules.

It also provides evidence that source `off` represents an advertised or scheduled UK-facing time rather than the exact actual start.

The current confirmed sample does not yet include:

* Britain or Ireland;
* France;
* Hong Kong;
* Japan.

A second targeted pilot will therefore select representative meetings directly from the source rather than requiring them to appear in the earlier stratified sample.

The second pilot will focus on:

* an Irish afternoon card;
* a French card crossing `12`;
* a Hong Kong card during UK winter or summer time;
* a Japanese race;
* and an ordinary British afternoon card.

External validation of these records will test whether the same reconstruction logic applies across the remaining major regions.

In [33]:
# Prepare a second targeted external-validation pilot.
#
# These meetings are selected explicitly to cover regions absent from the
# initial confirmed pilot. First, middle and final races are retained where
# full cards are available.

second_pilot_meetings = pd.DataFrame(
    [
        {
            "date": "2024-06-30",
            "course": "Curragh (IRE)",
            "pilot_reason": "Irish afternoon card using low hours",
        },
        {
            "date": "2017-03-05",
            "course": "Auteuil (FR)",
            "pilot_reason": "French card crossing 12",
        },
        {
            "date": "2021-04-11",
            "course": "Sha Tin (HK)",
            "pilot_reason": "Hong Kong card during UK BST",
        },
        {
            "date": "2019-11-24",
            "course": "Kyoto (JPN)",
            "pilot_reason": "Japanese race during UK GMT",
        },
        {
            "date": "2024-07-13",
            "course": "Newmarket (July)",
            "pilot_reason": "Ordinary British afternoon card",
        },
    ]
)

second_pilot_races = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_id,
        race_name,
        CAST(SUBSTR(off, 1, INSTR(off, ':') - 1) AS INTEGER) AS hour_value,
        CAST(SUBSTR(off, INSTR(off, ':') + 1) AS INTEGER) AS minute_value
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND (
          (date = '2024-06-30' AND course = 'Curragh (IRE)')
          OR
          (date = '2017-03-05' AND course = 'Auteuil (FR)')
          OR
          (date = '2021-04-11' AND course = 'Sha Tin (HK)')
          OR
          (date = '2019-11-24' AND course = 'Kyoto (JPN)')
          OR
          (date = '2024-07-13' AND course = 'Newmarket (July)')
      )
    GROUP BY
        date,
        course,
        off,
        race_id,
        race_name
    """,
    connection,
)

second_pilot_races["clock_12_minutes"] = (
    (second_pilot_races["hour_value"] % 12) * 60
    + second_pilot_races["minute_value"]
)

ordered_second_pilot_frames = []

for (meeting_date, meeting_course), meeting_group in second_pilot_races.groupby(
    ["date", "course"],
    sort=False,
):
    ordered_meeting = unwrap_meeting(meeting_group)

    ordered_meeting["date"] = meeting_date
    ordered_meeting["course"] = meeting_course

    ordered_second_pilot_frames.append(ordered_meeting)

ordered_second_pilot = pd.concat(
    ordered_second_pilot_frames,
    ignore_index=True,
)

ordered_second_pilot["race_sequence"] = (
    ordered_second_pilot
    .groupby(["date", "course"])
    .cumcount()
    + 1
)

ordered_second_pilot["meeting_race_count"] = (
    ordered_second_pilot
    .groupby(["date", "course"])["off"]
    .transform("size")
)

ordered_second_pilot["representative_position"] = (
    ordered_second_pilot.apply(
        representative_position,
        axis=1,
    )
)

second_pilot_lookup = (
    ordered_second_pilot[
        ordered_second_pilot["representative_position"].notna()
    ]
    .merge(
        second_pilot_meetings,
        on=["date", "course"],
        how="left",
    )
)

candidate_pairs = second_pilot_lookup["off"].map(
    absolute_time_candidates
)

second_pilot_lookup["candidate_uk_time_a"] = candidate_pairs.map(
    lambda pair: pair[0]
)

second_pilot_lookup["candidate_uk_time_b"] = candidate_pairs.map(
    lambda pair: pair[1]
)

second_pilot_lookup = second_pilot_lookup[
    [
        "date",
        "course",
        "pilot_reason",
        "representative_position",
        "race_sequence",
        "off",
        "candidate_uk_time_a",
        "candidate_uk_time_b",
        "race_id",
        "race_name",
    ]
].sort_values(
    ["date", "course", "race_sequence"]
).reset_index(drop=True)

second_pilot_lookup

,date,course,pilot_reason,representative_position,race_sequence,off,candidate_uk_time_a,candidate_uk_time_b,race_id,race_name
0,2017-03-05,Auteuil (FR),French card crossing 12,first,1,10:40,10:40,22:40,670347,Prix Rivoli (Chase) (AQPS Conditions) (4yo) (T...
1,2017-03-05,Auteuil (FR),French card crossing 12,middle,6,1:40,01:40,13:40,670353,Prix Souviens Toi (Hurdle) (Handicap) (5yo) (T...
2,2017-03-05,Auteuil (FR),French card crossing 12,last,12,4:50,04:50,16:50,670362,Prix Bougie (Hurdle) (Handicap) (5yo+) (Turf)
3,2019-11-24,Kyoto (JPN),Japanese race during UK GMT,first_middle_last,1,7:15,07:15,19:15,745923,Keihan Hai (3yo+) (Turf)
4,2021-04-11,Sha Tin (HK),Hong Kong card during UK BST,first,1,5:30,05:30,17:30,782251,Windy Gap Plate (Conditions) (2yo+) (Course C)...
5,2021-04-11,Sha Tin (HK),Hong Kong card during UK BST,middle,6,8:00,08:00,20:00,782258,Stanley Gap Handicap (Div I) (3yo+) (Course C...
6,2021-04-11,Sha Tin (HK),Hong Kong card during UK BST,last,11,10:45,10:45,22:45,782253,Cheung Lin Shan Handicap (3yo+) (Course C) (T...
7,2024-06-30,Curragh (IRE),Irish afternoon card using low hours,first,1,1:10,01:10,13:10,871044,Paddy Power From The Horses Mouth Podcast Hand...
8,2024-06-30,Curragh (IRE),Irish afternoon card using low hours,middle,5,3:25,03:25,15:25,871657,Colm McLoughlin Celebration Stakes ()
9,2024-06-30,Curragh (IRE),Irish afternoon card using low hours,last,9,5:50,05:50,17:50,871048,Dubai Duty Free Irish EBF Ragusa Handicap (Pre...


### Partial findings from the second validation pilot

Direct external evidence confirms the afternoon candidate for the Irish and
British examples.

For the Curragh on 30 June 2024:

* source `off`: `1:10`;
* reconstructed advertised UK/Irish time: `13:10`;
* externally reported actual off-time: approximately `13:10:07`.

For Newmarket on 13 July 2024:

* source `off`: `1:40`;
* reconstructed advertised UK time: `13:40`;
* externally reported actual off-time: approximately `13:41`.

These examples further support treating source `off` as an advertised or
scheduled minute rather than the exact actual starting timestamp.

The Auteuil, Kyoto and Sha Tin candidates remain pending direct race-level
external confirmation. Their plausible timezone interpretation alone is not
sufficient to mark them validated.

In [34]:
# Record only the second-pilot findings directly confirmed by external
# race-level evidence.
#
# Auteuil, Kyoto and Sha Tin remain pending rather than being inferred from
# timezone plausibility alone.

second_pilot_validation_results = pd.DataFrame(
    [
        {
            "date": "2024-06-30",
            "course": "Curragh (IRE)",
            "race_id": 871044,
            "raw_off": "1:10",
            "reconstructed_uk_time": "13:10",
            "selected_candidate": "B",
            "external_advertised_time": "13:10",
            "external_actual_off_time": "13:10:07",
            "external_source": "Racing Post",
            "validation_result": "confirmed",
            "temporal_semantics": (
                "advertised_time_with_second_level_actual_off"
            ),
            "confidence": "high",
        },
        {
            "date": "2024-07-13",
            "course": "Newmarket (July)",
            "race_id": 870497,
            "raw_off": "1:40",
            "reconstructed_uk_time": "13:40",
            "selected_candidate": "B",
            "external_advertised_time": "13:40",
            "external_actual_off_time": "13:41:02",
            "external_source": "Racing TV / Sky Sports",
            "validation_result": "confirmed",
            "temporal_semantics": (
                "advertised_time_not_exact_actual_off"
            ),
            "confidence": "high",
        },
    ]
)

second_pilot_validation_results

,date,course,race_id,raw_off,reconstructed_uk_time,selected_candidate,external_advertised_time,external_actual_off_time,external_source,validation_result,temporal_semantics,confidence
0,2024-06-30,Curragh (IRE),871044,1:10,13:10,B,13:10,13:10:07,Racing Post,confirmed,advertised_time_with_second_level_actual_off,high
1,2024-07-13,Newmarket (July),870497,1:40,13:40,B,13:40,13:41:02,Racing TV / Sky Sports,confirmed,advertised_time_not_exact_actual_off,high


### Combined confirmed validation evidence

The two validation pilots currently provide direct race-level confirmation for:

* Caulfield, Australia;
* Randwick, Australia;
* Gulfstream Park, United States;
* Keeneland, United States;
* the Curragh, Ireland;
* Newmarket, Britain.

Both possible pre-boundary 12-hour candidates are required in practice.

The evidence therefore rules out any global AM/PM conversion rule. The correct
candidate must be determined using meeting context, racecourse location,
date-specific timezone relationships and external evidence.

The confirmed records also support interpreting source `off` as the advertised
or scheduled UK-facing race time. Where precise actual off-times are available,
they can differ from the source value by seconds or minutes.

In [35]:
# Combine all directly confirmed external-validation results and summarise
# what the evidence currently establishes.

confirmed_validation_results = pd.concat(
    [
        pilot_validation_results,
        second_pilot_validation_results,
    ],
    ignore_index=True,
)

confirmed_validation_summary = pd.DataFrame(
    [
        {
            "confirmed_races": len(confirmed_validation_results),
            "confirmed_meetings": confirmed_validation_results[
                ["date", "course"]
            ].drop_duplicates().shape[0],
            "courses": confirmed_validation_results["course"].nunique(),
            "candidate_a_confirmations": int(
                (
                    confirmed_validation_results["selected_candidate"] == "A"
                ).sum()
            ),
            "candidate_b_confirmations": int(
                (
                    confirmed_validation_results["selected_candidate"] == "B"
                ).sum()
            ),
            "records_with_precise_actual_off": int(
                confirmed_validation_results[
                    "external_actual_off_time"
                ].notna().sum()
            ),
            "advertised_time_matches_source": int(
                (
                    confirmed_validation_results["reconstructed_uk_time"]
                    == confirmed_validation_results[
                        "external_advertised_time"
                    ]
                ).sum()
            ),
            "validation_failures": int(
                (
                    confirmed_validation_results["validation_result"]
                    != "confirmed"
                ).sum()
            ),
        }
    ]
)

confirmed_validation_summary

,confirmed_races,confirmed_meetings,courses,candidate_a_confirmations,candidate_b_confirmations,records_with_precise_actual_off,advertised_time_matches_source,validation_failures
0,10,6,6,4,6,3,10,0


## Database treatment of `off`

The reconstruction target is the advertised UK-facing race time.

Precise actual off-times are not required for the current database because they
would introduce a different temporal concept and reduce consistency across the
source population.

The database should therefore preserve and distinguish:

* `off_raw`: the exact source text;
* `off_format_regime`: the applicable raw-format regime;
* `off_uk_time`: the reconstructed advertised UK civil time;
* `off_reconstruction_method`: direct 24-hour parsing, meeting-context recovery
  or external validation;
* `off_reconstruction_confidence`: deterministic, high, provisional or
  unresolved.

Actual off-times should not overwrite the advertised source time. They may be
added later as a separate enrichment field if a specific analytical use
requires them.

For records from 15 October 2025 onward, `off_uk_time` is deterministically
available from the fixed-width `HH:MM` source value.

For records before 15 October 2025, the source clock omits AM/PM. The correct
UK civil time must therefore be reconstructed at meeting level before conversion
to UTC. The reconstruction will use the meeting sequence, course and date
timezone context, and external validation where the source-only evidence does
not determine the correct 12-hour branch.

## Agreed temporal reconstruction model

The source `date + off` pair will be treated as a UK-facing advertised civil
datetime.

Reconstruction will proceed as follows:

1. preserve source `date` and `off` exactly;
2. recover the correct pre-boundary 12-hour candidate where required;
3. interpret the reconstructed datetime in `Europe/London`;
4. convert it to UTC;
5. use UTC as the canonical database timestamp;
6. derive racecourse-local datetime later from UTC and the course's IANA
   timezone.

The database may therefore contain three legitimate date representations:

* source or UK-facing date;
* UTC date;
* racecourse-local date.

These describe the same advertised race-start instant in different temporal
systems.

The canonical field should be:

`advertised_start_utc`

Source-facing and local representations should remain available for audit,
display and external matching, but should not replace the UTC timestamp.

## Source-supported reconstruction of pre-boundary meetings

The temporal model is now settled:

* source `date + off` represents a UK-facing advertised datetime;
* records from 15 October 2025 onward provide direct `HH:MM` UK civil times;
* earlier records omit AM/PM and require meeting-level reconstruction;
* the selected UK civil datetime will be converted to canonical UTC.

Before relying on manual external validation for every earlier meeting, the
source itself may provide additional evidence.

Many racecourses occur in both temporal regimes. Their post-boundary records
show directly which UK clock windows those courses commonly occupy. These
observed windows can be compared with the two 12-hour candidates for earlier
meetings.

This evidence must be used cautiously:

* race schedules can change by season and meeting type;
* individual courses may stage daytime, evening or night meetings;
* daylight-saving relationships vary by date;
* and empirical course patterns cannot replace external validation where both
  candidates remain plausible.

The next step will measure how much of the pre-boundary population belongs to
courses that also have directly interpretable post-boundary records.

In [36]:
# Measure how much of the pre-boundary population belongs to courses that also
# appear after the fixed-width format transition.
#
# Post-boundary records provide directly interpretable UK civil times for those
# courses. This does not by itself resolve every earlier meeting, but it shows
# how much of the old population can potentially benefit from course-specific
# empirical evidence.

course_regime_coverage = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    course_profiles AS (
        SELECT
            course,
            SUM(
                CASE
                    WHEN date < '2025-10-15'
                    THEN 1 ELSE 0
                END
            ) AS pre_boundary_races,
            COUNT(
                DISTINCT CASE
                    WHEN date < '2025-10-15'
                    THEN date
                END
            ) AS pre_boundary_meeting_dates,
            SUM(
                CASE
                    WHEN date >= '2025-10-15'
                    THEN 1 ELSE 0
                END
            ) AS post_boundary_races,
            COUNT(
                DISTINCT CASE
                    WHEN date >= '2025-10-15'
                    THEN date
                END
            ) AS post_boundary_meeting_dates
        FROM provisional_races
        GROUP BY course
    )
    SELECT
        CASE
            WHEN pre_boundary_races > 0
             AND post_boundary_races > 0
                THEN 'present_in_both_regimes'
            WHEN pre_boundary_races > 0
                THEN 'pre_boundary_only'
            WHEN post_boundary_races > 0
                THEN 'post_boundary_only'
        END AS course_regime_coverage,
        COUNT(*) AS courses,
        SUM(pre_boundary_races) AS pre_boundary_races,
        SUM(pre_boundary_meeting_dates) AS pre_boundary_meeting_dates,
        SUM(post_boundary_races) AS post_boundary_races,
        SUM(post_boundary_meeting_dates) AS post_boundary_meeting_dates
    FROM course_profiles
    GROUP BY
        CASE
            WHEN pre_boundary_races > 0
             AND post_boundary_races > 0
                THEN 'present_in_both_regimes'
            WHEN pre_boundary_races > 0
                THEN 'pre_boundary_only'
            WHEN post_boundary_races > 0
                THEN 'post_boundary_only'
        END
    ORDER BY
        pre_boundary_races DESC,
        post_boundary_races DESC
    """,
    connection,
)

course_regime_coverage

,course_regime_coverage,courses,pre_boundary_races,pre_boundary_meeting_dates,post_boundary_races,post_boundary_meeting_dates
0,present_in_both_regimes,63,103991,14797,5970,834
1,pre_boundary_only,331,74700,16644,0,0
2,post_boundary_only,134,0,0,4382,871


### Stability of post-boundary course time windows

Courses present in both raw-format regimes account for 103,991 pre-boundary
races.

Their post-boundary records provide directly interpretable UK civil times, but
course presence alone is not enough. Some courses may stage meetings at
different times of day, in different seasons or under different schedules.

The next step profiles the post-boundary UK clock distribution for each shared
course. It will measure:

* the observed minimum and maximum time;
* the median time;
* the number of represented meetings;
* and whether the course occupies one compact clock window or a broad range.

Broad or multimodal course patterns should not be used as simple AM/PM rules.

In [37]:
# Profile directly interpretable post-boundary clock windows for courses that
# also occur before the format transition.
#
# A broad observed range does not necessarily mean bad data: a course may stage
# daytime and evening meetings. The profile is intended to identify whether
# course-level evidence is compact enough to assist reconstruction.

shared_course_post_boundary_profile = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            CAST(SUBSTR(off, 1, 2) AS INTEGER) * 60
                + CAST(SUBSTR(off, 4, 2) AS INTEGER)
                AS off_minutes
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    shared_courses AS (
        SELECT course
        FROM provisional_races
        GROUP BY course
        HAVING
            SUM(CASE WHEN date < '2025-10-15' THEN 1 ELSE 0 END) > 0
            AND
            SUM(CASE WHEN date >= '2025-10-15' THEN 1 ELSE 0 END) > 0
    ),
    post_boundary AS (
        SELECT
            p.date,
            p.course,
            p.off,
            p.off_minutes
        FROM provisional_races AS p
        INNER JOIN shared_courses AS s
            ON p.course = s.course
        WHERE p.date >= '2025-10-15'
    )
    SELECT
        course,
        COUNT(*) AS provisional_races,
        COUNT(DISTINCT date) AS meeting_dates,
        MIN(off_minutes) AS minimum_off_minutes,
        MAX(off_minutes) AS maximum_off_minutes,
        MAX(off_minutes) - MIN(off_minutes) AS observed_range_minutes,
        ROUND(AVG(off_minutes), 1) AS average_off_minutes
    FROM post_boundary
    GROUP BY course
    ORDER BY
        observed_range_minutes DESC,
        provisional_races DESC,
        course
    """,
    connection,
)

shared_course_post_boundary_profile

,course,provisional_races,meeting_dates,minimum_off_minutes,maximum_off_minutes,observed_range_minutes,average_off_minutes
0,Newcastle,80,13,305,1055,750,853.0
1,Ascot,103,23,425,1050,625,812.2
2,Southwell (AW),413,51,690,1260,570,1053.3
3,Lingfield (AW),314,40,660,1220,560,878.7
4,Doncaster,145,20,697,1245,548,908.4
...,...,...,...,...,...,...,...
58,Beverley,37,5,820,1076,256,946.1
59,Chester,22,3,810,1040,230,913.2
60,York,28,4,810,1035,225,926.3
61,Brighton,7,1,825,1035,210,930.0


### Course-level evidence is too broad

The shared-course profiles show that many courses occupy wide UK-time ranges
after the format transition.

This is expected because individual courses may stage:

* daytime meetings;
* evening meetings;
* winter and summer programmes;
* fixtures with different race counts and start times.

Course identity alone is therefore too coarse to determine the missing
pre-boundary AM/PM branch.

The next step profiles directly interpretable post-boundary times by course and
calendar month. This provides a more comparable seasonal reference while still
remaining empirical evidence rather than a deterministic reconstruction rule.

In [38]:
# Profile post-boundary UK-time windows by course and calendar month for
# courses represented in both format regimes.
#
# Calendar month is used as a simple seasonal grouping. It captures some of
# the variation caused by daylight-saving relationships and fixture schedules
# without yet requiring a complete course-local timezone model.

shared_course_month_profile = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            CAST(SUBSTR(off, 1, 2) AS INTEGER) * 60
                + CAST(SUBSTR(off, 4, 2) AS INTEGER)
                AS off_minutes
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    shared_courses AS (
        SELECT course
        FROM provisional_races
        GROUP BY course
        HAVING
            SUM(CASE WHEN date < '2025-10-15' THEN 1 ELSE 0 END) > 0
            AND
            SUM(CASE WHEN date >= '2025-10-15' THEN 1 ELSE 0 END) > 0
    )
    SELECT
        p.course,
        SUBSTR(p.date, 6, 2) AS calendar_month,
        COUNT(*) AS provisional_races,
        COUNT(DISTINCT p.date) AS meeting_dates,
        MIN(p.off_minutes) AS minimum_off_minutes,
        MAX(p.off_minutes) AS maximum_off_minutes,
        MAX(p.off_minutes) - MIN(p.off_minutes) AS observed_range_minutes,
        ROUND(AVG(p.off_minutes), 1) AS average_off_minutes
    FROM provisional_races AS p
    INNER JOIN shared_courses AS s
        ON p.course = s.course
    WHERE p.date >= '2025-10-15'
    GROUP BY
        p.course,
        SUBSTR(p.date, 6, 2)
    ORDER BY
        observed_range_minutes DESC,
        provisional_races DESC,
        p.course,
        calendar_month
    """,
    connection,
)

shared_course_month_profile

,course,calendar_month,provisional_races,meeting_dates,minimum_off_minutes,maximum_off_minutes,observed_range_minutes,average_off_minutes
0,Newcastle,03,21,4,355,1055,700,908.1
1,Newcastle,11,16,3,305,940,635,758.6
2,Ascot,11,30,6,425,980,555,748.8
3,Southwell (AW),11,45,5,690,1230,540,1026.2
4,Ascot,05,22,4,550,1050,500,905.7
...,...,...,...,...,...,...,...,...
328,Hexham,12,6,1,775,925,150,850.0
329,Leicester,01,6,1,812,962,150,887.0
330,Ludlow,10,6,1,817,967,150,892.0
331,Sedgefield,01,6,1,820,970,150,895.0


### Raw course text may conflate distinct racecourses

The course-and-month profile still contains implausibly broad time windows.

For example, `Newcastle` in March spans from approximately `05:55` to `17:35`
UK time. That range may not represent one racecourse staging unusually varied
fixtures.

The raw course label may:

* identify different racecourses with the same place name;
* have lost or changed jurisdiction suffixes over time;
* or require the existing course-jurisdiction mapping before temporal patterns
  can be compared safely.

Course text alone must therefore not be used as a timezone or AM/PM
reconstruction key.

The next step inspects the underlying `Newcastle` records responsible for the
broad March range, including race names, types and source dates.

In [39]:
# Inspect post-boundary Newcastle records in March.
#
# The aim is to determine whether the broad clock range belongs to one course
# with varied fixture times or whether the raw label conflates distinct
# jurisdictions or source naming conventions.

newcastle_march_examples = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_id,
        race_name,
        type,
        going,
        COUNT(*) AS runner_rows
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND date >= '2025-10-15'
      AND course = 'Newcastle'
      AND SUBSTR(date, 6, 2) = '03'
    GROUP BY
        date,
        course,
        off,
        race_id,
        race_name,
        type,
        going
    ORDER BY
        date,
        CAST(SUBSTR(off, 1, 2) AS INTEGER),
        CAST(SUBSTR(off, 4, 2) AS INTEGER)
    """,
    connection,
)

newcastle_march_examples

,date,course,off,race_id,race_name,type,going,runner_rows
0,2026-03-03,Newcastle,14:05,912439,Border Minstrel Sunday Lunch Novices Limited H...,Hurdle,Good To Soft,12
1,2026-03-03,Newcastle,14:35,912438,Mini Golf At High Gosforth Park Novices Handic...,Chase,Good To Soft,8
2,2026-03-03,Newcastle,15:05,912435,Virgin Bet Supports Safe Gambling Novices Hurd...,Hurdle,Good To Soft,12
3,2026-03-03,Newcastle,15:35,912437,Get Raceday Ready Handicap Hurdle (Go North Br...,Hurdle,Good To Soft,9
4,2026-03-03,Newcastle,16:05,912436,Virgin Bet A Good Bet Handicap Chase (GBB Race),Chase,Good To Soft,8
5,2026-03-03,Newcastle,16:35,912440,High Gosforth Park Golf Club Mares National Hu...,NH Flat,Good To Soft,7
6,2026-03-06,Newcastle,05:55,914933,Horsepower Newcastle Stakes (Handicap) (Turf),Flat,Good,10
7,2026-03-14,Newcastle,13:40,913371,Guinness Draught Novices Hurdle (GBB Race),Hurdle,Good To Soft,9
8,2026-03-14,Newcastle,14:15,913370,Guinness At The Hawk Ponteland Handicap Chase ...,Chase,Good To Soft,6
9,2026-03-14,Newcastle,14:45,913373,Smirnoff At The Apartment Grand Wedding Handic...,Hurdle,Good To Soft,15


### Course identity must precede temporal inference

The `Newcastle` inspection confirms that one raw course label can represent
different physical racecourses.

Most records shown are National Hunt meetings at Newcastle in Britain. The
single `Horsepower Newcastle Stakes` turf race at `05:55` belongs to Newcastle
in Australia.

Therefore:

* raw `course` text is not a reliable timezone key;
* course-level time windows can be corrupted by jurisdiction conflation;
* temporal reconstruction must use resolved course identity and jurisdiction;
* the existing course-jurisdiction work should be applied before deriving
  course-local datetime or using course history to choose a pre-boundary
  12-hour candidate.

The source `date + course + off` grouping remains empirically collision-free,
but `course` must not be treated as a unique physical venue identifier.

The next step will quantify how many raw course labels contain records assigned
to more than one resolved jurisdiction.

In [40]:
from inside_rails.course_jurisdiction import (
    derive_candidate_course_label,
    derive_candidate_race_jurisdiction,
)

In [41]:
# Assign the established candidate jurisdiction to each post-boundary
# provisional race.
#
# Jurisdiction is derived at race level because raw course labels such as
# Newcastle and Ascot can identify different physical venues depending on race
# context.

post_boundary_race_jurisdictions = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_id,
        race_name,
        type
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND date >= '2025-10-15'
    GROUP BY
        date,
        course,
        off,
        race_id,
        race_name,
        type
    """,
    connection,
)

post_boundary_race_jurisdictions[
    ["candidate_jurisdiction", "jurisdiction_evidence"]
] = post_boundary_race_jurisdictions.apply(
    derive_candidate_race_jurisdiction,
    axis=1,
)

post_boundary_race_jurisdictions["candidate_course_label"] = (
    post_boundary_race_jurisdictions["course"].map(
        derive_candidate_course_label
    )
)

post_boundary_race_jurisdictions[
    [
        "date",
        "course",
        "off",
        "candidate_course_label",
        "candidate_jurisdiction",
        "jurisdiction_evidence",
        "race_name",
    ]
].head()

,date,course,off,candidate_course_label,candidate_jurisdiction,jurisdiction_evidence,race_name
0,2025-10-15,Caulfield,07:10,Caulfield,Australia,historical_suffixed_course_link,Sportsbet Coongy Cup Handicap) (3yo+) (Turf)
1,2025-10-15,Happy Valley,11:40,Happy Valley,Hong Kong,historical_suffixed_course_link,Ngau Chi Wan Handicap (3yo+) (Course A) (Turf)
2,2025-10-15,Happy Valley,12:10,Happy Valley,Hong Kong,historical_suffixed_course_link,Hung Luen Handicap (Div I) (3yo+) (Course A) (...
3,2025-10-15,Happy Valley,12:40,Happy Valley,Hong Kong,historical_suffixed_course_link,Fung Mo Handicap (Div I) (3yo+) (Course A) (Turf)
4,2025-10-15,Happy Valley,13:10,Happy Valley,Hong Kong,historical_suffixed_course_link,Fung Mo Handicap (Div II) (3yo+) (Course A) (T...


In [42]:
# Identify raw course labels that represent races in more than one candidate
# jurisdiction after the format transition.
#
# These labels cannot safely be used as standalone timezone or venue keys.

post_boundary_course_collisions = (
    post_boundary_race_jurisdictions
    .groupby("course", as_index=False)
    .agg(
        provisional_races=("race_id", "size"),
        candidate_jurisdictions=("candidate_jurisdiction", "nunique"),
        jurisdiction_values=(
            "candidate_jurisdiction",
            lambda values: ", ".join(sorted(set(values))),
        ),
        evidence_rules=(
            "jurisdiction_evidence",
            lambda values: ", ".join(sorted(set(values))),
        ),
    )
    .query("candidate_jurisdictions > 1")
    .sort_values(
        ["candidate_jurisdictions", "provisional_races", "course"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

post_boundary_course_collisions

,course,provisional_races,candidate_jurisdictions,jurisdiction_values,evidence_rules
0,Ascot,103,2,"Australia, Great Britain",race_context_course_collision_rule
1,Newcastle,80,2,"Australia, Great Britain",race_context_course_collision_rule


### Resolved course identity removes known jurisdiction collisions

Only two post-boundary raw course labels map to more than one candidate
jurisdiction:

* `Ascot`;
* `Newcastle`.

Both represent British and Australian racecourses and are already handled by
the reusable race-context jurisdiction rules.

Temporal evidence must therefore be grouped by candidate course identity rather
than raw course text.

For the present investigation, candidate course identity consists of:

* `candidate_course_label`;
* `candidate_jurisdiction`.

This remains a derived identity rather than a permanent venue identifier, but
it is sufficient to prevent the known British/Australian course collisions from
corrupting post-boundary UK-time profiles.

In [43]:
# Rebuild the post-boundary time-window profile using resolved candidate course
# identity rather than raw course text.

post_boundary_race_jurisdictions["off_minutes"] = (
    post_boundary_race_jurisdictions["off"].str.slice(0, 2).astype(int) * 60
    + post_boundary_race_jurisdictions["off"].str.slice(3, 5).astype(int)
)

resolved_course_post_boundary_profile = (
    post_boundary_race_jurisdictions
    .groupby(
        ["candidate_course_label", "candidate_jurisdiction"],
        as_index=False,
    )
    .agg(
        provisional_races=("race_id", "size"),
        meeting_dates=("date", "nunique"),
        minimum_off_minutes=("off_minutes", "min"),
        maximum_off_minutes=("off_minutes", "max"),
        average_off_minutes=("off_minutes", "mean"),
    )
)

resolved_course_post_boundary_profile["observed_range_minutes"] = (
    resolved_course_post_boundary_profile["maximum_off_minutes"]
    - resolved_course_post_boundary_profile["minimum_off_minutes"]
)

resolved_course_post_boundary_profile["average_off_minutes"] = (
    resolved_course_post_boundary_profile["average_off_minutes"].round(1)
)

resolved_course_post_boundary_profile = (
    resolved_course_post_boundary_profile
    .sort_values(
        ["observed_range_minutes", "provisional_races"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

resolved_course_post_boundary_profile

,candidate_course_label,candidate_jurisdiction,provisional_races,meeting_dates,minimum_off_minutes,maximum_off_minutes,average_off_minutes,observed_range_minutes
0,Del Mar,United States,26,7,2,1439,1163.8,1437
1,Los Alamitos,United States,2,2,2,1439,720.5,1437
2,Santa Anita,United States,44,30,4,1429,941.6,1425
3,Fair Grounds,United States,11,5,1,1410,1215.1,1409
4,Oaklawn Park,United States,14,11,20,1428,1269.6,1408
...,...,...,...,...,...,...,...,...
194,Scottsville,South Africa,1,1,815,815,815.0,0
195,Strasbourg,France,1,1,678,678,678.0,0
196,Sunland Park,United States,1,1,1397,1397,1397.0,0
197,Valparaiso Sporting Club,Chile,1,1,1365,1365,1365.0,0


### Clock-time ranges must be measured circularly

The resolved course profile still shows apparently enormous ranges for several
United States courses.

These are caused by treating the 24-hour clock as a linear scale. Times near
midnight are close together in reality but far apart numerically:

* `23:59` = 1,439 minutes;
* `00:02` = 2 minutes;
* linear range = 1,437 minutes;
* actual separation across midnight = 3 minutes.

Ordinary minimum and maximum clock values are therefore unsuitable for
course-time profiles that cross UK midnight.

The next calculation will replace the linear range with the shortest circular
clock span containing all observed times for each candidate course identity.

In [44]:
# Calculate the shortest circular 24-hour span containing all observed
# post-boundary times for each resolved candidate course identity.
#
# The method sorts distinct clock-minute values, finds the largest empty gap
# around the 24-hour circle, and treats the remainder as the occupied span.

def shortest_circular_span_minutes(values, period=1440):
    distinct_values = sorted(set(int(value) for value in values))

    if len(distinct_values) <= 1:
        return pd.Series(
            {
                "circular_span_minutes": 0,
                "circular_window_start": distinct_values[0] if distinct_values else None,
                "circular_window_end": distinct_values[0] if distinct_values else None,
                "largest_empty_gap_minutes": period,
            }
        )

    gaps = []

    for current_value, next_value in zip(
        distinct_values,
        distinct_values[1:] + [distinct_values[0] + period],
    ):
        gaps.append(
            {
                "gap_start": current_value,
                "gap_end": next_value % period,
                "gap_minutes": next_value - current_value,
            }
        )

    largest_gap = max(gaps, key=lambda item: item["gap_minutes"])

    window_start = largest_gap["gap_end"]
    window_end = largest_gap["gap_start"]
    circular_span = period - largest_gap["gap_minutes"]

    return pd.Series(
        {
            "circular_span_minutes": circular_span,
            "circular_window_start": window_start,
            "circular_window_end": window_end,
            "largest_empty_gap_minutes": largest_gap["gap_minutes"],
        }
    )


resolved_course_circular_profile = (
    post_boundary_race_jurisdictions
    .groupby(
        ["candidate_course_label", "candidate_jurisdiction"]
    )["off_minutes"]
    .apply(shortest_circular_span_minutes)
    .unstack()
    .reset_index()
)

resolved_course_circular_profile = (
    resolved_course_circular_profile
    .merge(
        resolved_course_post_boundary_profile[
            [
                "candidate_course_label",
                "candidate_jurisdiction",
                "provisional_races",
                "meeting_dates",
            ]
        ],
        on=["candidate_course_label", "candidate_jurisdiction"],
        how="left",
    )
    .sort_values(
        ["circular_span_minutes", "provisional_races"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

resolved_course_circular_profile

,candidate_course_label,candidate_jurisdiction,circular_span_minutes,circular_window_start,circular_window_end,largest_empty_gap_minutes,provisional_races,meeting_dates
0,Sha Tin,Hong Kong,685,265,950,755,383,37
1,Happy Valley,Hong Kong,665,285,950,775,251,28
2,Southwell (AW),Great Britain,570,690,1260,870,413,51
3,Lingfield (AW),Great Britain,560,660,1220,880,314,40
4,Doncaster,Great Britain,548,697,1245,892,145,20
...,...,...,...,...,...,...,...,...
194,Scottsville,South Africa,0,815,815,1440,1,1
195,Strasbourg,France,0,678,678,1440,1,1
196,Sunland Park,United States,0,1397,1397,1440,1,1
197,Valparaiso Sporting Club,Chile,0,1365,1365,1440,1,1


### Post-boundary course windows are descriptive only

The circular clock-span calculation corrected the artificial midnight-boundary
ranges produced by linear minimum and maximum values.

However, even resolved course identities can occupy broad UK-time windows across
different dates, seasons and meeting types.

These post-boundary distributions will therefore not be used as deterministic
or probabilistic AM/PM reconstruction rules for earlier records.

They remain useful only as descriptive context and as a way to identify records
that merit external checking.

Pre-boundary times will be accepted only when supported by externally verifiable
race or meeting evidence. Unverified records will remain unresolved rather than
receive an inferred timestamp.

For records before 15 October 2025, no AM/PM candidate will be accepted solely
because it resembles the usual time window for that course.

The reconstructed UK civil time must be supported by externally verifiable
race or meeting evidence. Once one race in a meeting is securely anchored, the
remaining races may be reconstructed from the internally consistent meeting
sequence.

Records that cannot be established with sufficient evidence will retain both
12-hour candidates and an unresolved reconstruction status rather than receive
a guessed timestamp.

## Meeting-level validation and reconstruction workflow

Pre-boundary `off` values will be reconstructed at meeting level rather than
race by race.

For each candidate meeting identified by `date + course`, the workflow will:

1. preserve every raw `off` value;
2. generate the two possible UK civil-time branches for each race;
3. reconstruct the internally consistent race sequence for each branch;
4. resolve candidate course identity and jurisdiction;
5. obtain external evidence for at least one race or the meeting card;
6. use the verified anchor to select the correct 12-hour branch for the meeting;
7. assign reconstructed UK civil datetimes to the remaining races only where
   their sequence is internally consistent with the verified meeting;
8. convert accepted UK datetimes through `Europe/London` to canonical UTC;
9. retain unresolved status where no sufficiently reliable external anchor can
   be obtained.

The resulting reconstruction should distinguish:

* `externally_verified`;
* `meeting_sequence_derived_from_verified_anchor`;
* `unresolved`.

No timestamp should be assigned solely from a course's usual racing window.

In [45]:
# Build one compact reconstruction record per pre-boundary meeting.
#
# Each meeting receives two possible UK civil-time branches separated by
# exactly 12 hours. Race order is reconstructed on the circular 12-hour clock
# by cutting at the largest gap between consecutive displayed times.

pre_boundary_races = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_id,
        race_name,
        type
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND date < '2025-10-15'
    GROUP BY
        date,
        course,
        off,
        race_id,
        race_name,
        type
    """,
    connection,
)

pre_boundary_races[
    ["candidate_jurisdiction", "jurisdiction_evidence"]
] = pre_boundary_races.apply(
    derive_candidate_race_jurisdiction,
    axis=1,
)

pre_boundary_races["candidate_course_label"] = (
    pre_boundary_races["course"].map(derive_candidate_course_label)
)

off_parts = pre_boundary_races["off"].str.split(":", expand=True)

pre_boundary_races["raw_12h_minutes"] = (
    off_parts[0].astype(int).mod(12) * 60
    + off_parts[1].astype(int)
)


def summarise_pre_boundary_meeting(group):
    race_times = (
        group[
            [
                "off",
                "raw_12h_minutes",
                "race_id",
                "race_name",
            ]
        ]
        .drop_duplicates()
        .sort_values("raw_12h_minutes")
        .reset_index(drop=True)
    )

    values = race_times["raw_12h_minutes"].tolist()

    if len(values) == 1:
        ordered_times = race_times.copy()
        meeting_span_minutes = 0
    else:
        circular_gaps = [
            (
                (values[(index + 1) % len(values)] - values[index]) % 720,
                index,
            )
            for index in range(len(values))
        ]

        _, largest_gap_index = max(circular_gaps)

        start_index = (largest_gap_index + 1) % len(values)

        ordered_times = pd.concat(
            [
                race_times.iloc[start_index:],
                race_times.iloc[:start_index],
            ],
            ignore_index=True,
        )

        ordered_minutes = ordered_times["raw_12h_minutes"].tolist()

        unwrapped_minutes = [ordered_minutes[0]]

        for value in ordered_minutes[1:]:
            while value < unwrapped_minutes[-1]:
                value += 720
            unwrapped_minutes.append(value)

        meeting_span_minutes = (
            unwrapped_minutes[-1] - unwrapped_minutes[0]
        )

    first_raw_minutes = int(ordered_times.iloc[0]["raw_12h_minutes"])
    last_raw_minutes = first_raw_minutes + meeting_span_minutes

    branch_a_start = pd.Timestamp(group.name[0]) + pd.Timedelta(
        minutes=first_raw_minutes
    )
    branch_a_end = pd.Timestamp(group.name[0]) + pd.Timedelta(
        minutes=last_raw_minutes
    )

    branch_b_start = branch_a_start + pd.Timedelta(hours=12)
    branch_b_end = branch_a_end + pd.Timedelta(hours=12)

    return pd.Series(
        {
            "candidate_course_label": ", ".join(
                sorted(group["candidate_course_label"].unique())
            ),
            "candidate_jurisdictions": ", ".join(
                sorted(group["candidate_jurisdiction"].unique())
            ),
            "jurisdiction_evidence": ", ".join(
                sorted(group["jurisdiction_evidence"].unique())
            ),
            "race_count": len(ordered_times),
            "first_off_raw": ordered_times.iloc[0]["off"],
            "last_off_raw": ordered_times.iloc[-1]["off"],
            "meeting_span_minutes": meeting_span_minutes,
            "branch_a_start_uk": branch_a_start,
            "branch_a_end_uk": branch_a_end,
            "branch_b_start_uk": branch_b_start,
            "branch_b_end_uk": branch_b_end,
            "first_race_name": ordered_times.iloc[0]["race_name"],
            "last_race_name": ordered_times.iloc[-1]["race_name"],
        }
    )


pre_boundary_meeting_candidates = (
    pre_boundary_races
    .groupby(["date", "course"], sort=False)
    .apply(summarise_pre_boundary_meeting)
    .reset_index()
)

pre_boundary_meeting_candidates.head(10)

,date,course,candidate_course_label,candidate_jurisdictions,jurisdiction_evidence,race_count,first_off_raw,last_off_raw,meeting_span_minutes,branch_a_start_uk,branch_a_end_uk,branch_b_start_uk,branch_b_end_uk,first_race_name,last_race_name
0,2015-01-01,Aqueduct (USA),Aqueduct,United States,explicit_terminal_course_code,1,6:20,6:20,0,2015-01-01 06:20:00,2015-01-01 06:20:00,2015-01-01 18:20:00,2015-01-01 18:20:00,Affectionately Stakes () (4yo+ Fillies & Mares...,Affectionately Stakes () (4yo+ Fillies & Mares...
1,2015-01-01,Ascot (AUS),Ascot,Australia,explicit_terminal_course_code,2,6:50,8:50,120,2015-01-01 06:50:00,2015-01-01 08:50:00,2015-01-01 18:50:00,2015-01-01 20:50:00,La Trice Classic (Fillies & Mares),Golden River Development Perth Cup (Handicap)
2,2015-01-01,Catterick,Catterick,Great Britain,established_unsuffixed_british_course,6,12:30,3:25,175,2015-01-01 00:30:00,2015-01-01 03:25:00,2015-01-01 12:30:00,2015-01-01 15:25:00,Happy New Year Novices Hurdle,yorkshire-outdoors.co.uk Handicap Hurdle
3,2015-01-01,Cheltenham,Cheltenham,Great Britain,established_unsuffixed_british_course,7,12:10,3:40,210,2015-01-01 00:10:00,2015-01-01 03:40:00,2015-01-01 12:10:00,2015-01-01 15:40:00,Neptune Investment Management Novices Hurdle,EBF Cheltenham Pony Club Standard Open Nationa...
4,2015-01-01,Ellerslie (NZ),Ellerslie,New Zealand,explicit_terminal_course_code,6,1:35,5:35,240,2015-01-01 01:35:00,2015-01-01 05:35:00,2015-01-01 13:35:00,2015-01-01 17:35:00,Barneswood Farm Eclipse Stakes,Scot Thrust City of Auckland Cup
5,2015-01-01,Exeter,Exeter,Great Britain,established_unsuffixed_british_course,7,12:35,4:00,205,2015-01-01 00:35:00,2015-01-01 04:00:00,2015-01-01 12:35:00,2015-01-01 16:00:00,Passage House Inn Topsham Novices Handicap Hurdle,Billy Williams Memorial Maiden Open National H...
6,2015-01-01,Fairyhouse (IRE),Fairyhouse,Ireland,explicit_terminal_course_code,7,12:25,3:40,195,2015-01-01 00:25:00,2015-01-01 03:40:00,2015-01-01 12:25:00,2015-01-01 15:40:00,Happy New Year Maiden Hurdle,Follow Fairyhouse On Twitter (Pro/Am) Flat Race
7,2015-01-01,Fakenham,Fakenham,Great Britain,established_unsuffixed_british_course,6,12:40,3:35,175,2015-01-01 00:40:00,2015-01-01 03:35:00,2015-01-01 12:40:00,2015-01-01 15:35:00,Bet totejackpot Selling Hurdle,Happy New Year Handicap Hurdle
8,2015-01-01,Flemington (AUS),Flemington,Australia,explicit_terminal_course_code,1,5:20,5:20,0,2015-01-01 05:20:00,2015-01-01 05:20:00,2015-01-01 17:20:00,2015-01-01 17:20:00,Standish Handicap,Standish Handicap
9,2015-01-01,Laurel Park (USA),Laurel Park,United States,explicit_terminal_course_code,1,7:52,7:52,0,2015-01-01 07:52:00,2015-01-01 07:52:00,2015-01-01 19:52:00,2015-01-01 19:52:00,Maiden Special Weight (4yo+ Fillies & Mares) (...,Maiden Special Weight (4yo+ Fillies & Mares) (...


In [46]:
# Measure the external-validation workload by jurisdiction and meeting size.
#
# Multi-race meetings can be reconstructed from one securely verified anchor
# when the remaining race sequence is internally consistent. Single-race
# meetings require direct evidence for that race.

pre_boundary_validation_workload = (
    pre_boundary_meeting_candidates
    .assign(
        meeting_structure=lambda frame: frame["race_count"].map(
            lambda count: "single_race" if count == 1 else "multi_race"
        )
    )
    .groupby(
        ["candidate_jurisdictions", "meeting_structure"],
        as_index=False,
    )
    .agg(
        meetings=("date", "size"),
        races=("race_count", "sum"),
        earliest_date=("date", "min"),
        latest_date=("date", "max"),
    )
    .sort_values(
        ["candidate_jurisdictions", "meeting_structure"]
    )
    .reset_index(drop=True)
)

pre_boundary_validation_workload

,candidate_jurisdictions,meeting_structure,meetings,races,earliest_date,latest_date
0,Argentina,multi_race,104,336,2015-02-07,2025-10-04
1,Argentina,single_race,80,80,2015-01-03,2025-10-11
2,Australia,multi_race,770,3285,2015-01-01,2025-10-11
3,Australia,single_race,513,513,2015-01-01,2025-09-06
4,Bahrain,multi_race,24,74,2021-11-19,2025-03-20
...,...,...,...,...,...,...
59,United Arab Emirates,single_race,264,264,2015-01-04,2025-04-12
60,United States,multi_race,1186,4147,2015-01-03,2025-10-05
61,United States,single_race,1716,1716,2015-01-01,2025-10-12
62,Uruguay,multi_race,25,53,2015-01-06,2025-01-06


In [47]:
# Summarise the total pre-boundary validation workload by meeting structure.

pre_boundary_validation_summary = (
    pre_boundary_meeting_candidates
    .assign(
        meeting_structure=lambda frame: frame["race_count"].map(
            lambda count: "single_race" if count == 1 else "multi_race"
        )
    )
    .groupby("meeting_structure", as_index=False)
    .agg(
        meetings=("date", "size"),
        races=("race_count", "sum"),
        jurisdictions=("candidate_jurisdictions", "nunique"),
    )
)

pre_boundary_validation_summary["meeting_share_pct"] = (
    pre_boundary_validation_summary["meetings"]
    / pre_boundary_validation_summary["meetings"].sum()
    * 100
).round(2)

pre_boundary_validation_summary["race_share_pct"] = (
    pre_boundary_validation_summary["races"]
    / pre_boundary_validation_summary["races"].sum()
    * 100
).round(2)

pre_boundary_validation_summary

,meeting_structure,meetings,races,jurisdictions,meeting_share_pct,race_share_pct
0,multi_race,25380,172630,31,80.72,96.61
1,single_race,6061,6061,33,19.28,3.39


### Validation workload is concentrated efficiently

Multi-race meetings account for 25,380 of the 31,441 pre-boundary meetings and
172,630 of the 178,691 pre-boundary races.

Therefore, one securely verified anchor per multi-race meeting could establish
the correct 12-hour branch for 96.61% of the old race population, subject to
internal sequence consistency.

Single-race meetings account for 6,061 records, or 3.39% of pre-boundary races.
They cannot inherit a branch from neighbouring races and therefore require
direct external evidence.

The validation workflow will consequently maintain two queues:

* multi-race meetings requiring one reliable meeting or race anchor;
* single-race meetings requiring direct race-level validation.

Unverified records in either queue will remain unresolved.

In [48]:
# Create a reproducible stratified sample of pre-boundary multi-race meetings
# across jurisdiction, year and meeting-size band.

multi_race_validation_candidates = (
    pre_boundary_meeting_candidates
    .query("race_count > 1")
    .copy()
)

multi_race_validation_candidates["year"] = (
    multi_race_validation_candidates["date"].str.slice(0, 4).astype(int)
)

multi_race_validation_candidates["meeting_size_band"] = pd.cut(
    multi_race_validation_candidates["race_count"],
    bins=[1, 3, 6, 9, float("inf")],
    labels=["2_to_3", "4_to_6", "7_to_9", "10_plus"],
)

# Shuffle reproducibly, then retain the first meeting from each stratum.
multi_race_validation_sample = (
    multi_race_validation_candidates
    .sample(frac=1, random_state=11)
    .drop_duplicates(
        subset=[
            "candidate_jurisdictions",
            "year",
            "meeting_size_band",
        ],
        keep="first",
    )
    .sort_values(
        [
            "candidate_jurisdictions",
            "year",
            "meeting_size_band",
            "date",
            "course",
        ]
    )
    .reset_index(drop=True)
)

print("Sample rows:", len(multi_race_validation_sample))

multi_race_validation_sample[
    [
        "date",
        "course",
        "candidate_course_label",
        "candidate_jurisdictions",
        "race_count",
        "first_off_raw",
        "last_off_raw",
        "branch_a_start_uk",
        "branch_a_end_uk",
        "branch_b_start_uk",
        "branch_b_end_uk",
        "first_race_name",
    ]
].head(10)

Sample rows: 489


,date,course,candidate_course_label,candidate_jurisdictions,race_count,first_off_raw,last_off_raw,branch_a_start_uk,branch_a_end_uk,branch_b_start_uk,branch_b_end_uk,first_race_name
0,2015-11-07,Palermo (ARG),Palermo,Argentina,3,7:50,11:00,2015-11-07 07:50:00,2015-11-07 11:00:00,2015-11-07 19:50:00,2015-11-07 23:00:00,Gran Premio Hipodromo de Palermo (3yo+) (Dirt)
1,2015-05-01,Palermo (ARG),Palermo,Argentina,6,6:05,11:50,2015-05-01 06:05:00,2015-05-01 11:50:00,2015-05-01 18:05:00,2015-05-01 23:50:00,Gran Premio Jorge de Atucha (2yo Fillies) (Dirt)
2,2016-07-30,San Isidro (ARG),San Isidro,Argentina,2,8:25,9:25,2016-07-30 08:25:00,2016-07-30 09:25:00,2016-07-30 20:25:00,2016-07-30 21:25:00,Gran Premio Mil Guineas (3yo Fillies) (Turf)
3,2016-06-25,Palermo (ARG),Palermo,Argentina,5,6:30,11:15,2016-06-25 06:30:00,2016-06-25 11:15:00,2016-06-25 18:30:00,2016-06-25 23:15:00,Gran Premio Estrellas Juvenile Fillies (2yo F...
4,2017-09-02,Palermo (ARG),Palermo,Argentina,3,7:30,9:45,2017-09-02 07:30:00,2017-09-02 09:45:00,2017-09-02 19:30:00,2017-09-02 21:45:00,Gran Premio General San Martin (4yo+) (Turf)
5,2017-05-01,Palermo (ARG),Palermo,Argentina,6,5:45,11:30,2017-05-01 05:45:00,2017-05-01 11:30:00,2017-05-01 17:45:00,2017-05-01 23:30:00,Gran Premio Jorge De Atucha (2yo Fillies) (Dirt)
6,2018-12-15,San Isidro (ARG),San Isidro,Argentina,3,7:15,10:10,2018-12-15 07:15:00,2018-12-15 10:10:00,2018-12-15 19:15:00,2018-12-15 22:10:00,Gran Premio Felix de Alzaga Unzue - Internacio...
7,2018-05-01,Palermo (ARG),Palermo,Argentina,5,6:30,11:15,2018-05-01 06:30:00,2018-05-01 11:15:00,2018-05-01 18:30:00,2018-05-01 23:15:00,Premio Montevideo (2yo) (Dirt)
8,2019-12-14,San Isidro (ARG),San Isidro,Argentina,3,9:00,10:35,2019-12-14 09:00:00,2019-12-14 10:35:00,2019-12-14 21:00:00,2019-12-14 22:35:00,Gran Premio Felix de Alzaga Unzue (3yo+) (Str...
9,2019-05-01,Palermo (ARG),Palermo,Argentina,5,6:30,11:15,2019-05-01 06:30:00,2019-05-01 11:15:00,2019-05-01 18:30:00,2019-05-01 23:15:00,Gran Premio Montevideo (2yo) (Dirt)


In [49]:
# Create a compact external-validation pilot:
# the earliest and latest eligible multi-race meeting in each jurisdiction.

ordered_multi_race_candidates = (
    multi_race_validation_candidates
    .sort_values(
        [
            "candidate_jurisdictions",
            "date",
            "course",
        ]
    )
)

earliest_by_jurisdiction = (
    ordered_multi_race_candidates
    .drop_duplicates(
        subset=["candidate_jurisdictions"],
        keep="first",
    )
    .assign(sample_position="earliest")
)

latest_by_jurisdiction = (
    ordered_multi_race_candidates
    .drop_duplicates(
        subset=["candidate_jurisdictions"],
        keep="last",
    )
    .assign(sample_position="latest")
)

compact_multi_race_validation_sample = (
    pd.concat(
        [
            earliest_by_jurisdiction,
            latest_by_jurisdiction,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["date", "course"],
        keep="first",
    )
    .sort_values(
        [
            "candidate_jurisdictions",
            "sample_position",
            "date",
            "course",
        ]
    )
    .reset_index(drop=True)
)

print("Sample rows:", len(compact_multi_race_validation_sample))

compact_multi_race_validation_sample[
    [
        "sample_position",
        "date",
        "course",
        "candidate_jurisdictions",
        "race_count",
        "first_off_raw",
        "last_off_raw",
        "branch_a_start_uk",
        "branch_a_end_uk",
        "branch_b_start_uk",
        "branch_b_end_uk",
        "first_race_name",
    ]
].head(10)

Sample rows: 60


,sample_position,date,course,candidate_jurisdictions,race_count,first_off_raw,last_off_raw,branch_a_start_uk,branch_a_end_uk,branch_b_start_uk,branch_b_end_uk,first_race_name
0,earliest,2015-02-07,San Isidro (ARG),Argentina,2,8:35,10:15,2015-02-07 08:35:00,2015-02-07 10:15:00,2015-02-07 20:35:00,2015-02-07 22:15:00,Premio Juan Shaw (3yo+ Fillies & Mares) (Turf)
1,latest,2025-10-04,San Isidro (ARG),Argentina,3,7:15,10:20,2025-10-04 07:15:00,2025-10-04 10:20:00,2025-10-04 19:15:00,2025-10-04 22:20:00,Gran Premio Suipacha (3yo+) (Straight) (Turf)
2,earliest,2015-01-01,Ascot (AUS),Australia,2,6:50,8:50,2015-01-01 06:50:00,2015-01-01 08:50:00,2015-01-01 18:50:00,2015-01-01 20:50:00,La Trice Classic (Fillies & Mares)
3,latest,2025-10-11,Rosehill (AUS),Australia,5,4:20,6:50,2025-10-11 04:20:00,2025-10-11 06:50:00,2025-10-11 16:20:00,2025-10-11 18:50:00,Canadian Club Roman Consul Stakes (3yo) (Turf)
4,earliest,2021-11-19,Sakhir (BHR),Bahrain,2,11:50,1:00,2021-11-19 11:50:00,2021-11-19 13:00:00,2021-11-19 23:50:00,2021-11-20 01:00:00,Bahrain Petroleum Company (Bapco) Cup (Importe...
5,latest,2025-03-20,Bahrain (BHR),Bahrain,2,5:45,7:45,2025-03-20 05:45:00,2025-03-20 07:45:00,2025-03-20 17:45:00,2025-03-20 19:45:00,REHC Cup (Handicap) (3yo+) (Inner Track) (Turf)
6,earliest,2015-01-20,Gavea (BRZ),Brazil,3,6:15,8:15,2015-01-20 06:15:00,2015-01-20 08:15:00,2015-01-20 18:15:00,2015-01-20 20:15:00,Grande Premio Prefeitura da Cidade do Rio de J...
7,latest,2025-09-06,Cidade Jardim (BRZ),Brazil,2,8:09,8:47,2025-09-06 08:09:00,2025-09-06 08:47:00,2025-09-06 20:09:00,2025-09-06 20:47:00,Grande Premio Ipiranga (3yo) (Turf)
8,earliest,2015-07-05,Woodbine (CAN),Canada,4,8:02,10:38,2015-07-05 08:02:00,2015-07-05 10:38:00,2015-07-05 20:02:00,2015-07-05 22:38:00,Dance Smartly Stakes (3yo+ Fillies & Mares) (...
9,latest,2025-10-04,Woodbine (CAN),Canada,3,8:28,10:41,2025-10-04 08:28:00,2025-10-04 10:41:00,2025-10-04 20:28:00,2025-10-04 22:41:00,Nearctic Stakes presented by The Thoroughbred ...


### Local-time feasibility can eliminate impossible branches

Course and timezone evidence may be used to reject a pre-boundary branch when
that branch places the meeting at a demonstrably implausible racecourse-local
time.

This differs from choosing the branch that merely resembles the course's usual
schedule.

For each meeting:

1. convert both candidate UK civil-time branches to the resolved course-local
   timezone for that historical date;
2. calculate the local start and end of each candidate meeting;
3. reject a branch only where its complete local meeting window falls within a
   verified non-racing period;
4. accept the remaining branch where exactly one candidate remains feasible;
5. retain both candidates where both local windows remain plausible.

A course's typical time distribution may support investigation, but it should
not by itself establish the branch. The exclusion rule must be based on strong
course or jurisdiction evidence that racing is not staged during the rejected
local-time window.

### Accepted use of local-time feasibility

Local racecourse time may be used to resolve the missing pre-boundary AM/PM
branch where one candidate produces a clearly unreasonable racing schedule.

This is treated as a defensible temporal constraint rather than a statistical
prediction from typical race times.

A branch may be selected where:

* course identity and jurisdiction are resolved;
* the applicable historical IANA timezone is known;
* one candidate produces a normal or credible local meeting window;
* and the alternative places the meeting wholly within an unreasonable
  overnight period.

Where both candidates remain credible, the record will remain unresolved unless
external racecard evidence is available.

The selected branch should retain:

* the reconstruction rule;
* the course timezone used;
* the rejected local-time window;
* and a confidence or review status.

The course-to-IANA-timezone mapping should be implemented as reusable project
code rather than defined only inside this notebook.

In [50]:
# Inventory the resolved candidate course identities requiring an IANA timezone.
#
# Timezones must be assigned to physical course identities, not merely to
# jurisdictions, because countries such as Australia, Canada and the United
# States span multiple timezones.

pre_boundary_course_timezone_scope = (
    pre_boundary_races
    .groupby(
        [
            "candidate_course_label",
            "candidate_jurisdiction",
        ],
        as_index=False,
    )
    .agg(
        provisional_races=("race_id", "size"),
        meeting_dates=("date", "nunique"),
        earliest_date=("date", "min"),
        latest_date=("date", "max"),
        raw_course_labels=(
            "course",
            lambda values: ", ".join(sorted(set(values))),
        ),
    )
    .sort_values(
        [
            "candidate_jurisdiction",
            "provisional_races",
            "candidate_course_label",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

timezone_scope_summary = (
    pre_boundary_course_timezone_scope
    .groupby("candidate_jurisdiction", as_index=False)
    .agg(
        candidate_course_identities=("candidate_course_label", "size"),
        provisional_races=("provisional_races", "sum"),
        meeting_dates=("meeting_dates", "sum"),
    )
    .sort_values(
        ["candidate_course_identities", "provisional_races"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print(
    "Candidate course identities:",
    len(pre_boundary_course_timezone_scope),
)

timezone_scope_summary

Candidate course identities: 394


,candidate_jurisdiction,candidate_course_identities,provisional_races,meeting_dates
0,France,73,19361,3103
1,Great Britain,65,105688,15044
2,United States,56,5863,2902
3,Australia,51,3798,1283
4,Ireland,27,29215,3960
5,Japan,21,1470,1437
6,Germany,17,710,519
7,New Zealand,14,236,185
8,South Africa,9,453,195
9,Italy,7,372,187


### Course location is the basis of timezone assignment

An IANA timezone should not be assigned directly from raw course text or broad
jurisdiction.

Each candidate course identity must first be linked to a physical racing venue.
The venue's geographical location then determines the applicable historical
IANA timezone.

The reusable course-location reference should contain:

* `candidate_course_label`;
* `candidate_jurisdiction`;
* `physical_venue_name`;
* `locality`;
* `region`;
* `country`;
* `latitude`;
* `longitude`;
* `iana_timezone`;
* `location_evidence`;
* `location_validation_status`.

The temporal reconstruction workflow will therefore be:

1. resolve the candidate physical venue;
2. obtain its geographical location;
3. assign the location's IANA timezone;
4. convert both UK-time candidates into historical course-local time;
5. reject a candidate only where it creates an unreasonable local meeting
   window.

This course-location reference should become a reusable project dimension
rather than a Notebook 11-only lookup.

In [51]:
# Create the controlled course-location reference scaffold.
#
# Venue location and timezone fields remain deliberately blank until they are
# supported by an external venue source. Race times must not be used to infer
# the location or timezone.

course_location_reference = (
    pre_boundary_course_timezone_scope[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "raw_course_labels",
            "provisional_races",
            "meeting_dates",
            "earliest_date",
            "latest_date",
        ]
    ]
    .copy()
)

course_location_reference["physical_venue_name"] = pd.NA
course_location_reference["locality"] = pd.NA
course_location_reference["region"] = pd.NA
course_location_reference["country"] = pd.NA
course_location_reference["latitude"] = pd.NA
course_location_reference["longitude"] = pd.NA
course_location_reference["iana_timezone"] = pd.NA
course_location_reference["location_evidence"] = pd.NA
course_location_reference["location_validation_status"] = "unassigned"

course_location_reference = course_location_reference[
    [
        "candidate_course_label",
        "candidate_jurisdiction",
        "physical_venue_name",
        "locality",
        "region",
        "country",
        "latitude",
        "longitude",
        "iana_timezone",
        "location_evidence",
        "location_validation_status",
        "raw_course_labels",
        "provisional_races",
        "meeting_dates",
        "earliest_date",
        "latest_date",
    ]
].sort_values(
    [
        "candidate_jurisdiction",
        "candidate_course_label",
    ]
).reset_index(drop=True)

print("Course identities requiring location validation:", len(course_location_reference))

course_location_reference.head(20)

Course identities requiring location validation: 394


,candidate_course_label,candidate_jurisdiction,physical_venue_name,locality,region,country,latitude,longitude,iana_timezone,location_evidence,location_validation_status,raw_course_labels,provisional_races,meeting_dates,earliest_date,latest_date
0,La Plata,Argentina,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,unassigned,La Plata (ARG),37,28,2015-01-18,2025-09-21
1,Palermo,Argentina,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,unassigned,Palermo (ARG),200,76,2015-01-03,2025-10-11
2,San Isidro,Argentina,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,unassigned,San Isidro (ARG),179,80,2015-02-07,2025-10-04
3,Albury,Australia,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,unassigned,Albury (AUS),4,1,2015-03-27,2015-03-27
4,Alice springs,Australia,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,unassigned,Alice springs (AUS),1,1,2019-05-06,2019-05-06
5,Armidale,Australia,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,unassigned,Armidale (AUS),1,1,2020-03-01,2020-03-01
6,Ascot,Australia,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,unassigned,Ascot (AUS),204,140,2015-01-01,2025-05-10
7,Balaklava,Australia,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,unassigned,Balaklava (AUS),1,1,2020-04-29,2020-04-29
8,Ballarat,Australia,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,unassigned,Ballarat (AUS),2,2,2015-04-30,2015-05-10
9,Belmont Park (Perth),Australia,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,unassigned,Belmont Park (Perth) (AUS),31,31,2015-05-16,2024-06-15


## Reusable course-location reference

The 394 candidate course identities will be maintained in a reusable reference
dataset rather than embedded in Notebook 11.

The proposed project files are:

* `data/reference/course_locations.csv`
* `src/inside_rails/course_locations.py`

The CSV will contain the curated physical venue and timezone assignments.

The Python module will:

* load the reference;
* validate required columns;
* reject duplicate course identities;
* validate IANA timezone names;
* and provide a reusable merge function for notebooks and later database
  construction.

Notebook 11 will use this reference to convert both candidate UK datetimes into
historical racecourse-local time.

Location and timezone assignments must be based on external venue-location
evidence, not inferred from the source race times.

In [52]:
# Write the controlled course-location scaffold to the reusable reference area.
#
# Existing manually curated values are preserved if the file already exists.
# Newly discovered course identities are appended with blank location fields.

reference_directory = project_root / "data" / "reference"
reference_directory.mkdir(parents=True, exist_ok=True)

course_locations_path = reference_directory / "course_locations.csv"

reference_columns = [
    "candidate_course_label",
    "candidate_jurisdiction",
    "physical_venue_name",
    "locality",
    "region",
    "country",
    "latitude",
    "longitude",
    "iana_timezone",
    "location_evidence",
    "location_validation_status",
    "raw_course_labels",
    "provisional_races",
    "meeting_dates",
    "earliest_date",
    "latest_date",
]

if course_locations_path.exists():
    existing_course_locations = pd.read_csv(course_locations_path)

    identity_columns = [
        "candidate_course_label",
        "candidate_jurisdiction",
    ]

    course_location_export = (
        existing_course_locations
        .merge(
            course_location_reference,
            on=identity_columns,
            how="outer",
            suffixes=("_existing", "_current"),
        )
    )

    for column in reference_columns:
        if column in identity_columns:
            continue

        existing_column = f"{column}_existing"
        current_column = f"{column}_current"

        if existing_column in course_location_export.columns:
            course_location_export[column] = (
                course_location_export[existing_column]
                .combine_first(course_location_export.get(current_column))
            )
        elif current_column in course_location_export.columns:
            course_location_export[column] = course_location_export[current_column]

    course_location_export = course_location_export[reference_columns]

else:
    course_location_export = course_location_reference[
        reference_columns
    ].copy()

course_location_export = (
    course_location_export
    .sort_values(
        [
            "candidate_jurisdiction",
            "candidate_course_label",
        ]
    )
    .reset_index(drop=True)
)

course_location_export.to_csv(
    course_locations_path,
    index=False,
)

print("Reference file:", course_locations_path)
print("Course identities:", len(course_location_export))
print(
    "Assigned timezones:",
    course_location_export["iana_timezone"].notna().sum(),
)

Reference file: /home/rob/Documents/inside-rails-horse-racing/data/reference/course_locations.csv
Course identities: 394
Assigned timezones: 394


In [53]:
# Create the reusable course-location reference module.
#
# The module validates the reference structure, uniqueness, coordinates and
# IANA timezone names. It does not assign or infer any locations.

course_locations_module_path = (
    project_root / "src" / "inside_rails" / "course_locations.py"
)

course_locations_module_text = '''"""Load and validate the curated course-location reference."""

from __future__ import annotations

from pathlib import Path
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

import pandas as pd


IDENTITY_COLUMNS = [
    "candidate_course_label",
    "candidate_jurisdiction",
]

REQUIRED_COLUMNS = [
    "candidate_course_label",
    "candidate_jurisdiction",
    "physical_venue_name",
    "locality",
    "region",
    "country",
    "latitude",
    "longitude",
    "iana_timezone",
    "location_evidence",
    "location_validation_status",
]


def load_course_locations(path: str | Path) -> pd.DataFrame:
    """Load and validate the curated course-location reference."""

    reference_path = Path(path)
    frame = pd.read_csv(reference_path)

    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in frame.columns
    ]

    if missing_columns:
        raise ValueError(
            "Course-location reference is missing required columns: "
            + ", ".join(missing_columns)
        )

    duplicate_mask = frame.duplicated(
        subset=IDENTITY_COLUMNS,
        keep=False,
    )

    if duplicate_mask.any():
        duplicates = (
            frame.loc[duplicate_mask, IDENTITY_COLUMNS]
            .drop_duplicates()
            .to_dict("records")
        )
        raise ValueError(
            f"Duplicate candidate course identities found: {duplicates}"
        )

    assigned_timezone_mask = frame["iana_timezone"].notna()

    invalid_timezones = []

    for timezone_name in sorted(
        frame.loc[assigned_timezone_mask, "iana_timezone"].unique()
    ):
        try:
            ZoneInfo(str(timezone_name))
        except ZoneInfoNotFoundError:
            invalid_timezones.append(str(timezone_name))

    if invalid_timezones:
        raise ValueError(
            "Invalid IANA timezone names: "
            + ", ".join(invalid_timezones)
        )

    latitude_values = pd.to_numeric(
        frame["latitude"],
        errors="coerce",
    )
    longitude_values = pd.to_numeric(
        frame["longitude"],
        errors="coerce",
    )

    invalid_latitude_mask = (
        frame["latitude"].notna()
        & ~latitude_values.between(-90, 90)
    )
    invalid_longitude_mask = (
        frame["longitude"].notna()
        & ~longitude_values.between(-180, 180)
    )

    if invalid_latitude_mask.any():
        raise ValueError("Latitude values must fall between -90 and 90.")

    if invalid_longitude_mask.any():
        raise ValueError("Longitude values must fall between -180 and 180.")

    return frame


def merge_course_locations(
    races: pd.DataFrame,
    course_locations: pd.DataFrame,
) -> pd.DataFrame:
    """Attach curated course-location fields to resolved race identities."""

    return races.merge(
        course_locations,
        on=IDENTITY_COLUMNS,
        how="left",
        validate="many_to_one",
    )
'''

course_locations_module_path.write_text(
    course_locations_module_text,
    encoding="utf-8",
)

print("Module file:", course_locations_module_path)
print("Bytes written:", course_locations_module_path.stat().st_size)

Module file: /home/rob/Documents/inside-rails-horse-racing/src/inside_rails/course_locations.py
Bytes written: 2935


In [54]:
# Reload the new module so this notebook sees the file created during the
# current kernel session.

import importlib
import inside_rails.course_locations as course_locations

course_locations = importlib.reload(course_locations)

validated_course_locations = course_locations.load_course_locations(
    course_locations_path
)

print("Validated rows:", len(validated_course_locations))
print(
    "Duplicate identities:",
    validated_course_locations.duplicated(
        subset=[
            "candidate_course_label",
            "candidate_jurisdiction",
        ]
    ).sum(),
)
print(
    "Assigned timezones:",
    validated_course_locations["iana_timezone"].notna().sum(),
)

Validated rows: 394
Duplicate identities: 0
Assigned timezones: 394


## Resume after completion of the course-location reference

Notebook 12 has completed the permanent course-location reference:

* 394 candidate course identities;
* 394 valid IANA timezone assignments;
* zero unresolved timezone assignments;
* 51 distinct IANA timezones.

Notebook 11 will now use:

`data/reference/course_locations.csv`

through the reusable functions in:

`src/inside_rails/course_locations.py`

The established temporal interpretation remains:

1. source `date + off` represents a UK-facing advertised civil datetime;
2. pre-15 October 2025 values require reconstruction of the missing AM/PM branch;
3. the reconstructed UK datetime is interpreted using `Europe/London`;
4. it is converted to canonical UTC;
5. the racecourse-local datetime is then derived from UTC using the governed
   course IANA timezone.

The course timezone must not be attached directly to the raw source
`date + off` pair, because the source clock is not racecourse-local.

The next step is to load and validate the completed course-location reference,
then measure whether every provisional race can be joined to exactly one
governed course identity.

In [55]:
# Load the governed course-location reference and attach it to every
# provisional race using the resolved candidate course identity.

validated_course_locations = course_locations.load_course_locations(
    course_locations_path
)

all_provisional_races = pd.concat(
    [
        pre_boundary_races,
        post_boundary_race_jurisdictions,
    ],
    ignore_index=True,
)

all_provisional_races_with_locations = (
    course_locations.merge_course_locations(
        all_provisional_races,
        validated_course_locations,
    )
)

course_location_join_summary = pd.DataFrame(
    [
        {
            "measure": "provisional races",
            "value": len(all_provisional_races_with_locations),
        },
        {
            "measure": "matched course locations",
            "value": all_provisional_races_with_locations[
                "iana_timezone"
            ].notna().sum(),
        },
        {
            "measure": "unmatched course locations",
            "value": all_provisional_races_with_locations[
                "iana_timezone"
            ].isna().sum(),
        },
        {
            "measure": "distinct IANA timezones",
            "value": all_provisional_races_with_locations[
                "iana_timezone"
            ].nunique(),
        },
    ]
)

course_location_join_summary

,measure,value
0,provisional races,189043
1,matched course locations,189042
2,unmatched course locations,1
3,distinct IANA timezones,51


In [56]:
# Inspect the one provisional race that did not join to the governed
# course-location reference.

unmatched_course_location_races = (
    all_provisional_races_with_locations.loc[
        all_provisional_races_with_locations["iana_timezone"].isna(),
        [
            "date",
            "course",
            "off",
            "race_id",
            "race_name",
            "type",
            "candidate_course_label",
            "candidate_jurisdiction",
            "jurisdiction_evidence",
        ],
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

unmatched_course_location_races

,date,course,off,race_id,race_name,type,candidate_course_label,candidate_jurisdiction,jurisdiction_evidence
0,2026-04-25,Firenze,16:15,919235,Premio Natale di Roma (Turf),Flat,Firenze,Italy,curated_venue_reference


In [57]:
# Confirm whether Firenze is absent from the governed reference and inspect
# the existing Italian course-location records before extending it.

validated_course_locations.loc[
    validated_course_locations["candidate_jurisdiction"].eq("Italy"),
    [
        "candidate_course_label",
        "physical_venue_name",
        "locality",
        "region",
        "country",
        "iana_timezone",
        "location_validation_status",
    ],
].sort_values("candidate_course_label").reset_index(drop=True)

,candidate_course_label,physical_venue_name,locality,region,country,iana_timezone,location_validation_status
0,Capannelle,NaN,NaN,NaN,NaN,Europe/Rome,unassigned
1,Merano,NaN,NaN,NaN,NaN,Europe/Rome,unassigned
2,Naples,NaN,NaN,NaN,NaN,Europe/Rome,unassigned
3,Pisa,NaN,NaN,NaN,NaN,Europe/Rome,unassigned
4,San Siro,NaN,NaN,NaN,NaN,Europe/Rome,unassigned
5,Siracusa,NaN,NaN,NaN,NaN,Europe/Rome,unassigned
6,Varese,NaN,NaN,NaN,NaN,Europe/Rome,unassigned


In [58]:
# Add the new post-boundary Firenze identity to the governed course reference.
#
# The permanent reference currently governs timezone assignments even where
# detailed venue metadata remains unpopulated.

firenze_reference_row = {
    "candidate_course_label": "Firenze",
    "candidate_jurisdiction": "Italy",
    "physical_venue_name": "Ippodromo del Visarno Cesare Meli",
    "locality": "Florence",
    "region": "Tuscany",
    "country": "Italy",
    "latitude": pd.NA,
    "longitude": pd.NA,
    "iana_timezone": "Europe/Rome",
    "location_evidence": (
        "Italian racing authority and City of Florence identify Firenze racing "
        "as Ippodromo del Visarno Cesare Meli"
    ),
    "location_validation_status": "timezone_validated",
    "raw_course_labels": "Firenze",
    "provisional_races": 1,
    "meeting_dates": 1,
    "earliest_date": "2026-04-25",
    "latest_date": "2026-04-25",
}

course_location_export = pd.read_csv(course_locations_path)

identity_mask = (
    course_location_export["candidate_course_label"].eq("Firenze")
    & course_location_export["candidate_jurisdiction"].eq("Italy")
)

if not identity_mask.any():
    course_location_export = pd.concat(
        [
            course_location_export,
            pd.DataFrame([firenze_reference_row]),
        ],
        ignore_index=True,
    )

course_location_export = (
    course_location_export
    .sort_values(
        ["candidate_jurisdiction", "candidate_course_label"]
    )
    .reset_index(drop=True)
)

course_location_export.to_csv(course_locations_path, index=False)

validated_course_locations = course_locations.load_course_locations(
    course_locations_path
)

print("Reference rows:", len(validated_course_locations))
print(
    "Assigned timezones:",
    validated_course_locations["iana_timezone"].notna().sum(),
)

Reference rows: 395
Assigned timezones: 395


In [59]:
# Reattach the updated governed course-location reference and verify complete
# provisional-race coverage.

all_provisional_races_with_locations = (
    course_locations.merge_course_locations(
        all_provisional_races,
        validated_course_locations,
    )
)

course_location_join_summary = pd.DataFrame(
    [
        {
            "measure": "provisional races",
            "value": len(all_provisional_races_with_locations),
        },
        {
            "measure": "matched course locations",
            "value": all_provisional_races_with_locations[
                "iana_timezone"
            ].notna().sum(),
        },
        {
            "measure": "unmatched course locations",
            "value": all_provisional_races_with_locations[
                "iana_timezone"
            ].isna().sum(),
        },
        {
            "measure": "distinct governed course identities",
            "value": all_provisional_races_with_locations[
                [
                    "candidate_course_label",
                    "candidate_jurisdiction",
                ]
            ].drop_duplicates().shape[0],
        },
        {
            "measure": "distinct IANA timezones",
            "value": all_provisional_races_with_locations[
                "iana_timezone"
            ].nunique(),
        },
    ]
)

course_location_join_summary

,measure,value
0,provisional races,189043
1,matched course locations,189043
2,unmatched course locations,0
3,distinct governed course identities,395
4,distinct IANA timezones,51


### Complete course-timezone coverage

The governed course-location reference now joins successfully to every
provisional race.

Coverage is:

* 189,043 provisional races;
* 189,043 matched course identities;
* zero unmatched races;
* 395 governed course identities;
* 51 distinct IANA timezones.

The physical venue and timezone dependency is therefore complete for the
current source population.

The next step is to construct both pre-boundary UK datetime candidates, attach
historical `Europe/London`, convert each candidate to UTC, and then derive both
racecourse-local candidate datetimes using the governed course timezone.

In [62]:
# Reconstruct two race-level UK civil-time candidates for every pre-boundary
# race. Candidate B is exactly 12 hours after Candidate A.
#
# Meeting order is recovered by cutting the circular 12-hour clock at its
# largest gap, then unwrapping the remaining sequence.

def assign_meeting_candidate_minutes(group):
    result = group.copy()

    distinct_minutes = sorted(
        result["raw_12h_minutes"].drop_duplicates().astype(int)
    )

    if len(distinct_minutes) == 1:
        unwrapped_lookup = {
            distinct_minutes[0]: distinct_minutes[0]
        }
    else:
        gaps = []

        for index, current_value in enumerate(distinct_minutes):
            next_value = (
                distinct_minutes[(index + 1) % len(distinct_minutes)]
                + (720 if index == len(distinct_minutes) - 1 else 0)
            )

            gaps.append(
                {
                    "index": index,
                    "gap_minutes": next_value - current_value,
                }
            )

        largest_gap_index = max(
            gaps,
            key=lambda item: item["gap_minutes"],
        )["index"]

        start_index = (largest_gap_index + 1) % len(distinct_minutes)

        ordered_minutes = (
            distinct_minutes[start_index:]
            + distinct_minutes[:start_index]
        )

        unwrapped_minutes = [ordered_minutes[0]]

        for value in ordered_minutes[1:]:
            while value < unwrapped_minutes[-1]:
                value += 720
            unwrapped_minutes.append(value)

        unwrapped_lookup = dict(
            zip(ordered_minutes, unwrapped_minutes)
        )

    result["candidate_a_minutes_from_source_date"] = (
        result["raw_12h_minutes"].map(unwrapped_lookup)
    )

    result["candidate_b_minutes_from_source_date"] = (
        result["candidate_a_minutes_from_source_date"] + 720
    )

    return result


meeting_candidate_frames = []

for _, meeting_group in pre_boundary_races.groupby(
    ["date", "course"],
    sort=False,
):
    meeting_candidate_frames.append(
        assign_meeting_candidate_minutes(meeting_group)
    )

pre_boundary_race_candidates = pd.concat(
    meeting_candidate_frames,
    ignore_index=True,
)

source_dates = pd.to_datetime(
    pre_boundary_race_candidates["date"]
)

pre_boundary_race_candidates["candidate_a_uk_naive"] = (
    source_dates
    + pd.to_timedelta(
        pre_boundary_race_candidates[
            "candidate_a_minutes_from_source_date"
        ],
        unit="m",
    )
)

pre_boundary_race_candidates["candidate_b_uk_naive"] = (
    source_dates
    + pd.to_timedelta(
        pre_boundary_race_candidates[
            "candidate_b_minutes_from_source_date"
        ],
        unit="m",
    )
)

pre_boundary_race_candidates[
    [
        "date",
        "course",
        "off",
        "candidate_a_uk_naive",
        "candidate_b_uk_naive",
        "race_name",
    ]
].head(20)

,date,course,off,candidate_a_uk_naive,candidate_b_uk_naive,race_name
0,2015-01-01,Aqueduct (USA),6:20,2015-01-01 06:20:00,2015-01-01 18:20:00,Affectionately Stakes () (4yo+ Fillies & Mares...
1,2015-01-01,Ascot (AUS),6:50,2015-01-01 06:50:00,2015-01-01 18:50:00,La Trice Classic (Fillies & Mares)
2,2015-01-01,Ascot (AUS),8:50,2015-01-01 08:50:00,2015-01-01 20:50:00,Golden River Development Perth Cup (Handicap)
3,2015-01-01,Catterick,12:30,2015-01-01 00:30:00,2015-01-01 12:30:00,Happy New Year Novices Hurdle
4,2015-01-01,Catterick,1:05,2015-01-01 01:05:00,2015-01-01 13:05:00,Buy Your 2015 Annual Badge Today Handicap Hurdle
5,2015-01-01,Catterick,1:40,2015-01-01 01:40:00,2015-01-01 13:40:00,Dine And View At Catterick Races Novices Chase
6,2015-01-01,Catterick,2:15,2015-01-01 02:15:00,2015-01-01 14:15:00,Racing Again On 8th January Novices Hurdle
7,2015-01-01,Catterick,2:50,2015-01-01 02:50:00,2015-01-01 14:50:00,Watch On 3 Devices racinguk.com/anywhere Handi...
8,2015-01-01,Catterick,3:25,2015-01-01 03:25:00,2015-01-01 15:25:00,yorkshire-outdoors.co.uk Handicap Hurdle
9,2015-01-01,Cheltenham,12:10,2015-01-01 00:10:00,2015-01-01 12:10:00,Neptune Investment Management Novices Hurdle


In [63]:
# Attach the governed course timezone, then convert both UK candidate branches
# to UTC and racecourse-local datetime.
#
# The raw source datetime is UK-facing, so Europe/London must be applied before
# converting to the course timezone.

from zoneinfo import ZoneInfo

pre_boundary_race_candidates_with_locations = (
    course_locations.merge_course_locations(
        pre_boundary_race_candidates,
        validated_course_locations,
    )
)

london_timezone = ZoneInfo("Europe/London")


def convert_candidate_datetime(row, candidate_column):
    uk_datetime = row[candidate_column].replace(
        tzinfo=london_timezone
    )

    utc_datetime = uk_datetime.astimezone(
        ZoneInfo("UTC")
    )

    local_datetime = utc_datetime.astimezone(
        ZoneInfo(row["iana_timezone"])
    )

    return pd.Series(
        {
            f"{candidate_column}_aware": uk_datetime,
            f"{candidate_column.replace('_uk_naive', '_utc')}": utc_datetime,
            f"{candidate_column.replace('_uk_naive', '_course_local')}": (
                local_datetime
            ),
        }
    )


candidate_a_conversions = (
    pre_boundary_race_candidates_with_locations.apply(
        convert_candidate_datetime,
        axis=1,
        candidate_column="candidate_a_uk_naive",
    )
)

candidate_b_conversions = (
    pre_boundary_race_candidates_with_locations.apply(
        convert_candidate_datetime,
        axis=1,
        candidate_column="candidate_b_uk_naive",
    )
)

pre_boundary_race_candidates_with_locations = pd.concat(
    [
        pre_boundary_race_candidates_with_locations,
        candidate_a_conversions,
        candidate_b_conversions,
    ],
    axis=1,
)

pre_boundary_race_candidates_with_locations[
    [
        "date",
        "course",
        "off",
        "iana_timezone",
        "candidate_a_uk_naive",
        "candidate_a_utc",
        "candidate_a_course_local",
        "candidate_b_uk_naive",
        "candidate_b_utc",
        "candidate_b_course_local",
    ]
].head(20)

,date,course,off,iana_timezone,candidate_a_uk_naive,candidate_a_utc,candidate_a_course_local,candidate_b_uk_naive,candidate_b_utc,candidate_b_course_local
0,2015-01-01,Aqueduct (USA),6:20,America/New_York,2015-01-01 06:20:00,2015-01-01 06:20:00+00:00,2015-01-01 01:20:00-05:00,2015-01-01 18:20:00,2015-01-01 18:20:00+00:00,2015-01-01 13:20:00-05:00
1,2015-01-01,Ascot (AUS),6:50,Australia/Perth,2015-01-01 06:50:00,2015-01-01 06:50:00+00:00,2015-01-01 14:50:00+08:00,2015-01-01 18:50:00,2015-01-01 18:50:00+00:00,2015-01-02 02:50:00+08:00
2,2015-01-01,Ascot (AUS),8:50,Australia/Perth,2015-01-01 08:50:00,2015-01-01 08:50:00+00:00,2015-01-01 16:50:00+08:00,2015-01-01 20:50:00,2015-01-01 20:50:00+00:00,2015-01-02 04:50:00+08:00
3,2015-01-01,Catterick,12:30,Europe/London,2015-01-01 00:30:00,2015-01-01 00:30:00+00:00,2015-01-01 00:30:00+00:00,2015-01-01 12:30:00,2015-01-01 12:30:00+00:00,2015-01-01 12:30:00+00:00
4,2015-01-01,Catterick,1:05,Europe/London,2015-01-01 01:05:00,2015-01-01 01:05:00+00:00,2015-01-01 01:05:00+00:00,2015-01-01 13:05:00,2015-01-01 13:05:00+00:00,2015-01-01 13:05:00+00:00
5,2015-01-01,Catterick,1:40,Europe/London,2015-01-01 01:40:00,2015-01-01 01:40:00+00:00,2015-01-01 01:40:00+00:00,2015-01-01 13:40:00,2015-01-01 13:40:00+00:00,2015-01-01 13:40:00+00:00
6,2015-01-01,Catterick,2:15,Europe/London,2015-01-01 02:15:00,2015-01-01 02:15:00+00:00,2015-01-01 02:15:00+00:00,2015-01-01 14:15:00,2015-01-01 14:15:00+00:00,2015-01-01 14:15:00+00:00
7,2015-01-01,Catterick,2:50,Europe/London,2015-01-01 02:50:00,2015-01-01 02:50:00+00:00,2015-01-01 02:50:00+00:00,2015-01-01 14:50:00,2015-01-01 14:50:00+00:00,2015-01-01 14:50:00+00:00
8,2015-01-01,Catterick,3:25,Europe/London,2015-01-01 03:25:00,2015-01-01 03:25:00+00:00,2015-01-01 03:25:00+00:00,2015-01-01 15:25:00,2015-01-01 15:25:00+00:00,2015-01-01 15:25:00+00:00
9,2015-01-01,Cheltenham,12:10,Europe/London,2015-01-01 00:10:00,2015-01-01 00:10:00+00:00,2015-01-01 00:10:00+00:00,2015-01-01 12:10:00,2015-01-01 12:10:00+00:00,2015-01-01 12:10:00+00:00


In [66]:
# Classify each distinct UK civil-time candidate before timezone conversion.
#
# Candidate times can be:
# - valid;
# - nonexistent during the spring-forward gap;
# - ambiguous during the autumn clock reversal.
#
# These statuses are useful evidence and should remain explicit.

def classify_london_civil_time(value):
    timestamp = pd.Timestamp(value)

    try:
        timestamp.tz_localize(
            "Europe/London",
            ambiguous="raise",
            nonexistent="raise",
        )
        return "valid"

    except ValueError as error:
        message = str(error).lower()

        if "nonexistent" in message:
            return "nonexistent_dst_time"

        if (
            "ambiguous" in message
            or "cannot infer dst time" in message
        ):
            return "ambiguous_dst_time"

        raise


candidate_datetime_values = (
    pd.concat(
        [
            pre_boundary_race_candidates_with_locations[
                ["candidate_a_uk_naive"]
            ].rename(
                columns={
                    "candidate_a_uk_naive": "uk_naive"
                }
            ),
            pre_boundary_race_candidates_with_locations[
                ["candidate_b_uk_naive"]
            ].rename(
                columns={
                    "candidate_b_uk_naive": "uk_naive"
                }
            ),
        ],
        ignore_index=True,
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

candidate_datetime_values["london_time_status"] = (
    candidate_datetime_values["uk_naive"].map(
        classify_london_civil_time
    )
)

candidate_datetime_status_summary = (
    candidate_datetime_values[
        "london_time_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="distinct_datetimes")
)

display(candidate_datetime_status_summary)

candidate_datetime_values.loc[
    candidate_datetime_values[
        "london_time_status"
    ].ne("valid")
].sort_values("uk_naive").head(30)

,status,distinct_datetimes
0,valid,334749
1,ambiguous_dst_time,84
2,nonexistent_dst_time,45


,uk_naive,london_time_status
3146,2015-03-29 01:00:00,nonexistent_dst_time
3147,2015-03-29 01:30:00,nonexistent_dst_time
3158,2015-03-29 01:45:00,nonexistent_dst_time
13260,2015-10-25 01:00:00,ambiguous_dst_time
13237,2015-10-25 01:15:00,ambiguous_dst_time
13253,2015-10-25 01:20:00,ambiguous_dst_time
13230,2015-10-25 01:25:00,ambiguous_dst_time
13240,2015-10-25 01:30:00,ambiguous_dst_time
13248,2015-10-25 01:35:00,ambiguous_dst_time
13267,2015-10-25 01:40:00,ambiguous_dst_time


In [69]:
# Convert valid UK civil-time candidates to UTC and course-local time.
# Ambiguous or nonexistent London times remain NaT and are not guessed.

def convert_valid_candidate(row, branch):
    naive_column = f"candidate_{branch}_uk_naive"
    status_column = f"candidate_{branch}_london_status"

    if row[status_column] != "valid":
        return pd.Series(
            {
                f"candidate_{branch}_uk_aware": pd.NaT,
                f"candidate_{branch}_utc": pd.NaT,
                f"candidate_{branch}_course_local": pd.NaT,
            }
        )

    uk_aware = pd.Timestamp(
        row[naive_column]
    ).tz_localize("Europe/London")

    utc_datetime = uk_aware.tz_convert("UTC")

    course_local = utc_datetime.tz_convert(
        row["iana_timezone"]
    )

    return pd.Series(
        {
            f"candidate_{branch}_uk_aware": uk_aware,
            f"candidate_{branch}_utc": utc_datetime,
            f"candidate_{branch}_course_local": course_local,
        }
    )


candidate_status_lookup = (
    candidate_datetime_values
    .set_index("uk_naive")["london_time_status"]
)

pre_boundary_race_candidates_with_locations[
    "candidate_a_london_status"
] = pre_boundary_race_candidates_with_locations[
    "candidate_a_uk_naive"
].map(candidate_status_lookup)

pre_boundary_race_candidates_with_locations[
    "candidate_b_london_status"
] = pre_boundary_race_candidates_with_locations[
    "candidate_b_uk_naive"
].map(candidate_status_lookup)

candidate_a_conversions = (
    pre_boundary_race_candidates_with_locations.apply(
        convert_valid_candidate,
        axis=1,
        branch="a",
    )
)

candidate_b_conversions = (
    pre_boundary_race_candidates_with_locations.apply(
        convert_valid_candidate,
        axis=1,
        branch="b",
    )
)

conversion_columns = list(candidate_a_conversions.columns) + list(
    candidate_b_conversions.columns
)

pre_boundary_race_candidates_with_locations = (
    pre_boundary_race_candidates_with_locations
    .drop(columns=conversion_columns, errors="ignore")
)

pre_boundary_race_candidates_with_locations = pd.concat(
    [
        pre_boundary_race_candidates_with_locations,
        candidate_a_conversions,
        candidate_b_conversions,
    ],
    axis=1,
)

pre_boundary_race_candidates_with_locations[
    [
        "candidate_a_london_status",
        "candidate_a_course_local",
        "candidate_b_london_status",
        "candidate_b_course_local",
    ]
].notna().sum()

candidate_a_london_status    178691
candidate_a_course_local     178553
candidate_b_london_status    178691
candidate_b_course_local     178688
dtype: int64

In [71]:
# Summarise both course-local branches at meeting level.
#
# A branch is "wholly overnight" only when:
# - every race in the meeting has a valid conversion; and
# - every local race time falls between 00:00 and 05:59.
#
# Meetings containing withheld DST candidates remain explicit.
import numpy as np
working_candidates = (
    pre_boundary_race_candidates_with_locations.copy()
)

for branch in ["a", "b"]:
    local_column = f"candidate_{branch}_course_local"

    working_candidates[
        f"candidate_{branch}_local_minutes"
    ] = working_candidates[local_column].map(
        lambda value: (
            value.hour * 60 + value.minute
            if pd.notna(value)
            else pd.NA
        )
    )

    working_candidates[
        f"candidate_{branch}_local_naive"
    ] = working_candidates[local_column].map(
        lambda value: (
            value.replace(tzinfo=None)
            if pd.notna(value)
            else pd.NaT
        )
    )

    working_candidates[
        f"candidate_{branch}_overnight_row"
    ] = working_candidates[
        f"candidate_{branch}_local_minutes"
    ].map(
        lambda value: (
            0 <= value <= 359
            if pd.notna(value)
            else False
        )
    )


local_meeting_candidate_summary = (
    working_candidates
    .groupby(
        ["date", "course"],
        as_index=False,
        sort=False,
    )
    .agg(
        candidate_course_label=(
            "candidate_course_label",
            "first",
        ),
        candidate_jurisdiction=(
            "candidate_jurisdiction",
            "first",
        ),
        iana_timezone=("iana_timezone", "first"),
        race_count=("race_id", "size"),
        candidate_a_valid_races=(
            "candidate_a_course_local",
            "count",
        ),
        candidate_b_valid_races=(
            "candidate_b_course_local",
            "count",
        ),
        candidate_a_local_start=(
            "candidate_a_local_naive",
            "min",
        ),
        candidate_a_local_end=(
            "candidate_a_local_naive",
            "max",
        ),
        candidate_b_local_start=(
            "candidate_b_local_naive",
            "min",
        ),
        candidate_b_local_end=(
            "candidate_b_local_naive",
            "max",
        ),
        candidate_a_overnight_races=(
            "candidate_a_overnight_row",
            "sum",
        ),
        candidate_b_overnight_races=(
            "candidate_b_overnight_row",
            "sum",
        ),
    )
)

local_meeting_candidate_summary[
    "candidate_a_wholly_overnight"
] = (
    local_meeting_candidate_summary[
        "candidate_a_valid_races"
    ].eq(local_meeting_candidate_summary["race_count"])
    & local_meeting_candidate_summary[
        "candidate_a_overnight_races"
    ].eq(local_meeting_candidate_summary["race_count"])
)

local_meeting_candidate_summary[
    "candidate_b_wholly_overnight"
] = (
    local_meeting_candidate_summary[
        "candidate_b_valid_races"
    ].eq(local_meeting_candidate_summary["race_count"])
    & local_meeting_candidate_summary[
        "candidate_b_overnight_races"
    ].eq(local_meeting_candidate_summary["race_count"])
)

has_dst_withheld = (
    local_meeting_candidate_summary[
        "candidate_a_valid_races"
    ].lt(local_meeting_candidate_summary["race_count"])
    | local_meeting_candidate_summary[
        "candidate_b_valid_races"
    ].lt(local_meeting_candidate_summary["race_count"])
)

candidate_a_overnight = local_meeting_candidate_summary[
    "candidate_a_wholly_overnight"
]

candidate_b_overnight = local_meeting_candidate_summary[
    "candidate_b_wholly_overnight"
]

local_meeting_candidate_summary[
    "preliminary_branch_result"
] = np.select(
    [
        has_dst_withheld,
        candidate_a_overnight & ~candidate_b_overnight,
        candidate_b_overnight & ~candidate_a_overnight,
        candidate_a_overnight & candidate_b_overnight,
    ],
    [
        "dst_edge_requires_review",
        "candidate_b_only_feasible",
        "candidate_a_only_feasible",
        "both_wholly_overnight",
    ],
    default="both_not_wholly_overnight",
)

local_meeting_candidate_summary[
    "preliminary_branch_result"
].value_counts(
    dropna=False
).rename_axis(
    "result"
).reset_index(
    name="meetings"
)

,result,meetings
0,candidate_b_only_feasible,17362
1,both_not_wholly_overnight,10043
2,candidate_a_only_feasible,3943
3,dst_edge_requires_review,93


### Conservative course-local feasibility result

The two pre-boundary 12-hour branches were converted from UK civil time through
UTC into the governed historical racecourse timezone.

A branch was rejected only when every race in the meeting fell between
00:00 and 05:59 course-local time. This is intentionally a hard, conservative
sanity constraint rather than a model of normal race scheduling.

Across 31,441 pre-boundary meetings:

* 17,362 meetings retain candidate B only;
* 3,943 meetings retain candidate A only;
* 10,043 meetings retain both non-overnight branches;
* 93 meetings require separate review because at least one candidate touches
  an ambiguous or nonexistent `Europe/London` DST time.

The overnight-feasibility rule therefore resolves 21,305 meetings, or 67.8% of
the pre-boundary meeting population, while leaving 32.2% unresolved rather than
selecting a branch from weaker assumptions about customary race times.

In [73]:
# Profile meetings where both branches survive the conservative overnight test.

unresolved_local_meetings = (
    local_meeting_candidate_summary.loc[
        local_meeting_candidate_summary[
            "preliminary_branch_result"
        ].eq("both_not_wholly_overnight")
    ]
    .copy()
)

for branch in ["a", "b"]:
    unresolved_local_meetings[
        f"candidate_{branch}_start_hour"
    ] = unresolved_local_meetings[
        f"candidate_{branch}_local_start"
    ].dt.hour

    unresolved_local_meetings[
        f"candidate_{branch}_end_hour"
    ] = unresolved_local_meetings[
        f"candidate_{branch}_local_end"
    ].dt.hour

unresolved_start_hour_summary = (
    unresolved_local_meetings
    .groupby(
        [
            "candidate_a_start_hour",
            "candidate_b_start_hour",
        ],
        as_index=False,
    )
    .agg(
        meetings=("race_count", "size"),
        races=("race_count", "sum"),
    )
    .sort_values(
        ["meetings", "races"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    pd.DataFrame(
        {
            "measure": [
                "unresolved meetings",
                "unresolved races",
                "distinct jurisdictions",
                "distinct course timezones",
            ],
            "value": [
                len(unresolved_local_meetings),
                unresolved_local_meetings["race_count"].sum(),
                unresolved_local_meetings[
                    "candidate_jurisdiction"
                ].nunique(),
                unresolved_local_meetings[
                    "iana_timezone"
                ].nunique(),
            ],
        }
    )
)

unresolved_start_hour_summary.head(30)

,measure,value
0,unresolved meetings,10043
1,unresolved races,66199
2,distinct jurisdictions,29
3,distinct course timezones,41


,candidate_a_start_hour,candidate_b_start_hour,meetings,races
0,5,17,3153,21681
1,4,16,2045,15343
2,6,18,1160,5845
3,11,23,705,5416
4,2,14,568,4338
5,3,15,513,3751
6,1,13,499,4218
7,7,19,397,1418
8,18,6,289,1642
9,19,7,243,867


In [74]:
# Show where the unresolved start-hour patterns occur.

unresolved_jurisdiction_summary = (
    unresolved_local_meetings
    .groupby(
        [
            "candidate_jurisdiction",
            "candidate_a_start_hour",
            "candidate_b_start_hour",
        ],
        as_index=False,
    )
    .agg(
        meetings=("race_count", "size"),
        races=("race_count", "sum"),
        distinct_courses=("candidate_course_label", "nunique"),
    )
    .sort_values(
        ["meetings", "races"],
        ascending=False,
    )
    .reset_index(drop=True)
)

unresolved_jurisdiction_summary.head(40)

,candidate_jurisdiction,candidate_a_start_hour,candidate_b_start_hour,meetings,races,distinct_courses
0,Great Britain,5,17,2217,15375,61
1,Great Britain,4,16,1178,9154,56
2,Ireland,5,17,762,5539,24
3,Great Britain,6,18,653,4153,49
4,Ireland,4,16,569,4312,24
5,France,1,13,414,3617,7
6,France,11,23,313,2469,20
7,Great Britain,11,23,289,2206,34
8,Great Britain,2,14,223,1757,47
9,France,2,14,205,1648,8


In [75]:
# Sensitivity test for stricter early-morning feasibility cutoffs.
#
# For each cutoff, a branch is rejected only when every valid race in the
# meeting occurs before that local time.
#
# This does not yet choose a final cutoff. It shows the effect of using
# 06:00, 07:00, 08:00, or 09:00 as the hard lower boundary.

cutoff_results = []

for cutoff_hour in [6, 7, 8, 9]:
    cutoff_minutes = cutoff_hour * 60

    candidate_a_wholly_too_early = (
        local_meeting_candidate_summary[
            "candidate_a_valid_races"
        ].eq(local_meeting_candidate_summary["race_count"])
        & local_meeting_candidate_summary[
            "candidate_a_local_end"
        ].dt.hour.mul(60).add(
            local_meeting_candidate_summary[
                "candidate_a_local_end"
            ].dt.minute
        ).lt(cutoff_minutes)
    )

    candidate_b_wholly_too_early = (
        local_meeting_candidate_summary[
            "candidate_b_valid_races"
        ].eq(local_meeting_candidate_summary["race_count"])
        & local_meeting_candidate_summary[
            "candidate_b_local_end"
        ].dt.hour.mul(60).add(
            local_meeting_candidate_summary[
                "candidate_b_local_end"
            ].dt.minute
        ).lt(cutoff_minutes)
    )

    candidate_a_only = (
        candidate_b_wholly_too_early
        & ~candidate_a_wholly_too_early
    )

    candidate_b_only = (
        candidate_a_wholly_too_early
        & ~candidate_b_wholly_too_early
    )

    unresolved = ~(
        candidate_a_only
        | candidate_b_only
    )

    cutoff_results.append(
        {
            "cutoff": f"{cutoff_hour:02d}:00",
            "candidate_a_only_meetings": int(candidate_a_only.sum()),
            "candidate_b_only_meetings": int(candidate_b_only.sum()),
            "resolved_meetings": int(
                (candidate_a_only | candidate_b_only).sum()
            ),
            "unresolved_meetings": int(unresolved.sum()),
            "resolved_percent": round(
                100
                * (candidate_a_only | candidate_b_only).sum()
                / len(local_meeting_candidate_summary),
                2,
            ),
        }
    )

pd.DataFrame(cutoff_results)

,cutoff,candidate_a_only_meetings,candidate_b_only_meetings,resolved_meetings,unresolved_meetings,resolved_percent
0,06:00,4654,17372,22026,9415,70.06
1,07:00,4850,19083,23933,7508,76.12
2,08:00,5003,20579,25582,5859,81.37
3,09:00,5112,24522,29634,1807,94.25


In [76]:
# Construct observed course-local race times from the post-boundary records,
# where `off` is already an explicit 24-hour UK-facing time.
#
# These observations will later provide course-specific scheduling evidence.

post_boundary_races_with_locations = (
    all_provisional_races_with_locations.loc[
        pd.to_datetime(
            all_provisional_races_with_locations["date"]
        ).ge(pd.Timestamp("2025-10-15"))
    ]
    .copy()
)

post_boundary_races_with_locations[
    "advertised_start_uk_naive"
] = pd.to_datetime(
    post_boundary_races_with_locations["date"].astype(str)
    + " "
    + post_boundary_races_with_locations["off"].astype(str),
    errors="raise",
)

post_boundary_races_with_locations[
    "advertised_start_uk"
] = post_boundary_races_with_locations[
    "advertised_start_uk_naive"
].dt.tz_localize(
    "Europe/London",
    ambiguous="NaT",
    nonexistent="NaT",
)

post_boundary_races_with_locations[
    "advertised_start_utc"
] = post_boundary_races_with_locations[
    "advertised_start_uk"
].dt.tz_convert("UTC")

post_boundary_races_with_locations[
    "advertised_start_course_local"
] = [
    (
        utc_datetime.tz_convert(iana_timezone)
        if pd.notna(utc_datetime)
        else pd.NaT
    )
    for utc_datetime, iana_timezone in zip(
        post_boundary_races_with_locations[
            "advertised_start_utc"
        ],
        post_boundary_races_with_locations[
            "iana_timezone"
        ],
    )
]

pd.DataFrame(
    {
        "measure": [
            "post-boundary races",
            "valid UK datetimes",
            "valid course-local datetimes",
            "distinct governed course identities",
            "distinct course timezones",
        ],
        "value": [
            len(post_boundary_races_with_locations),
            post_boundary_races_with_locations[
                "advertised_start_uk"
            ].notna().sum(),
            post_boundary_races_with_locations[
                "advertised_start_course_local"
            ].notna().sum(),
            post_boundary_races_with_locations[
                [
                    "candidate_course_label",
                    "candidate_jurisdiction",
                ]
            ].drop_duplicates().shape[0],
            post_boundary_races_with_locations[
                "iana_timezone"
            ].nunique(),
        ],
    }
)

,measure,value
0,post-boundary races,10352
1,valid UK datetimes,10352
2,valid course-local datetimes,10352
3,distinct governed course identities,199
4,distinct course timezones,33


In [77]:
# Build post-boundary meeting-level local-time observations and course profiles.

post_boundary_races_with_locations[
    "course_local_minutes"
] = post_boundary_races_with_locations[
    "advertised_start_course_local"
].map(
    lambda value: value.hour * 60 + value.minute
)

post_boundary_meetings = (
    post_boundary_races_with_locations
    .groupby(
        [
            "date",
            "course",
            "candidate_course_label",
            "candidate_jurisdiction",
            "iana_timezone",
        ],
        as_index=False,
        sort=False,
    )
    .agg(
        race_count=("race_id", "size"),
        local_start_minutes=("course_local_minutes", "min"),
        local_end_minutes=("course_local_minutes", "max"),
    )
)

post_boundary_course_profiles = (
    post_boundary_meetings
    .groupby(
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "iana_timezone",
        ],
        as_index=False,
    )
    .agg(
        observed_meetings=("date", "size"),
        observed_races=("race_count", "sum"),
        earliest_observed_start=("local_start_minutes", "min"),
        latest_observed_start=("local_start_minutes", "max"),
        median_start=(
            "local_start_minutes",
            lambda values: values.median(),
        ),
        earliest_observed_end=("local_end_minutes", "min"),
        latest_observed_end=("local_end_minutes", "max"),
        median_end=(
            "local_end_minutes",
            lambda values: values.median(),
        ),
    )
)

def format_minutes(value):
    hours = int(value) // 60
    minutes = int(value) % 60
    return f"{hours:02d}:{minutes:02d}"

for column in [
    "earliest_observed_start",
    "latest_observed_start",
    "median_start",
    "earliest_observed_end",
    "latest_observed_end",
    "median_end",
]:
    post_boundary_course_profiles[
        f"{column}_time"
    ] = post_boundary_course_profiles[column].map(format_minutes)

display(
    pd.DataFrame(
        {
            "measure": [
                "post-boundary meetings",
                "courses with observed meetings",
                "courses with at least 5 observed meetings",
                "courses with exactly 1 observed meeting",
            ],
            "value": [
                len(post_boundary_meetings),
                len(post_boundary_course_profiles),
                post_boundary_course_profiles[
                    "observed_meetings"
                ].ge(5).sum(),
                post_boundary_course_profiles[
                    "observed_meetings"
                ].eq(1).sum(),
            ],
        }
    )
)

post_boundary_course_profiles[
    [
        "candidate_course_label",
        "candidate_jurisdiction",
        "observed_meetings",
        "observed_races",
        "earliest_observed_start_time",
        "median_start_time",
        "latest_observed_start_time",
        "median_end_time",
        "latest_observed_end_time",
    ]
].sort_values(
    ["observed_meetings", "observed_races"],
    ascending=False,
).head(30)

,measure,value
0,post-boundary meetings,1709
1,courses with observed meetings,199
2,courses with at least 5 observed meetings,103
3,courses with exactly 1 observed meeting,46


,candidate_course_label,candidate_jurisdiction,observed_meetings,observed_races,earliest_observed_start_time,median_start_time,latest_observed_start_time,median_end_time,latest_observed_end_time
194,Wolverhampton (AW),Great Britain,64,514,12:57,16:30,18:30,20:30,21:00
170,Southwell (AW),Great Britain,51,413,11:30,16:30,18:00,20:15,21:00
133,Newcastle (AW),Great Britain,49,397,12:30,15:35,17:30,19:15,20:30
108,Lingfield (AW),Great Britain,40,314,11:00,12:37,17:15,16:19,20:20
166,Sha Tin,Hong Kong,37,383,12:25,12:45,19:15,17:55,22:50
49,Dundalk (AW),Ireland,37,279,13:30,16:30,17:30,19:55,20:30
26,Chantilly,France,36,298,11:18,11:57,15:57,16:32,20:05
91,Kempton (AW),Great Britain,33,259,12:27,16:30,18:00,20:10,21:00
41,Deauville,France,32,266,13:40,15:57,16:32,20:10,20:48
162,Santa Anita,United States,30,44,14:05,16:38,18:10,16:46,18:18


In [78]:
# Measure how much of the unresolved pre-boundary population is covered by
# post-boundary course profiles of different sample sizes.

unresolved_with_course_profiles = (
    unresolved_local_meetings
    .merge(
        post_boundary_course_profiles[
            [
                "candidate_course_label",
                "candidate_jurisdiction",
                "observed_meetings",
                "observed_races",
                "earliest_observed_start",
                "latest_observed_start",
                "median_start",
                "earliest_observed_end",
                "latest_observed_end",
                "median_end",
            ]
        ],
        on=[
            "candidate_course_label",
            "candidate_jurisdiction",
        ],
        how="left",
        validate="many_to_one",
    )
)

coverage_rows = []

for minimum_meetings in [1, 3, 5, 10]:
    covered = unresolved_with_course_profiles[
        "observed_meetings"
    ].ge(minimum_meetings)

    coverage_rows.append(
        {
            "minimum_post_boundary_meetings": minimum_meetings,
            "covered_unresolved_meetings": int(covered.sum()),
            "covered_unresolved_races": int(
                unresolved_with_course_profiles.loc[
                    covered,
                    "race_count",
                ].sum()
            ),
            "meeting_coverage_percent": round(
                100 * covered.mean(),
                2,
            ),
            "race_coverage_percent": round(
                100
                * unresolved_with_course_profiles.loc[
                    covered,
                    "race_count",
                ].sum()
                / unresolved_with_course_profiles[
                    "race_count"
                ].sum(),
                2,
            ),
        }
    )

display(pd.DataFrame(coverage_rows))

pd.DataFrame(
    {
        "measure": [
            "unresolved meetings with no post-boundary course profile",
            "unresolved races with no post-boundary course profile",
        ],
        "value": [
            unresolved_with_course_profiles[
                "observed_meetings"
            ].isna().sum(),
            unresolved_with_course_profiles.loc[
                unresolved_with_course_profiles[
                    "observed_meetings"
                ].isna(),
                "race_count",
            ].sum(),
        ],
    }
)

,minimum_post_boundary_meetings,covered_unresolved_meetings,covered_unresolved_races,meeting_coverage_percent,race_coverage_percent
0,1,9277,63625,92.37,96.11
1,3,8529,60590,84.92,91.53
2,5,7605,54713,75.72,82.65
3,10,6834,50119,68.05,75.71


,measure,value
0,unresolved meetings with no post-boundary cour...,766
1,unresolved races with no post-boundary course ...,2574


In [79]:
# Test course-specific schedule compatibility using post-boundary evidence.
#
# A candidate is compatible when both its meeting start and end fall inside
# the course's observed post-boundary range, expanded by a stated margin.
#
# Only courses with at least five observed post-boundary meetings are used.
# This is a sensitivity analysis, not yet a final reconstruction rule.

profile_test_meetings = unresolved_with_course_profiles.loc[
    unresolved_with_course_profiles["observed_meetings"].ge(5)
].copy()

for branch in ["a", "b"]:
    profile_test_meetings[
        f"candidate_{branch}_start_minutes"
    ] = (
        profile_test_meetings[
            f"candidate_{branch}_local_start"
        ].dt.hour * 60
        + profile_test_meetings[
            f"candidate_{branch}_local_start"
        ].dt.minute
    )

    profile_test_meetings[
        f"candidate_{branch}_end_minutes"
    ] = (
        profile_test_meetings[
            f"candidate_{branch}_local_end"
        ].dt.hour * 60
        + profile_test_meetings[
            f"candidate_{branch}_local_end"
        ].dt.minute
    )


course_profile_sensitivity = []

for margin_minutes in [60, 90, 120, 180]:
    earliest_start_allowed = (
        profile_test_meetings["earliest_observed_start"]
        - margin_minutes
    ).clip(lower=0)

    latest_start_allowed = (
        profile_test_meetings["latest_observed_start"]
        + margin_minutes
    ).clip(upper=1439)

    earliest_end_allowed = (
        profile_test_meetings["earliest_observed_end"]
        - margin_minutes
    ).clip(lower=0)

    latest_end_allowed = (
        profile_test_meetings["latest_observed_end"]
        + margin_minutes
    ).clip(upper=1439)

    candidate_a_compatible = (
        profile_test_meetings[
            "candidate_a_start_minutes"
        ].between(
            earliest_start_allowed,
            latest_start_allowed,
        )
        & profile_test_meetings[
            "candidate_a_end_minutes"
        ].between(
            earliest_end_allowed,
            latest_end_allowed,
        )
    )

    candidate_b_compatible = (
        profile_test_meetings[
            "candidate_b_start_minutes"
        ].between(
            earliest_start_allowed,
            latest_start_allowed,
        )
        & profile_test_meetings[
            "candidate_b_end_minutes"
        ].between(
            earliest_end_allowed,
            latest_end_allowed,
        )
    )

    candidate_a_only = (
        candidate_a_compatible
        & ~candidate_b_compatible
    )

    candidate_b_only = (
        candidate_b_compatible
        & ~candidate_a_compatible
    )

    both_compatible = (
        candidate_a_compatible
        & candidate_b_compatible
    )

    neither_compatible = (
        ~candidate_a_compatible
        & ~candidate_b_compatible
    )

    course_profile_sensitivity.append(
        {
            "margin_minutes": margin_minutes,
            "candidate_a_only": int(candidate_a_only.sum()),
            "candidate_b_only": int(candidate_b_only.sum()),
            "both_compatible": int(both_compatible.sum()),
            "neither_compatible": int(neither_compatible.sum()),
            "resolved_by_profile": int(
                (candidate_a_only | candidate_b_only).sum()
            ),
            "resolved_percent_of_tested": round(
                100
                * (candidate_a_only | candidate_b_only).mean(),
                2,
            ),
        }
    )

pd.DataFrame(course_profile_sensitivity)

,margin_minutes,candidate_a_only,candidate_b_only,both_compatible,neither_compatible,resolved_by_profile,resolved_percent_of_tested
0,60,1001,5472,0,1132,6473,85.12
1,90,1042,5655,0,908,6697,88.06
2,120,1097,5775,0,733,6872,90.36
3,180,1126,6039,0,440,7165,94.21


In [80]:
# Measure whether course-profile branch decisions remain stable across all
# tested margins.
#
# A meeting is called stable only when the same single compatible branch is
# selected at 60, 90, 120, and 180 minutes.

profile_margin_decisions = profile_test_meetings[
    [
        "date",
        "course",
        "candidate_course_label",
        "candidate_jurisdiction",
        "race_count",
        "observed_meetings",
    ]
].copy()

for margin_minutes in [60, 90, 120, 180]:
    earliest_start_allowed = (
        profile_test_meetings["earliest_observed_start"]
        - margin_minutes
    ).clip(lower=0)

    latest_start_allowed = (
        profile_test_meetings["latest_observed_start"]
        + margin_minutes
    ).clip(upper=1439)

    earliest_end_allowed = (
        profile_test_meetings["earliest_observed_end"]
        - margin_minutes
    ).clip(lower=0)

    latest_end_allowed = (
        profile_test_meetings["latest_observed_end"]
        + margin_minutes
    ).clip(upper=1439)

    candidate_a_compatible = (
        profile_test_meetings[
            "candidate_a_start_minutes"
        ].between(
            earliest_start_allowed,
            latest_start_allowed,
        )
        & profile_test_meetings[
            "candidate_a_end_minutes"
        ].between(
            earliest_end_allowed,
            latest_end_allowed,
        )
    )

    candidate_b_compatible = (
        profile_test_meetings[
            "candidate_b_start_minutes"
        ].between(
            earliest_start_allowed,
            latest_start_allowed,
        )
        & profile_test_meetings[
            "candidate_b_end_minutes"
        ].between(
            earliest_end_allowed,
            latest_end_allowed,
        )
    )

    profile_margin_decisions[
        f"decision_{margin_minutes}"
    ] = np.select(
        [
            candidate_a_compatible & ~candidate_b_compatible,
            candidate_b_compatible & ~candidate_a_compatible,
            candidate_a_compatible & candidate_b_compatible,
        ],
        [
            "candidate_a",
            "candidate_b",
            "both",
        ],
        default="neither",
    )

decision_columns = [
    "decision_60",
    "decision_90",
    "decision_120",
    "decision_180",
]

profile_margin_decisions[
    "stable_profile_decision"
] = profile_margin_decisions[decision_columns].apply(
    lambda row: (
        row.iloc[0]
        if row.nunique() == 1
        and row.iloc[0] in {"candidate_a", "candidate_b"}
        else "not_stable"
    ),
    axis=1,
)

stable_profile_summary = (
    profile_margin_decisions
    .groupby(
        "stable_profile_decision",
        as_index=False,
    )
    .agg(
        meetings=("race_count", "size"),
        races=("race_count", "sum"),
    )
    .sort_values(
        "meetings",
        ascending=False,
    )
    .reset_index(drop=True)
)

stable_profile_summary

,stable_profile_decision,meetings,races
0,candidate_b,5472,39855
1,not_stable,1132,7471
2,candidate_a,1001,7387


In [81]:
# Profile the decision trajectories for meetings whose course-profile result
# changes across the tested margins.

unstable_profile_decisions = (
    profile_margin_decisions.loc[
        profile_margin_decisions[
            "stable_profile_decision"
        ].eq("not_stable")
    ]
    .copy()
)

unstable_decision_patterns = (
    unstable_profile_decisions
    .groupby(
        [
            "decision_60",
            "decision_90",
            "decision_120",
            "decision_180",
        ],
        as_index=False,
    )
    .agg(
        meetings=("race_count", "size"),
        races=("race_count", "sum"),
        distinct_courses=("candidate_course_label", "nunique"),
    )
    .sort_values(
        ["meetings", "races"],
        ascending=False,
    )
    .reset_index(drop=True)
)

unstable_decision_patterns

,decision_60,decision_90,decision_120,decision_180,meetings,races,distinct_courses
0,neither,neither,neither,neither,440,2857,33
1,neither,neither,neither,candidate_b,264,1828,32
2,neither,candidate_b,candidate_b,candidate_b,183,1189,31
3,neither,neither,candidate_b,candidate_b,120,769,32
4,neither,neither,candidate_a,candidate_a,55,467,8
5,neither,candidate_a,candidate_a,candidate_a,41,107,8
6,neither,neither,neither,candidate_a,29,254,4


In [82]:
# Inspect meetings where neither candidate fits the course profile even with
# a three-hour margin.

persistent_neither = (
    unstable_profile_decisions.loc[
        unstable_profile_decisions[
            [
                "decision_60",
                "decision_90",
                "decision_120",
                "decision_180",
            ]
        ].eq("neither").all(axis=1)
    ]
    .merge(
        profile_test_meetings[
            [
                "date",
                "course",
                "candidate_course_label",
                "candidate_jurisdiction",
                "race_count",
                "observed_meetings",
                "candidate_a_local_start",
                "candidate_a_local_end",
                "candidate_b_local_start",
                "candidate_b_local_end",
                "earliest_observed_start",
                "latest_observed_start",
                "earliest_observed_end",
                "latest_observed_end",
            ]
        ],
        on=[
            "date",
            "course",
            "candidate_course_label",
            "candidate_jurisdiction",
            "race_count",
            "observed_meetings",
        ],
        how="left",
        validate="one_to_one",
    )
)

persistent_neither_summary = (
    persistent_neither
    .groupby(
        [
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        as_index=False,
    )
    .agg(
        meetings=("race_count", "size"),
        races=("race_count", "sum"),
        post_boundary_profile_meetings=(
            "observed_meetings",
            "first",
        ),
    )
    .sort_values(
        ["meetings", "races"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    pd.DataFrame(
        {
            "measure": [
                "persistent-neither meetings",
                "persistent-neither races",
                "distinct courses",
                "distinct jurisdictions",
            ],
            "value": [
                len(persistent_neither),
                persistent_neither["race_count"].sum(),
                persistent_neither[
                    "candidate_course_label"
                ].nunique(),
                persistent_neither[
                    "candidate_jurisdiction"
                ].nunique(),
            ],
        }
    )
)

persistent_neither_summary.head(30)

,measure,value
0,persistent-neither meetings,440
1,persistent-neither races,2857
2,distinct courses,33
3,distinct jurisdictions,8


,candidate_jurisdiction,candidate_course_label,meetings,races,post_boundary_profile_meetings
0,Great Britain,Haydock,56,342,15.0
1,Great Britain,Sandown,51,303,8.0
2,Great Britain,Newbury,36,239,18.0
3,Great Britain,Carlisle,31,215,11.0
4,Great Britain,Ayr,29,193,14.0
5,Great Britain,Stratford,28,189,6.0
6,Great Britain,Beverley,25,159,5.0
7,Ireland,Down Royal,22,155,7.0
8,Great Britain,Perth,19,129,6.0
9,Ireland,Fairyhouse,16,116,13.0


In [83]:
# Profile the persistent-neither meetings by month and candidate local times.
#
# This tests whether the apparent profile failures are concentrated in
# spring and summer periods missing from the short post-boundary sample.

persistent_neither["meeting_month"] = pd.to_datetime(
    persistent_neither["date"]
).dt.month

persistent_neither_month_summary = (
    persistent_neither
    .groupby("meeting_month", as_index=False)
    .agg(
        meetings=("race_count", "size"),
        races=("race_count", "sum"),
        distinct_courses=("candidate_course_label", "nunique"),
    )
)

persistent_neither_month_summary["month_name"] = (
    pd.to_datetime(
        persistent_neither_month_summary["meeting_month"],
        format="%m",
    ).dt.month_name()
)

display(
    persistent_neither_month_summary[
        [
            "meeting_month",
            "month_name",
            "meetings",
            "races",
            "distinct_courses",
        ]
    ]
)

persistent_neither[
    [
        "date",
        "course",
        "candidate_jurisdiction",
        "candidate_a_local_start",
        "candidate_a_local_end",
        "candidate_b_local_start",
        "candidate_b_local_end",
        "observed_meetings",
    ]
].sort_values(
    ["date", "course"]
).head(30)

,meeting_month,month_name,meetings,races,distinct_courses
0,1,January,3,28,2
1,3,March,1,8,1
2,4,April,8,50,5
3,5,May,115,762,21
4,6,June,110,744,16
5,7,July,124,803,14
6,8,August,61,388,12
7,9,September,6,19,5
8,10,October,1,1,1
9,11,November,2,21,2


,date,course,candidate_jurisdiction,candidate_a_local_start,candidate_a_local_end,candidate_b_local_start,candidate_b_local_end,observed_meetings
0,2015-04-29,Cheltenham,Great Britain,2015-04-29 04:50:00,2015-04-29 08:15:00,2015-04-29 16:50:00,2015-04-29 20:15:00,13.0
1,2015-05-01,Bangor-on-Dee,Great Britain,2015-05-01 05:20:00,2015-05-01 08:05:00,2015-05-01 17:20:00,2015-05-01 20:05:00,9.0
2,2015-05-05,Catterick,Great Britain,2015-05-05 06:00:00,2015-05-05 08:30:00,2015-05-05 18:00:00,2015-05-05 20:30:00,15.0
3,2015-05-07,Carlisle,Great Britain,2015-05-07 05:40:00,2015-05-07 08:40:00,2015-05-07 17:40:00,2015-05-07 20:40:00,11.0
4,2015-05-07,Wincanton,Great Britain,2015-05-07 05:50:00,2015-05-07 08:20:00,2015-05-07 17:50:00,2015-05-07 20:20:00,13.0
5,2015-05-08,Ascot,Great Britain,2015-05-08 05:35:00,2015-05-08 08:15:00,2015-05-08 17:35:00,2015-05-08 20:15:00,12.0
6,2015-05-13,Perth,Great Britain,2015-05-13 05:55:00,2015-05-13 08:55:00,2015-05-13 17:55:00,2015-05-13 20:55:00,6.0
7,2015-05-14,Newmarket,Great Britain,2015-05-14 05:15:00,2015-05-14 08:30:00,2015-05-14 17:15:00,2015-05-14 20:30:00,11.0
8,2015-05-20,Southwell,Great Britain,2015-05-20 05:50:00,2015-05-20 08:50:00,2015-05-20 17:50:00,2015-05-20 20:50:00,10.0
9,2015-05-21,Sandown,Great Britain,2015-05-21 05:55:00,2015-05-21 08:35:00,2015-05-21 17:55:00,2015-05-21 20:35:00,8.0


### Seasonal limitation of post-boundary course profiles

Course-specific post-boundary scheduling profiles provide strong evidence, but
the current post-boundary population begins on 15 October 2025 and therefore
does not yet contain a complete summer racing season.

This limitation is visible in the 440 historical meetings for which neither
12-hour branch fits the observed course profile even after a three-hour margin:

* 410 meetings, or 93.2%, occur from May through August;
* the affected courses are dominated by British and Irish turf venues;
* detailed inspection shows repeated morning-versus-evening alternatives such
  as 06:15–08:50 against 18:15–20:50 course-local time.

These are consistent with historical summer evening meetings that are absent
from the current post-boundary observation window. A failure to match the
post-boundary profile must therefore not be treated as evidence that both
branches are implausible.

Stable single-branch matches remain useful evidence. Profile mismatches,
especially during spring and summer, remain unresolved until the reference
period includes comparable seasonal racing or another independent anchor is
available.

In [85]:
# Build a reproducible manual-validation sample that:
# - covers the principal evidence categories;
# - avoids duplicate meetings across categories;
# - spreads selections across the pre-boundary date range.

def select_spread_sample(
    frame,
    category,
    count,
    used_meetings,
):
    candidates = (
        frame
        .drop_duplicates(["date", "course"])
        .copy()
    )

    candidates["meeting_key"] = list(
        zip(candidates["date"], candidates["course"])
    )

    candidates = candidates.loc[
        ~candidates["meeting_key"].isin(used_meetings)
    ].sort_values(["date", "course"])

    if len(candidates) <= count:
        selected = candidates.copy()
    else:
        selected_positions = np.linspace(
            0,
            len(candidates) - 1,
            count,
        ).round().astype(int)

        selected = candidates.iloc[
            np.unique(selected_positions)
        ].copy()

    selected["validation_category"] = category

    used_meetings.update(selected["meeting_key"])

    return selected


used_validation_meetings = set()
validation_sample_parts = []

validation_sample_parts.append(
    select_spread_sample(
        stable_profile_details.loc[
            stable_profile_details[
                "stable_profile_decision"
            ].eq("candidate_a")
        ],
        "stable_profile_candidate_a",
        6,
        used_validation_meetings,
    )
)

validation_sample_parts.append(
    select_spread_sample(
        stable_profile_details.loc[
            stable_profile_details[
                "stable_profile_decision"
            ].eq("candidate_b")
        ],
        "stable_profile_candidate_b",
        6,
        used_validation_meetings,
    )
)

validation_sample_parts.append(
    select_spread_sample(
        persistent_neither.loc[
            persistent_neither[
                "meeting_month"
            ].between(5, 8)
        ],
        "summer_profile_mismatch",
        6,
        used_validation_meetings,
    )
)

validation_sample_parts.append(
    select_spread_sample(
        local_meeting_candidate_summary.loc[
            local_meeting_candidate_summary[
                "preliminary_branch_result"
            ].eq("dst_edge_requires_review")
        ],
        "dst_edge",
        6,
        used_validation_meetings,
    )
)

validation_sample_parts.append(
    select_spread_sample(
        local_meeting_candidate_summary.loc[
            local_meeting_candidate_summary[
                "race_count"
            ].eq(1)
        ],
        "single_race_meeting",
        8,
        used_validation_meetings,
    )
)

validation_sample_parts.append(
    select_spread_sample(
        local_meeting_candidate_summary.loc[
            local_meeting_candidate_summary[
                "candidate_jurisdiction"
            ].isin(
                [
                    "Hong Kong",
                    "Japan",
                    "United Arab Emirates",
                    "United States",
                    "Argentina",
                    "Chile",
                ]
            )
        ],
        "international_sanity_check",
        10,
        used_validation_meetings,
    )
)

manual_validation_sample = (
    pd.concat(
        validation_sample_parts,
        ignore_index=True,
    )
    [
        [
            "validation_category",
            "date",
            "course",
            "candidate_jurisdiction",
            "race_count",
            "candidate_a_local_start",
            "candidate_a_local_end",
            "candidate_b_local_start",
            "candidate_b_local_end",
        ]
    ]
    .sort_values(
        ["validation_category", "date", "course"]
    )
    .reset_index(drop=True)
)

display(
    pd.DataFrame(
        {
            "measure": [
                "sample meetings",
                "duplicate date-course meetings",
                "earliest sample date",
                "latest sample date",
                "distinct jurisdictions",
            ],
            "value": [
                len(manual_validation_sample),
                manual_validation_sample.duplicated(
                    ["date", "course"]
                ).sum(),
                manual_validation_sample["date"].min(),
                manual_validation_sample["date"].max(),
                manual_validation_sample[
                    "candidate_jurisdiction"
                ].nunique(),
            ],
        }
    )
)

manual_validation_sample

,measure,value
0,sample meetings,42
1,duplicate date-course meetings,0
2,earliest sample date,2015-01-01
3,latest sample date,2025-10-14
4,distinct jurisdictions,9


,validation_category,date,course,candidate_jurisdiction,race_count,candidate_a_local_start,candidate_a_local_end,candidate_b_local_start,candidate_b_local_end
0,dst_edge,2015-03-29,Auteuil (FR),France,8,2015-03-29 03:08:00,2015-03-29 05:55:00,2015-03-29 14:00:00,2015-03-29 17:55:00
1,dst_edge,2017-03-26,Auteuil (FR),France,8,2017-03-26 03:15:00,2017-03-26 06:20:00,2017-03-26 14:05:00,2017-03-26 18:20:00
2,dst_edge,2019-10-27,Wexford (IRE),Ireland,7,2019-10-27 00:50:00,2019-10-27 04:20:00,2019-10-27 12:50:00,2019-10-27 16:20:00
3,dst_edge,2022-03-27,Ascot,Great Britain,7,2022-03-27 02:10:00,2022-03-27 04:30:00,2022-03-27 13:00:00,2022-03-27 16:30:00
4,dst_edge,2023-10-29,Aintree,Great Britain,7,2023-10-29 00:50:00,2023-10-29 04:20:00,2023-10-29 12:50:00,2023-10-29 16:20:00
5,dst_edge,2025-03-30,Downpatrick (IRE),Ireland,7,2025-03-30 02:20:00,2025-03-30 05:15:00,2025-03-30 13:45:00,2025-03-30 17:15:00
6,international_sanity_check,2015-01-01,Laurel Park (USA),United States,1,2015-01-01 02:52:00,2015-01-01 02:52:00,2015-01-01 14:52:00,2015-01-01 14:52:00
7,international_sanity_check,2016-01-09,Aqueduct (USA),United States,1,2016-01-09 01:20:00,2016-01-09 01:20:00,2016-01-09 13:20:00,2016-01-09 13:20:00
8,international_sanity_check,2017-03-19,Nakayama (JPN),Japan,1,2017-03-19 15:45:00,2017-03-19 15:45:00,2017-03-20 03:45:00,2017-03-20 03:45:00
9,international_sanity_check,2018-04-28,Santa Anita (USA),United States,2,2018-04-28 02:35:00,2018-04-28 03:07:00,2018-04-28 14:35:00,2018-04-28 15:07:00


In [86]:
# Select a compact high-value subset for external racecard validation.

priority_validation_keys = [
    ("2015-04-12", "Oaklawn Park (USA)"),
    ("2015-01-04", "Abu Dhabi (UAE)"),
    ("2015-03-29", "Auteuil (FR)"),
    ("2019-10-27", "Wexford (IRE)"),
    ("2023-10-29", "Aintree"),
    ("2016-07-07", "Newbury"),
    ("2021-05-27", "Carlisle"),
    ("2023-07-26", "Sandown"),
    ("2015-01-02", "Valparaiso Sporting Club (CHI)"),
    ("2020-08-15", "San Sebastian (SPA)"),
    ("2024-06-26", "Happy Valley (HK)"),
    ("2017-03-19", "Nakayama (JPN)"),
    ("2018-04-28", "Santa Anita (USA)"),
    ("2022-08-07", "Del Mar (USA)"),
]

priority_validation_sample = (
    manual_validation_sample
    .assign(
        meeting_key=lambda frame: list(
            zip(frame["date"], frame["course"])
        )
    )
    .loc[
        lambda frame: frame["meeting_key"].isin(
            priority_validation_keys
        )
    ]
    .drop(columns="meeting_key")
    .copy()
)

priority_validation_sample[
    "expected_branch_from_current_evidence"
] = priority_validation_sample[
    "validation_category"
].map(
    {
        "stable_profile_candidate_a": "candidate_a",
        "stable_profile_candidate_b": "candidate_b",
        "summer_profile_mismatch": "candidate_b_expected",
        "dst_edge": "candidate_b_expected",
        "single_race_meeting": "manual_check",
        "international_sanity_check": "manual_check",
    }
)

priority_validation_sample.sort_values(
    ["validation_category", "date", "course"]
).reset_index(drop=True)

,validation_category,date,course,candidate_jurisdiction,race_count,candidate_a_local_start,candidate_a_local_end,candidate_b_local_start,candidate_b_local_end,expected_branch_from_current_evidence
0,dst_edge,2015-03-29,Auteuil (FR),France,8,2015-03-29 03:08:00,2015-03-29 05:55:00,2015-03-29 14:00:00,2015-03-29 17:55:00,candidate_b_expected
1,dst_edge,2019-10-27,Wexford (IRE),Ireland,7,2019-10-27 00:50:00,2019-10-27 04:20:00,2019-10-27 12:50:00,2019-10-27 16:20:00,candidate_b_expected
2,dst_edge,2023-10-29,Aintree,Great Britain,7,2023-10-29 00:50:00,2023-10-29 04:20:00,2023-10-29 12:50:00,2023-10-29 16:20:00,candidate_b_expected
3,international_sanity_check,2017-03-19,Nakayama (JPN),Japan,1,2017-03-19 15:45:00,2017-03-19 15:45:00,2017-03-20 03:45:00,2017-03-20 03:45:00,manual_check
4,international_sanity_check,2018-04-28,Santa Anita (USA),United States,2,2018-04-28 02:35:00,2018-04-28 03:07:00,2018-04-28 14:35:00,2018-04-28 15:07:00,manual_check
5,international_sanity_check,2024-06-26,Happy Valley (HK),Hong Kong,9,2024-06-26 18:40:00,2024-06-26 22:50:00,2024-06-27 06:40:00,2024-06-27 10:50:00,manual_check
6,single_race_meeting,2020-08-15,San Sebastian (SPA),Spain,1,2020-08-15 07:35:00,2020-08-15 07:35:00,2020-08-15 19:35:00,2020-08-15 19:35:00,manual_check
7,stable_profile_candidate_a,2022-08-07,Del Mar (USA),United States,1,2022-08-06 18:04:00,2022-08-06 18:04:00,2022-08-07 06:04:00,2022-08-07 06:04:00,candidate_a
8,summer_profile_mismatch,2016-07-07,Newbury,Great Britain,7,2016-07-07 06:00:00,2016-07-07 09:10:00,2016-07-07 18:00:00,2016-07-07 21:10:00,candidate_b_expected
9,summer_profile_mismatch,2021-05-27,Carlisle,Great Britain,7,2021-05-27 05:30:00,2021-05-27 08:50:00,2021-05-27 17:30:00,2021-05-27 20:50:00,candidate_b_expected


### External validation pilot

A focused external validation pilot tested reconstructed branches against
historical racecards and official results.

The checked meetings included:

* DST-transition meetings in France, Ireland and Great Britain;
* summer evening meetings in Great Britain;
* Japanese afternoon racing;
* Hong Kong evening racing;
* United States racing displayed through the UK-facing source clock.

Eight checked meetings across six jurisdictions matched the reconstructed
course-local branch:

* Auteuil, 29 March 2015 — candidate B, 14:00–17:55;
* Wexford, 27 October 2019 — candidate B, 12:50–16:20;
* Aintree, 29 October 2023 — candidate B, 12:50–16:20;
* Nakayama, 19 March 2017 — candidate A, 15:45;
* Happy Valley, 26 June 2024 — candidate A, 18:40–22:50;
* Newbury, 7 July 2016 — candidate B, 18:00–21:10;
* Carlisle, 27 May 2021 — candidate B, 17:30–20:50;
* Santa Anita, 28 April 2018 — candidate B, 14:35–15:07.

The pilot therefore produced eight correct branch selections from eight checks.
It supports the UK-facing clock interpretation, the historical timezone
conversion, the course-local feasibility rule, and the treatment of summer
evening racing as a seasonal limitation of the current post-boundary profiles.

In [88]:
# Combine the validated branch-selection rules into one meeting-level decision.
#
# Rule hierarchy:
# 1. Reject a branch when the whole meeting is course-local overnight.
# 2. For meetings still unresolved, accept a course-profile decision only when
#    the same branch is selected at every tested margin.
# 3. Otherwise retain both candidates.

meeting_branch_decisions = (
    local_meeting_candidate_summary.copy()
)

meeting_branch_decisions[
    "selected_branch"
] = pd.NA

meeting_branch_decisions[
    "decision_method"
] = "unresolved"

meeting_branch_decisions[
    "decision_confidence"
] = "unresolved"


# Hard course-local overnight constraint
candidate_a_only_mask = meeting_branch_decisions[
    "preliminary_branch_result"
].eq("candidate_a_only_feasible")

candidate_b_only_mask = meeting_branch_decisions[
    "preliminary_branch_result"
].eq("candidate_b_only_feasible")

meeting_branch_decisions.loc[
    candidate_a_only_mask,
    [
        "selected_branch",
        "decision_method",
        "decision_confidence",
    ],
] = [
    "candidate_a",
    "course_local_overnight_rejection",
    "high",
]

meeting_branch_decisions.loc[
    candidate_b_only_mask,
    [
        "selected_branch",
        "decision_method",
        "decision_confidence",
    ],
] = [
    "candidate_b",
    "course_local_overnight_rejection",
    "high",
]


# Stable post-boundary course-profile evidence
stable_profile_lookup = (
    profile_margin_decisions.loc[
        profile_margin_decisions[
            "stable_profile_decision"
        ].isin(["candidate_a", "candidate_b"]),
        [
            "date",
            "course",
            "stable_profile_decision",
        ],
    ]
    .rename(
        columns={
            "stable_profile_decision": "profile_selected_branch"
        }
    )
)

meeting_branch_decisions = (
    meeting_branch_decisions
    .merge(
        stable_profile_lookup,
        on=["date", "course"],
        how="left",
        validate="one_to_one",
    )
)

profile_decision_mask = (
    meeting_branch_decisions["selected_branch"].isna()
    & meeting_branch_decisions[
        "profile_selected_branch"
    ].notna()
)

meeting_branch_decisions.loc[
    profile_decision_mask,
    "selected_branch",
] = meeting_branch_decisions.loc[
    profile_decision_mask,
    "profile_selected_branch",
]

meeting_branch_decisions.loc[
    profile_decision_mask,
    "decision_method",
] = "stable_post_boundary_course_profile"

meeting_branch_decisions.loc[
    profile_decision_mask,
    "decision_confidence",
] = "supported"


decision_summary = (
    meeting_branch_decisions
    .groupby(
        [
            "decision_method",
            "selected_branch",
            "decision_confidence",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        meetings=("race_count", "size"),
        races=("race_count", "sum"),
    )
    .sort_values(
        ["meetings", "races"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(decision_summary)

pd.DataFrame(
    {
        "measure": [
            "pre-boundary meetings",
            "resolved meetings",
            "unresolved meetings",
            "resolved meeting percent",
            "pre-boundary races",
            "resolved races",
            "unresolved races",
            "resolved race percent",
        ],
        "value": [
            len(meeting_branch_decisions),
            meeting_branch_decisions[
                "selected_branch"
            ].notna().sum(),
            meeting_branch_decisions[
                "selected_branch"
            ].isna().sum(),
            round(
                100
                * meeting_branch_decisions[
                    "selected_branch"
                ].notna().mean(),
                2,
            ),
            meeting_branch_decisions["race_count"].sum(),
            meeting_branch_decisions.loc[
                meeting_branch_decisions[
                    "selected_branch"
                ].notna(),
                "race_count",
            ].sum(),
            meeting_branch_decisions.loc[
                meeting_branch_decisions[
                    "selected_branch"
                ].isna(),
                "race_count",
            ].sum(),
            round(
                100
                * meeting_branch_decisions.loc[
                    meeting_branch_decisions[
                        "selected_branch"
                    ].notna(),
                    "race_count",
                ].sum()
                / meeting_branch_decisions[
                    "race_count"
                ].sum(),
                2,
            ),
        ],
    }
)

,decision_method,selected_branch,decision_confidence,meetings,races
0,course_local_overnight_rejection,candidate_b,high,17362,98345
1,stable_post_boundary_course_profile,candidate_b,supported,5472,39855
2,course_local_overnight_rejection,candidate_a,high,3943,13526
3,unresolved,NaN,unresolved,3663,19578
4,stable_post_boundary_course_profile,candidate_a,supported,1001,7387


,measure,value
0,pre-boundary meetings,31441.00
1,resolved meetings,27778.00
2,unresolved meetings,3663.00
3,resolved meeting percent,88.35
4,pre-boundary races,178691.00
5,resolved races,159113.00
6,unresolved races,19578.00
7,resolved race percent,89.04


### Pre-boundary branch reconstruction result

The validated meeting-level reconstruction hierarchy is:

1. reject a branch when the entire meeting falls in a clearly implausible
   course-local overnight window;
2. for meetings still unresolved, accept a course-profile decision only when
   the same branch is selected across every tested profile margin from 60 to
   180 minutes;
3. retain both candidates when neither rule produces a stable decision.

Across 31,441 pre-boundary meetings:

* 17,362 meetings selected candidate B through course-local overnight
  rejection;
* 3,943 meetings selected candidate A through course-local overnight
  rejection;
* 5,472 meetings selected candidate B through stable post-boundary
  course-profile evidence;
* 1,001 meetings selected candidate A through stable post-boundary
  course-profile evidence;
* 3,663 meetings remain unresolved.

The combined rules therefore resolve 27,778 meetings, or 88.35% of the
pre-boundary population.

At race level:

* 159,113 of 178,691 races receive a reconstructed branch;
* 19,578 races remain unresolved;
* resolved race coverage is 89.04%.

The unresolved population is retained explicitly rather than assigning a
12-hour branch from weaker assumptions or customary scheduling alone.

In [89]:
# Attach meeting-level branch decisions to each pre-boundary race.
#
# Both candidate branches remain preserved.
# Canonical reconstructed timestamps are populated only for resolved races.

pre_boundary_reconstructed_races = (
    pre_boundary_race_candidates_with_locations
    .merge(
        meeting_branch_decisions[
            [
                "date",
                "course",
                "selected_branch",
                "decision_method",
                "decision_confidence",
            ]
        ],
        on=["date", "course"],
        how="left",
        validate="many_to_one",
    )
)

pre_boundary_reconstructed_races[
    "advertised_start_uk"
] = pd.NaT

pre_boundary_reconstructed_races[
    "advertised_start_utc"
] = pd.NaT

pre_boundary_reconstructed_races[
    "advertised_start_course_local"
] = pd.NaT


candidate_a_mask = pre_boundary_reconstructed_races[
    "selected_branch"
].eq("candidate_a")

candidate_b_mask = pre_boundary_reconstructed_races[
    "selected_branch"
].eq("candidate_b")


pre_boundary_reconstructed_races.loc[
    candidate_a_mask,
    "advertised_start_uk",
] = pre_boundary_reconstructed_races.loc[
    candidate_a_mask,
    "candidate_a_uk_aware",
]

pre_boundary_reconstructed_races.loc[
    candidate_a_mask,
    "advertised_start_utc",
] = pre_boundary_reconstructed_races.loc[
    candidate_a_mask,
    "candidate_a_utc",
]

pre_boundary_reconstructed_races.loc[
    candidate_a_mask,
    "advertised_start_course_local",
] = pre_boundary_reconstructed_races.loc[
    candidate_a_mask,
    "candidate_a_course_local",
]


pre_boundary_reconstructed_races.loc[
    candidate_b_mask,
    "advertised_start_uk",
] = pre_boundary_reconstructed_races.loc[
    candidate_b_mask,
    "candidate_b_uk_aware",
]

pre_boundary_reconstructed_races.loc[
    candidate_b_mask,
    "advertised_start_utc",
] = pre_boundary_reconstructed_races.loc[
    candidate_b_mask,
    "candidate_b_utc",
]

pre_boundary_reconstructed_races.loc[
    candidate_b_mask,
    "advertised_start_course_local",
] = pre_boundary_reconstructed_races.loc[
    candidate_b_mask,
    "candidate_b_course_local",
]


pd.DataFrame(
    {
        "measure": [
            "pre-boundary races",
            "resolved races",
            "unresolved races",
            "selected UK timestamps",
            "selected UTC timestamps",
            "selected course-local timestamps",
        ],
        "value": [
            len(pre_boundary_reconstructed_races),
            pre_boundary_reconstructed_races[
                "selected_branch"
            ].notna().sum(),
            pre_boundary_reconstructed_races[
                "selected_branch"
            ].isna().sum(),
            pre_boundary_reconstructed_races[
                "advertised_start_uk"
            ].notna().sum(),
            pre_boundary_reconstructed_races[
                "advertised_start_utc"
            ].notna().sum(),
            pre_boundary_reconstructed_races[
                "advertised_start_course_local"
            ].notna().sum(),
        ],
    }
)

TypeError: Invalid value '<DatetimeArray>
['2015-01-01 06:50:00+00:00', '2015-01-01 08:50:00+00:00',
 '2015-01-01 01:35:00+00:00', '2015-01-01 02:55:00+00:00',
 '2015-01-01 03:35:00+00:00', '2015-01-01 04:15:00+00:00',
 '2015-01-01 04:55:00+00:00', '2015-01-01 05:35:00+00:00',
 '2015-01-01 05:20:00+00:00', '2015-01-04 06:45:00+00:00',
 ...
 '2025-10-12 06:00:00+01:00', '2025-10-12 06:30:00+01:00',
 '2025-10-12 07:00:00+01:00', '2025-10-12 07:30:00+01:00',
 '2025-10-12 08:00:00+01:00', '2025-10-12 08:35:00+01:00',
 '2025-10-12 09:05:00+01:00', '2025-10-12 09:35:00+01:00',
 '2025-10-12 07:45:00+01:00', '2025-10-13 07:45:00+01:00']
Length: 20913, dtype: datetime64[us, Europe/London]' for dtype 'datetime64[ns]'

In [90]:
# Attach meeting-level decisions to every pre-boundary race.
#
# Both candidate branches remain preserved.
# Canonical timestamps are populated only where a branch is resolved.

pre_boundary_reconstructed_races = (
    pre_boundary_race_candidates_with_locations
    .merge(
        meeting_branch_decisions[
            [
                "date",
                "course",
                "selected_branch",
                "decision_method",
                "decision_confidence",
            ]
        ],
        on=["date", "course"],
        how="left",
        validate="many_to_one",
    )
)

candidate_a_mask = pre_boundary_reconstructed_races[
    "selected_branch"
].eq("candidate_a")

candidate_b_mask = pre_boundary_reconstructed_races[
    "selected_branch"
].eq("candidate_b")


# London and UTC columns each have one consistent timezone, so pandas can
# retain proper timezone-aware datetime dtypes.

pre_boundary_reconstructed_races[
    "advertised_start_uk"
] = (
    pre_boundary_reconstructed_races[
        "candidate_a_uk_aware"
    ].where(
        candidate_a_mask,
        pre_boundary_reconstructed_races[
            "candidate_b_uk_aware"
        ].where(candidate_b_mask),
    )
)

pre_boundary_reconstructed_races[
    "advertised_start_utc"
] = (
    pre_boundary_reconstructed_races[
        "candidate_a_utc"
    ].where(
        candidate_a_mask,
        pre_boundary_reconstructed_races[
            "candidate_b_utc"
        ].where(candidate_b_mask),
    )
)


# Course-local timestamps use many different IANA timezones, so this column
# must use object dtype rather than one pandas datetime64 timezone dtype.

selected_course_local = pd.Series(
    [pd.NaT] * len(pre_boundary_reconstructed_races),
    index=pre_boundary_reconstructed_races.index,
    dtype="object",
)

selected_course_local.loc[candidate_a_mask] = (
    pre_boundary_reconstructed_races.loc[
        candidate_a_mask,
        "candidate_a_course_local",
    ].to_numpy(dtype="object")
)

selected_course_local.loc[candidate_b_mask] = (
    pre_boundary_reconstructed_races.loc[
        candidate_b_mask,
        "candidate_b_course_local",
    ].to_numpy(dtype="object")
)

pre_boundary_reconstructed_races[
    "advertised_start_course_local"
] = selected_course_local


pd.DataFrame(
    {
        "measure": [
            "pre-boundary races",
            "resolved races",
            "unresolved races",
            "selected UK timestamps",
            "selected UTC timestamps",
            "selected course-local timestamps",
        ],
        "value": [
            len(pre_boundary_reconstructed_races),
            pre_boundary_reconstructed_races[
                "selected_branch"
            ].notna().sum(),
            pre_boundary_reconstructed_races[
                "selected_branch"
            ].isna().sum(),
            pre_boundary_reconstructed_races[
                "advertised_start_uk"
            ].notna().sum(),
            pre_boundary_reconstructed_races[
                "advertised_start_utc"
            ].notna().sum(),
            pre_boundary_reconstructed_races[
                "advertised_start_course_local"
            ].notna().sum(),
        ],
    }
)

,measure,value
0,pre-boundary races,178691
1,resolved races,159113
2,unresolved races,19578
3,selected UK timestamps,159113
4,selected UTC timestamps,159113
5,selected course-local timestamps,159113


In [91]:
# Give post-boundary races the same decision and canonical timestamp fields.
#
# These rows need no branch reconstruction because `off` already uses an
# explicit 24-hour UK-facing clock.

post_boundary_reconstructed_races = (
    post_boundary_races_with_locations.copy()
)

post_boundary_reconstructed_races[
    "selected_branch"
] = "explicit_24h"

post_boundary_reconstructed_races[
    "decision_method"
] = "explicit_post_boundary_time"

post_boundary_reconstructed_races[
    "decision_confidence"
] = "source_explicit"

pd.DataFrame(
    {
        "measure": [
            "post-boundary races",
            "selected UK timestamps",
            "selected UTC timestamps",
            "selected course-local timestamps",
            "distinct governed course identities",
        ],
        "value": [
            len(post_boundary_reconstructed_races),
            post_boundary_reconstructed_races[
                "advertised_start_uk"
            ].notna().sum(),
            post_boundary_reconstructed_races[
                "advertised_start_utc"
            ].notna().sum(),
            post_boundary_reconstructed_races[
                "advertised_start_course_local"
            ].notna().sum(),
            post_boundary_reconstructed_races[
                [
                    "candidate_course_label",
                    "candidate_jurisdiction",
                ]
            ].drop_duplicates().shape[0],
        ],
    }
)

,measure,value
0,post-boundary races,10352
1,selected UK timestamps,10352
2,selected UTC timestamps,10352
3,selected course-local timestamps,10352
4,distinct governed course identities,199


In [92]:
# Combine pre-boundary reconstructed races and post-boundary explicit races
# into one canonical temporal population.
#
# Unresolved pre-boundary races retain both candidate branches but have no
# selected advertised timestamp.

canonical_temporal_columns = [
    "date",
    "course",
    "off",
    "race_id",
    "race_name",
    "candidate_course_label",
    "candidate_jurisdiction",
    "iana_timezone",
    "selected_branch",
    "decision_method",
    "decision_confidence",
    "advertised_start_uk",
    "advertised_start_utc",
    "advertised_start_course_local",
]

canonical_race_times = pd.concat(
    [
        pre_boundary_reconstructed_races[
            canonical_temporal_columns
        ],
        post_boundary_reconstructed_races[
            canonical_temporal_columns
        ],
    ],
    ignore_index=True,
)

canonical_race_times[
    "temporal_resolution_status"
] = np.where(
    canonical_race_times["advertised_start_utc"].notna(),
    "resolved",
    "unresolved",
)

canonical_temporal_summary = pd.DataFrame(
    {
        "measure": [
            "all provisional races",
            "resolved races",
            "unresolved races",
            "resolved race percent",
            "distinct candidate race keys",
            "selected UK timestamps",
            "selected UTC timestamps",
            "selected course-local timestamps",
        ],
        "value": [
            len(canonical_race_times),
            canonical_race_times[
                "temporal_resolution_status"
            ].eq("resolved").sum(),
            canonical_race_times[
                "temporal_resolution_status"
            ].eq("unresolved").sum(),
            round(
                100
                * canonical_race_times[
                    "temporal_resolution_status"
                ].eq("resolved").mean(),
                2,
            ),
            canonical_race_times[
                ["date", "course", "off"]
            ].drop_duplicates().shape[0],
            canonical_race_times[
                "advertised_start_uk"
            ].notna().sum(),
            canonical_race_times[
                "advertised_start_utc"
            ].notna().sum(),
            canonical_race_times[
                "advertised_start_course_local"
            ].notna().sum(),
        ],
    }
)

display(canonical_temporal_summary)

canonical_race_times[
    [
        "decision_method",
        "decision_confidence",
        "temporal_resolution_status",
    ]
].value_counts(
    dropna=False
).rename(
    "races"
).reset_index()

,measure,value
0,all provisional races,189043.00
1,resolved races,169465.00
2,unresolved races,19578.00
3,resolved race percent,89.64
4,distinct candidate race keys,189043.00
5,selected UK timestamps,169465.00
6,selected UTC timestamps,169465.00
7,selected course-local timestamps,169465.00


,decision_method,decision_confidence,temporal_resolution_status,races
0,course_local_overnight_rejection,high,resolved,111871
1,stable_post_boundary_course_profile,supported,resolved,47242
2,unresolved,unresolved,unresolved,19578
3,explicit_post_boundary_time,source_explicit,resolved,10352


In [93]:
# Use the narrower and more accurate method name.

for frame in [
    meeting_branch_decisions,
    pre_boundary_reconstructed_races,
    canonical_race_times,
]:
    frame["decision_method"] = frame["decision_method"].replace(
        {
            "course_local_overnight_rejection": (
                "course_local_dead_of_night_rejection"
            )
        }
    )

canonical_race_times[
    [
        "decision_method",
        "decision_confidence",
        "temporal_resolution_status",
    ]
].value_counts(
    dropna=False
).rename(
    "races"
).reset_index()

,decision_method,decision_confidence,temporal_resolution_status,races
0,course_local_dead_of_night_rejection,high,resolved,111871
1,stable_post_boundary_course_profile,supported,resolved,47242
2,unresolved,unresolved,unresolved,19578
3,explicit_post_boundary_time,source_explicit,resolved,10352


### Canonical temporal population

The pre-boundary reconstruction and post-boundary explicit records were
combined into one canonical race-level temporal population.

Across 189,043 provisional races:

* 111,871 races are resolved through course-local dead-of-night rejection;
* 47,242 races are resolved through stable post-boundary course-profile
  evidence;
* 10,352 races use explicit post-boundary 24-hour source times;
* 19,578 races remain unresolved.

Course-local dead-of-night rejection does not assume that a course never holds
evening or night racing. It rejects only the branch that places the entire
particular meeting between 00:00 and 05:59 course-local time while the
alternative branch does not.

The combined population contains 169,465 resolved races, giving overall
temporal coverage of 89.64%.

Every resolved race has:

* an advertised UK civil timestamp;
* an `advertised_start_utc` timestamp;
* a derived racecourse-local timestamp;
* a recorded decision method and confidence level.

Unresolved races retain their source values and both reconstructed 12-hour
candidates but do not receive a selected canonical timestamp. They remain an
explicit evidence backlog for later course-, season-, meeting-type-, or
jurisdiction-specific investigation.

In [94]:
# Rebuild the canonical temporal table while preserving both pre-boundary
# candidate branches for audit and later resolution work.

candidate_audit_columns = [
    "candidate_a_uk_naive",
    "candidate_a_uk_aware",
    "candidate_a_utc",
    "candidate_a_course_local",
    "candidate_a_london_status",
    "candidate_b_uk_naive",
    "candidate_b_uk_aware",
    "candidate_b_utc",
    "candidate_b_course_local",
    "candidate_b_london_status",
]

pre_boundary_canonical = (
    pre_boundary_reconstructed_races[
        canonical_temporal_columns + candidate_audit_columns
    ]
    .copy()
)

post_boundary_canonical = (
    post_boundary_reconstructed_races[
        canonical_temporal_columns
    ]
    .copy()
)

for column in candidate_audit_columns:
    post_boundary_canonical[column] = pd.NA

canonical_race_times = pd.concat(
    [
        pre_boundary_canonical,
        post_boundary_canonical,
    ],
    ignore_index=True,
)

canonical_race_times[
    "temporal_resolution_status"
] = np.where(
    canonical_race_times["advertised_start_utc"].notna(),
    "resolved",
    "unresolved",
)

pd.DataFrame(
    {
        "measure": [
            "canonical races",
            "unresolved races",
            "unresolved races retaining candidate A",
            "unresolved races retaining candidate B",
            "resolved races with selected UTC",
            "distinct candidate race keys",
        ],
        "value": [
            len(canonical_race_times),
            canonical_race_times[
                "temporal_resolution_status"
            ].eq("unresolved").sum(),
            canonical_race_times.loc[
                canonical_race_times[
                    "temporal_resolution_status"
                ].eq("unresolved"),
                "candidate_a_uk_naive",
            ].notna().sum(),
            canonical_race_times.loc[
                canonical_race_times[
                    "temporal_resolution_status"
                ].eq("unresolved"),
                "candidate_b_uk_naive",
            ].notna().sum(),
            canonical_race_times[
                "advertised_start_utc"
            ].notna().sum(),
            canonical_race_times[
                ["date", "course", "off"]
            ].drop_duplicates().shape[0],
        ],
    }
)

,measure,value
0,canonical races,189043
1,unresolved races,19578
2,unresolved races retaining candidate A,19578
3,unresolved races retaining candidate B,19578
4,resolved races with selected UTC,169465
5,distinct candidate race keys,189043


### Preservation of unresolved candidate evidence

The canonical temporal table retains both reconstructed pre-boundary candidate
branches alongside the selected canonical timestamp.

Across the complete 189,043-race population:

* all 19,578 unresolved races retain candidate A;
* all 19,578 unresolved races retain candidate B;
* none of those unresolved races receives a selected UTC timestamp;
* all 169,465 resolved races receive a selected UTC timestamp;
* all 189,043 candidate race keys remain unique.

This allows later investigations to resolve additional meetings without
reconstructing the original ambiguity from scratch. Any future decision can be
attached to the preserved candidates together with a new method, confidence
level, and supporting evidence.

In [95]:
# Final integrity checks for the canonical temporal population.

resolved_mask = canonical_race_times[
    "temporal_resolution_status"
].eq("resolved")

unresolved_mask = canonical_race_times[
    "temporal_resolution_status"
].eq("unresolved")

pre_boundary_mask = pd.to_datetime(
    canonical_race_times["date"]
).lt(pd.Timestamp("2025-10-15"))

post_boundary_mask = ~pre_boundary_mask


integrity_checks = pd.DataFrame(
    {
        "check": [
            "one row per candidate race key",
            "resolved races have selected UK timestamp",
            "resolved races have selected UTC timestamp",
            "resolved races have selected course-local timestamp",
            "unresolved races have no selected UK timestamp",
            "unresolved races have no selected UTC timestamp",
            "unresolved races have no selected course-local timestamp",
            "unresolved pre-boundary races retain candidate A",
            "unresolved pre-boundary races retain candidate B",
            "post-boundary races are all resolved",
            "post-boundary races use explicit 24h method",
            "selected UTC values are unique within race keys",
        ],
        "passed": [
            canonical_race_times[
                ["date", "course", "off"]
            ].duplicated().sum() == 0,

            canonical_race_times.loc[
                resolved_mask,
                "advertised_start_uk",
            ].notna().all(),

            canonical_race_times.loc[
                resolved_mask,
                "advertised_start_utc",
            ].notna().all(),

            canonical_race_times.loc[
                resolved_mask,
                "advertised_start_course_local",
            ].notna().all(),

            canonical_race_times.loc[
                unresolved_mask,
                "advertised_start_uk",
            ].isna().all(),

            canonical_race_times.loc[
                unresolved_mask,
                "advertised_start_utc",
            ].isna().all(),

            canonical_race_times.loc[
                unresolved_mask,
                "advertised_start_course_local",
            ].isna().all(),

            canonical_race_times.loc[
                unresolved_mask & pre_boundary_mask,
                "candidate_a_uk_naive",
            ].notna().all(),

            canonical_race_times.loc[
                unresolved_mask & pre_boundary_mask,
                "candidate_b_uk_naive",
            ].notna().all(),

            canonical_race_times.loc[
                post_boundary_mask,
                "temporal_resolution_status",
            ].eq("resolved").all(),

            canonical_race_times.loc[
                post_boundary_mask,
                "decision_method",
            ].eq("explicit_post_boundary_time").all(),

            canonical_race_times.loc[
                resolved_mask,
                ["date", "course", "off", "advertised_start_utc"],
            ].duplicated().sum() == 0,
        ],
    }
)

integrity_checks

,check,passed
0,one row per candidate race key,True
1,resolved races have selected UK timestamp,True
2,resolved races have selected UTC timestamp,True
3,resolved races have selected course-local time...,True
4,unresolved races have no selected UK timestamp,True
5,unresolved races have no selected UTC timestamp,True
6,unresolved races have no selected course-local...,True
7,unresolved pre-boundary races retain candidate A,True
8,unresolved pre-boundary races retain candidate B,True
9,post-boundary races are all resolved,True


### Final temporal integrity validation

The canonical temporal population passed all final integrity checks.

Confirmed properties include:

* exactly one row per candidate race key;
* every resolved race has selected UK, UTC, and course-local timestamps;
* every unresolved race has no selected canonical timestamp;
* every unresolved pre-boundary race retains both candidate branches;
* every post-boundary race is resolved from an explicit 24-hour source time;
* no selected UTC timestamp is duplicated within a candidate race key.

The temporal reconstruction is therefore internally coherent and preserves the
distinction between source-explicit, high-confidence reconstructed, supported
reconstructed, and unresolved records.

### Reusable temporal reconstruction contract

The notebook has established the required behaviour for a reusable temporal
reconstruction module.

The module must:

1. preserve the raw source `date` and `off` values;
2. identify the format boundary at 15 October 2025;
3. parse post-boundary `HH:MM` values as explicit UK civil times;
4. reconstruct both 12-hour branches for pre-boundary meetings while
   preserving internal race order across the clock wrap;
5. attach historical `Europe/London` before converting to UTC;
6. derive racecourse-local timestamps from UTC and the governed IANA timezone;
7. classify ambiguous and nonexistent London civil times explicitly;
8. apply the course-local dead-of-night rejection rule at meeting level;
9. apply stable post-boundary course-profile evidence only where the same
   branch survives every tested margin;
10. preserve both candidate branches for unresolved races;
11. populate canonical selected timestamps only for resolved races;
12. record the selected branch, decision method, confidence level, and temporal
    resolution status;
13. validate one row per candidate race key and enforce the final integrity
    conditions demonstrated in this notebook.

The reusable implementation must reproduce the notebook totals:

* 189,043 canonical races;
* 169,465 resolved races;
* 19,578 unresolved races;
* 111,871 races resolved by course-local dead-of-night rejection;
* 47,242 races resolved by stable post-boundary course-profile evidence;
* 10,352 races resolved from explicit post-boundary source times.

In [97]:
package_files = sorted(
    path.relative_to(project_root).as_posix()
    for path in (project_root / "src" / "inside_rails").glob("*.py")
)

pd.DataFrame({"module_file": package_files})

,module_file
0,src/inside_rails/__init__.py
1,src/inside_rails/carried_weight.py
2,src/inside_rails/course_jurisdiction.py
3,src/inside_rails/course_locations.py
4,src/inside_rails/race_distance.py
5,src/inside_rails/source_sqlite.py


In [98]:
test_and_script_files = sorted(
    path.relative_to(project_root).as_posix()
    for directory in [
        project_root / "tests",
        project_root / "scripts",
    ]
    if directory.exists()
    for path in directory.rglob("*.py")
)

pd.DataFrame({"file": test_and_script_files})

,file
0,scripts/validate_carried_weight.py
1,scripts/validate_course_jurisdiction.py
2,scripts/validate_course_locations.py
3,scripts/validate_race_distance.py
4,scripts/validate_source_profile.py
